# Improved multi-task CNN: the whole pipeline, step by step

This notebook builds the **improved model** for the Face AI project from start to finish. One network looks at 8 face crops from each frontal video and predicts all **10 targets** at once: oxygen saturation, temperature, HbA1c, cholesterol, respiratory rate, arterial stiffness, stress, age, BMI and recorded sex. Step 13 also gives the 7 physiological targets the colour signals from all 1,800 frames of each video.

The project calls it the "improved CNN" because it started from the baseline CNN. Earlier versions compared four CNN backbones with a vision transformer (DINOv2), and the transformer won, so the final model is not strictly a CNN. The losing alternatives were removed from this notebook on 19 October 2026 to keep it short; their results are in the project plan. The name is kept so that the files and folders stay the same.

Every step shows its data and its graphs, so you can follow (and trace) how each decision was made:

| Step | What happens |
|---|---|
| 0 | Settings, paths and plot style |
| 1 | **The data**: labels, participant split and the face crops the CNN sees |
| 2 | **Problems in the data**: what makes this hard, shown with graphs |
| 3 | **Data cleaning**: explicit rules and a log of what changed |
| 4 | **Preparing tensors**: standardisation using training people only |
| 5 | **Data augmentation**: what we change in training images and why |
| 6 | **The model**: frozen DINOv2 + task-specific heads, the loss and the training function |
| 7 | **The layers**: the final model, block by block |
| 8 | **Final training**: 3 seeds, loss curves, early stopping |
| 9 | **Validation check**: overfitting and predictions on people the model has not trained on |
| 10 | **Test evaluation**: scored once, compared with the baseline CNN and a constant guess |
| 11 | **Summary**: what improved, what did not, and why |
| 12 | **Cross-validation**: the final set-up judged on all 510 non-test people in 5 rounds, the heads or Ridge for each target, and a final model trained on 510 people |
| 13 | **The 1,800 frames for the physiological targets**: the colour signals from every frame, as 110 summary numbers or read by a new time-series network, judged in the same 5 rounds |

**Rules followed throughout**

* People never cross splits: 420 training, 90 validation and 90 test people. Both sessions of a person stay together.
* Everything learned (scaling, constant baseline, model choice) uses training or validation people only.
* The **test** people are used only at the end: in Step 10, and again in Steps 12d and 13d to compare finished models. Nothing is chosen on them.

**How to run.** Choose the kernel **`.venv (3.14.6)`** and *Run All*.

* Saved training runs in `results/runs/` (Step 8) and `results/cv/` (Steps 12–13) are reused automatically, but only if they were trained with the same settings (Step 6b checks). A re-run then takes about 1.5 minutes on the GPU.
* Set `RETRAIN = True` to train everything again: about 4 minutes for Step 8, about 11 for Step 12 (plus about 5 to compute the saved DINOv2 numbers if they are missing) and about 13 for Step 13 on this laptop's GPU. On the CPU it takes many hours.
* While it trains, the notebook asks Windows to keep the screen on. With the screen off, this laptop holds the GPU at 33 W, and training runs about 4× slower.
* Step 0 uses the GPU automatically when PyTorch can see one, otherwise the CPU. The saved runs were trained on the GPU (Step 8 on 16 October 2026, Steps 12–13 on 19 October 2026).
* The colour signals of Step 2h are read from `cache/signals.npz`, and the synced finger pulse of Step 2h-5 from `cache/ppg_sync.npz`. Recomputing them (`RUN_SIGNALS = True`) takes about 70 minutes on the CPU.
* Step 12 reads DINOv2's numbers from `cache/dinov2_features.npz` (115 MB). If that file is missing, they are computed again, which takes about 5 minutes on the GPU.
* Set `QUICK_TEST = True` for a short check that the code runs; it writes to a separate folder. It takes about 9 minutes: about 1 for Steps 0–11, about 8 for Step 12 (Ridge is not shortened in quick mode) and under a minute for Step 13.
* All graphs are also saved as PNG files in `results/figures/`.

> **Result in brief** (saved run, 16 October 2026, GPU)
>
> * **Final model.** A frozen, pretrained **DINOv2** vision transformer turns each frame into 384 numbers. A small private head per task (task-specific heads) makes the 10 predictions. The final prediction averages 3 seeds. Only the heads are trained: 1–1.5 minutes per seed on this laptop's GPU. Everything was chosen on validation people only.
> * **Test vs the baseline CNN** (same 90 people):
>   * **Age:** error 1.63 vs 2.61 years (skill +53% vs +25%). The improved-minus-baseline interval excludes zero, so this gain is unlikely to be chance.
>   * **BMI:** 2.81 vs 2.98 kg/m² (skill +18% vs +13%).
>   * **Sex:** accuracy 97.8% vs 97.2% (4 vs 5 of 180 recordings wrong).
> * **Physiological targets** stay at the constant-guess level for both models. Eight still frames cannot show a pulse or a breath (Step 2g). Colour signals from every frame (Step 2h) show the pulse at rest; Step 13 gives them to these targets.
> * **Cross-validation (Step 12, 19 October 2026).** The final set-up was judged on all 510 non-test people in 5 rounds. The intervals are about 2.4× narrower: age +54% (+44% to +60%), BMI +25% (+20% to +30%). For BMI, a simple Ridge model on the same DINOv2 numbers did better and replaced the heads. The final model, retrained on all 510 people, scores 1.50 years for age and 2.45 kg/m² for BMI on the test people (Step 10: 1.63 and 2.81).
> * **The 1,800 frames for the physiological targets (Step 13, 19 October 2026).** At the student's decision, the 7 physiological targets also get the colour signals from every frame. Step 2h's 110 summary numbers did better than a new time-series network on the full traces. On 510 people they make two targets measurably better: respiratory rate (1.35 → 1.32 breaths/min) and arterial stiffness (skill +0.3% → +3.0%), apparently through the pulse rate. The other 5 do not change, and all 7 stay within a few percent of the constant guess.

## Step 0 · Settings, paths and plot style

The settings below are the only numbers you would normally change. The colours are fixed so each model keeps its colour in every chart:

* **Blue** = the improved CNN.
* **Orange** = the baseline CNN.
* **Grey** = the constant guess (always predicting the training median).

In [ ]:
# Step 0 · Settings, paths and plot style (used by every step below)
from pathlib import Path
import hashlib
import json
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display
os.environ.setdefault('HF_HUB_OFFLINE', '1')     # DINOv2 is read from the local weights folder, never downloaded
warnings.filterwarnings('ignore', message='IProgress not found')   # a progress-bar notice from a helper library
from transformers import Dinov2Model             # the model's backbone (Step 6)
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

QUICK_TEST = False   # True = tiny run (1 epoch, 3 batches) only to check the code; writes to results_quick_test/
RETRAIN = False      # True = train again even when a saved run with identical settings exists
RUN_SIGNALS = False  # True = extract the colour signals again from the raw videos (about 70 min; Step 2h)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # the GPU when PyTorch can see one, else the CPU
torch.backends.cudnn.deterministic = True                                # same seed -> same result on every GPU rerun
torch.backends.cudnn.conv.fp32_precision = 'ieee'                        # full 32-bit maths on the GPU, like the CPU runs


def find_project_root(start):
    """Walk upwards from the notebook folder until the Face_AI project folder (it holds db.csv)."""
    for folder in [start, *start.parents]:
        if (folder / 'db.csv').is_file() and (folder / 'baseline_test').is_dir():
            return folder
    raise FileNotFoundError('Open this notebook inside the Face_AI project folder.')


PROJECT = find_project_root(Path.cwd().resolve())
HERE = PROJECT / 'experiments' / 'improved_cnn'
RESULTS = HERE / ('results_quick_test' if QUICK_TEST else 'results')
RUNS, FIGURES = RESULTS / 'runs', RESULTS / 'figures'
for folder in (RUNS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)
FACES = PROJECT / 'baseline_test' / 'cache' / 'faces.npz'     # the same face crops the baseline CNN used
SPLIT = PROJECT / 'participant_split.csv'                    # 420 / 90 / 90 people, fixed in advance
BASELINE = PROJECT / 'baseline_test' / 'results'             # saved baseline CNN predictions (for comparison)
DINOV2 = HERE / 'weights' / 'dinov2-small'                   # Meta's DINOv2 ViT-S/14 (facebook/dinov2-small, pinned revision)
SIGNALS = HERE / 'cache' / 'signals.npz'                     # colour signals from every frame (rppg_signals.py, Step 2h)
PPG_SYNC = HERE / 'cache' / 'ppg_sync.npz'                    # the synced finger pulse on the same clock (Step 2h-5)

TARGETS = ['saturation', 'temperature', 'glycated_hemoglobin', 'cholesterol',
           'respiratory', 'rigidity', 'stress', 'age', 'bmi']
TASKS = TARGETS + ['sex']                                     # 9 numbers + 1 yes/no = 10 tasks
PERSON_LEVEL = ['glycated_hemoglobin', 'cholesterol', 'stress', 'age', 'bmi']   # same value in both sessions
NAME = {'saturation': 'Oxygen saturation', 'temperature': 'Temperature', 'glycated_hemoglobin': 'HbA1c',
        'cholesterol': 'Cholesterol', 'respiratory': 'Respiratory rate', 'rigidity': 'Arterial stiffness',
        'stress': 'Stress score', 'age': 'Age', 'bmi': 'BMI', 'sex': 'Recorded sex', 'pulse': 'Pulse (reference)'}
UNIT = {'saturation': '%', 'temperature': '°C', 'glycated_hemoglobin': '%', 'cholesterol': 'mmol/L',
        'respiratory': 'breaths/min', 'rigidity': 'index', 'stress': 'score', 'age': 'years',
        'bmi': 'kg/m²', 'pulse': 'bpm'}

SETTINGS = dict(
    epochs=15,            # maximum passes over the training videos per run
    patience=5,           # early stopping: stop after 5 epochs without a better validation loss
    batch_videos=16,      # videos per training step
    frames_train=4,       # random 4 of the 8 frames per video during training (all 8 for evaluation)
    lr_heads=1e-3,        # learning rate of the task heads
    weight_decay=1e-4,    # small penalty on large weights (regularisation)
    dropout=0.3,          # randomly hide 30% of the shared features during training
    head_hidden=64,       # size of the hidden layer when every task gets its own small head
    huber_delta=1.0,      # Huber loss: squared error for small mistakes, straight line for big ones
    bootstrap=2000,       # participant resamples for the 95% intervals
    seeds=[0, 1, 2],      # final model = average of 3 runs with different random starts
)
MAX_BATCHES = None
if QUICK_TEST:
    SETTINGS.update(epochs=1, seeds=[0])
    MAX_BATCHES = 3

# One colour per identity (blue/orange, violet/aqua and the 4-colour line set pass colour-blindness checks).
C = dict(spec='#2a78d6', cnn='#eb6834', aqua='#1baf7a', yellow='#eda100', group1='#6250d6', group2='#1baf7a',
         ref='#898781', ink='#0b0b0b', ink2='#52514e', grid='#e1e0d9', axis='#c3c2b7', surface='#fcfcfb')
plt.rcParams.update({
    'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'savefig.facecolor': C['surface'],
    'font.family': 'sans-serif', 'font.sans-serif': ['Segoe UI', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.labelcolor': C['ink2'], 'axes.edgecolor': C['axis'], 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': C['grid'], 'grid.linewidth': 0.8, 'grid.linestyle': '-',
    'xtick.color': C['axis'], 'ytick.color': C['axis'], 'xtick.labelcolor': C['ink2'], 'ytick.labelcolor': C['ink2'],
    'text.color': C['ink'], 'legend.frameon': False, 'legend.fontsize': 9,
    'lines.linewidth': 1.8, 'lines.solid_capstyle': 'round', 'lines.solid_joinstyle': 'round',
    'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})
pd.set_option('display.precision', 3)
pd.set_option('display.width', 160)


def save(fig, name):
    """Save a figure as a slide-ready PNG in results/figures/."""
    fig.savefig(FIGURES / f'{name}.png')


def title(fig, text):
    """Figure headline, left-aligned: the takeaway, not just the chart type."""
    fig.suptitle(text, x=0.01, ha='left', fontweight='semibold', fontsize=12)


def participant_mae(true, pred, ids):
    """Mean absolute error averaged within each person first (both sessions), then across people."""
    err = pd.Series(np.abs(np.asarray(true, float) - np.asarray(pred, float)), index=np.asarray(ids))
    return err.dropna().groupby(level=0).mean().mean()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(2**20), b''):
            h.update(block)
    return h.hexdigest()


assert sha256(DINOV2 / 'model.safetensors').startswith('ae1e99fc'), 'unexpected DINOv2 weight file'
print('Project folder:', PROJECT)
print('Results folder:', RESULTS.relative_to(PROJECT))
print('PyTorch', torch.__version__, '| computing on:',
      f'GPU ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else f'CPU ({torch.get_num_threads()} threads)')

## Step 1 · The data

**Labels.** `db.csv` has one row per **person × session × camera**: 600 people, recorded **before** and **after** exercise by 3 cameras, so 3,600 rows. Our face crops come from the **frontal camera** only, which gives **1,200 recordings** (600 people × 2 sessions). Each recording carries the 10 target values measured at that session.

**Split.** `participant_split.csv` assigns every person to one group: 420 training, 90 validation and 90 test people. Both sessions of a person always land in the same group, so the model is always judged on *new people*.

pandas does the table work: `read_csv` loads a table, `merge(..., validate='many_to_one')` attaches each person's split and stops if a person appears twice in the split file, and `groupby(...).agg(...)` counts people and recordings per split.

In [ ]:
# Step 1a · Labels, participant split and the recordings the CNN uses
db = pd.read_csv(PROJECT / 'db.csv', dtype={'patient_id': str})
split = pd.read_csv(SPLIT, dtype={'patient_id': str})
data = db[db.camera.eq('FullHDwebcam')].copy()                     # frontal camera = the face crops we have
data['key'] = data.video.map(lambda v: Path(v).stem)               # e.g. 1020_FullHDwebcam_after
data = (data.merge(split, on='patient_id', how='left', validate='many_to_one')
            .sort_values('key').reset_index(drop=True))

print(f'db.csv: {len(db)} rows = {db.patient_id.nunique()} people x 2 sessions x {db.camera.nunique()} cameras')
print(f'frontal recordings used here: {len(data)}')
display(data.groupby('split').agg(people=('patient_id', 'nunique'), recordings=('key', 'size'))
            .loc[['train', 'validation', 'test']])
display(pd.DataFrame({'meaning': [NAME[t] for t in TASKS], 'unit': [UNIT.get(t, 'F / M') for t in TASKS],
                      'first recording': [data[t].iloc[0] for t in TASKS]},
                     index=pd.Index(TASKS, name='column in db.csv')))

**The model input.** The baseline's face-extraction step (see `baseline_test/baseline_cnn.ipynb`, Step 1) already turned every frontal video into **8 face crops**. The crops are spread evenly over the first minute and are 112 × 112 pixels each. We reuse exactly the same crops, so any difference between the baseline and the improved model comes from the **model**, not from different inputs.

In [ ]:
# Step 1b · The model input: 8 face crops per video (made once by the baseline's face-extraction step)
with np.load(FACES, allow_pickle=False) as z:
    position = {str(k): i for i, k in enumerate(z['keys'])}
    order = [position[k] for k in data.key]             # same row order as `data`
    frames = z['frames'][order]                          # (1200 videos, 8 frames, 112, 112, 3) uint8 RGB
    detected = z['detected'][order]
print('frames:', frames.shape, frames.dtype, f'= {frames.nbytes / 1e6:.0f} MB in memory')
print(f'face detected in {detected.mean():.1%} of all {detected.size} frames')


def pixelate(image, block=8):
    """Coarse blocks for slides: shows the layout of a crop without showing a recognisable face."""
    small = image[::block, ::block]
    return np.repeat(np.repeat(small, block, axis=0), block, axis=1)


def show_videos(keys, headline, name, anonymise=False):
    fig, axes = plt.subplots(len(keys), 8, figsize=(12, 1.55 * len(keys)), layout='constrained')
    for r, key in enumerate(keys):
        i = int(np.flatnonzero(data.key.to_numpy() == key)[0])
        for c in range(8):
            ax = axes[r, c]
            ax.imshow(pixelate(frames[i, c]) if anonymise else frames[i, c])
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)
            if c == 0:
                ax.set_ylabel(key.replace('_FullHDwebcam_', '\n'), fontsize=8, rotation=0, ha='right', va='center')
            if r == 0:
                ax.set_title(f'frame {c + 1}', fontsize=8, fontweight='normal')
    title(fig, headline)
    save(fig, name)
    return fig


example_people = data[data.split.eq('train')].patient_id.drop_duplicates().sample(2, random_state=1).tolist()
example_keys = [f'{p}_FullHDwebcam_{s}' for p in example_people for s in ('before', 'after')]
plt.close(show_videos(example_keys, 'Step 1 · Faces pixelated for slides', 'step1_example_inputs_slides', anonymise=True))
show_videos(example_keys, 'Step 1 · What the CNN sees: 8 face crops spread over the first minute of each video',
            'step1_example_inputs')
plt.show()

## Step 2 · Problems in the data

Before building a model it pays to know what can go wrong. The graphs below use **training people only**, so nothing about the validation or test people influences our choices.

In [ ]:
# Step 2a · How are the labels distributed? (training people)
train = data[data.split.eq('train')]
people = train.groupby('patient_id').first()
fig, axes = plt.subplots(3, 3, figsize=(11.5, 8.2), layout='constrained')
for ax, t in zip(axes.flat, TARGETS):
    values = (people if t in PERSON_LEVEL else train)[t]
    counted = 'people' if t in PERSON_LEVEL else 'recordings'
    distinct = values.nunique()
    if distinct <= 15:                                    # a coarse scale: one bar per observed value
        counts = values.value_counts().sort_index()
        ax.bar(counts.index, counts.to_numpy(), width=0.8 * np.diff(counts.index.to_numpy()).min(), color=C['group1'])
    else:
        bins = np.arange(17.5, values.max() + 1.5, 1) if t == 'age' else 30
        ax.hist(values, bins=bins, color=C['group1'], edgecolor=C['surface'], linewidth=0.8)
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    ax.axvline(median, color=C['ink'], lw=1)
    right_heavy = median > (values.min() + values.max()) / 2
    ax.text(0.03 if right_heavy else 0.97, 0.95,
            f'{distinct} distinct values\nmedian {median:.4g} · IQR {q3 - q1:.3g}\nn = {len(values)} {counted}',
            transform=ax.transAxes, ha='left' if right_heavy else 'right', va='top', fontsize=8.5, color=C['ink2'])
    ax.set_title(f'{NAME[t]} ({UNIT[t]})')
title(fig, 'Step 2a · Several labels barely vary (saturation, temperature) and age is extremely skewed')
save(fig, 'step2a_label_distributions')
plt.show()
print(f"Saturation at 98–99%: {train.saturation.isin([98, 99]).mean():.0%} of training recordings | "
      f"temperature within 36.5–36.7 °C: {train.temperature.between(36.45, 36.75).mean():.0%} | "
      f"aged 18–24: {people.age.between(18, 24).mean():.0%} of training people ({(people.age > 24).sum()} older)")

**Reading 2a.**

* **Saturation and temperature barely vary.** Saturation takes only about 10 distinct values, mostly 98–99%. Temperature moves in 0.1 °C steps. When almost everyone has the same value, "always guess the median" is already hard to beat.
* **Age is extremely skewed.** About 9 in 10 people are 18–24, so the model sees very few older faces.
* **Stiffness has a long right tail.** A few very high values exist.
* **Stress shows spikes at whole numbers.** See 2e.

In [ ]:
# Step 2b · Same person, two sessions minutes apart: how stable is each label? (training people)
wide = train.pivot(index='patient_id', columns='step', values=TARGETS + ['pulse'])
jitter = np.random.default_rng(1)
session_targets = ['saturation', 'temperature', 'respiratory', 'rigidity', 'pulse']
fig, axes = plt.subplots(1, 5, figsize=(14, 3.6), layout='constrained')
for ax, t in zip(axes, session_targets):
    before, after = wide[(t, 'before')], wide[(t, 'after')]
    step = 0.1 if t == 'temperature' else 1.0
    spread = (lambda n: jitter.uniform(-0.3, 0.3, n) * step) if t in ('saturation', 'temperature', 'respiratory') else (lambda n: 0)
    ax.scatter(before + spread(len(before)), after + spread(len(after)), s=10, color=C['group1'], alpha=0.45, edgecolor='none')
    lo, hi = min(before.min(), after.min()), max(before.max(), after.max())
    ax.plot([lo, hi], [lo, hi], color=C['ref'], lw=1)
    ax.set_title(f'{NAME[t]} · r = {before.corr(after):.2f}', fontsize=9.5)
    ax.set_xlabel(f'Before exercise ({UNIT[t]})')
axes[0].set_ylabel('After exercise')
title(fig, 'Step 2b · Arterial stiffness changes more between two sessions of one person than between people')
save(fig, 'step2b_session_stability')
plt.show()

stability = pd.DataFrame([{
    'label': NAME[t],
    'identical in both sessions %': 100 * np.isclose(wide[(t, 'before')], wide[(t, 'after')]).mean(),
    'session-to-session r': wide[(t, 'before')].corr(wide[(t, 'after')]),
    'mean change after exercise': (wide[(t, 'after')] - wide[(t, 'before')]).mean(),
} for t in TARGETS + ['pulse']]).set_index('label')
display(stability.round(2))

**Reading 2b.** Each person was measured twice, minutes apart.

* **Copied labels.** HbA1c, cholesterol, stress, age and BMI are (almost) identical in both sessions, so each person contributes one value, recorded twice.
* **Noisy labels.** Arterial stiffness correlates only about r = 0.27 between the two sessions: one person's two readings differ more than different people do.
* **Exercise state.** Respiratory rate and saturation also move between sessions, partly because of the exercise.

A noisy label puts a ceiling on what any model can achieve, so we evaluate against that reality instead of hoping for perfect accuracy.

In [ ]:
# Step 2c · Are the targets related to each other? (training recordings, Spearman rank correlation)
numeric = train[TASKS].assign(sex=train.sex.map({'F': 0.0, 'M': 1.0}))
corr = numeric.corr(method='spearman')
diverging = LinearSegmentedColormap.from_list('blue_grey_red', ['#256abf', '#f0efec', '#e34948'])
fig, ax = plt.subplots(figsize=(7.6, 6.4), layout='constrained')
image = ax.imshow(corr.to_numpy(), cmap=diverging, vmin=-1, vmax=1)
ax.set_xticks(range(len(TASKS)), [NAME[t] for t in TASKS], rotation=45, ha='right')
ax.set_yticks(range(len(TASKS)), [NAME[t] for t in TASKS])
ax.grid(False)
for i in range(len(TASKS)):
    for j in range(len(TASKS)):
        value = corr.iat[i, j]
        if i != j and abs(value) >= 0.2:                 # label only the notable relationships
            ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8, color=C['ink'])
fig.colorbar(image, ax=ax, shrink=0.8, label='Spearman ρ (−1 opposite … +1 same direction)')
title(fig, 'Step 2c · Most targets are only weakly related to each other')
save(fig, 'step2c_label_correlations')
plt.show()

**Reading 2c.**

* Most pairs of targets are only weakly related (|ρ| < 0.2, cells left blank).
* The few visible links are modest:
  * BMI with sex (0.33);
  * age with BMI (0.23);
  * saturation slightly lower for older and heavier people (−0.21 each);
  * stress with HbA1c (0.23) and with sex (−0.22).
* **What this means for multi-task learning:** sharing one CNN helps the tasks that depend on the same *face appearance* (age, BMI, sex, and through them a little of saturation). It cannot give the physiological targets information they do not have.

In [ ]:
# Step 2d · Unusual values: robust z-score = distance from the median in units of typical spread (training)
checked = [t for t in TARGETS if t != 'age']          # age is skewed by design (2a): every older person would "stand out"
fig, ax = plt.subplots(figsize=(9.5, 4.6), layout='constrained')
flagged = []
for row, t in enumerate(checked[::-1]):
    values = (people if t in PERSON_LEVEL else train)[t].to_numpy(float)
    spread = 1.4826 * np.median(np.abs(values - np.median(values)))       # MAD, scaled to match an SD
    z = (values - np.median(values)) / spread
    y = row + np.random.default_rng(row).uniform(-0.18, 0.18, len(z))
    far = np.abs(z) > 6
    ax.scatter(z[~far], y[~far], s=8, color=C['group1'], alpha=0.3, edgecolor='none')
    ax.scatter(z[far], y[far], s=26, color=C['group2'], edgecolor=C['surface'], linewidth=0.8, zorder=3)
    flagged.append({'label': NAME[t], 'values with |z| > 6': int(far.sum()),
                    'those values': ', '.join(f'{v:g}' for v in sorted(set(values[far]))[:8]) or '—'})
ax.axvline(-6, color=C['ref'], lw=1)
ax.axvline(6, color=C['ref'], lw=1)
ax.set_yticks(range(len(checked)), [NAME[t] for t in checked[::-1]])
ax.set_xlabel('Robust z-score  ·  |z| > 6 highlighted: worth checking, not automatically wrong')
title(fig, 'Step 2d · A handful of extreme values; most labels sit in a believable range')
save(fig, 'step2d_unusual_values')
plt.show()
display(pd.DataFrame(flagged).set_index('label'))

**Reading 2d.**

* The **robust z-score** measures how far a value is from the median, in units of the typical spread. It uses the median and the median absolute deviation (MAD), so the outliers themselves do not distort it.
* A few values stand far out, for example very high stiffness values and one very low cholesterol value. A |z| larger than 6 means "check this", not "delete this".
* Step 3 removes only values that are physically implausible. Step 6 uses a **Huber loss**, which limits how much any single extreme label can pull on the model.

In [ ]:
# Step 2e · Records that disagree with themselves. The consistency table checks EVERY recording, because the
# cleaning rules of Step 3 apply to every recording; the stress picture below uses training people only.
checks = []
sex_mismatch = data.groupby('patient_id').sex.nunique().gt(1)
checks.append({'check': 'Recorded sex differs between the two sessions', 'people': int(sex_mismatch.sum()),
               'example': ', '.join(sex_mismatch[sex_mismatch].index[:3])})
for t in PERSON_LEVEL:
    differs = data.groupby('patient_id')[t].nunique().gt(1)
    checks.append({'check': f'{NAME[t]} differs between the two sessions', 'people': int(differs.sum()),
                   'example': ', '.join(differs[differs].index[:3])})
checks.append({'check': 'Missing values in any target column', 'people': int(data[TASKS].isna().any(axis=1).sum()), 'example': ''})
checks.append({'check': 'Duplicate recording keys', 'people': int(data.key.duplicated().sum()), 'example': ''})
display(pd.DataFrame(checks).set_index('check'))

people_train = data[data.split.eq('train')].groupby('patient_id').first()
whole = people_train.stress % 1 == 0
fig, ax = plt.subplots(figsize=(8.5, 3.8), layout='constrained')
bins = np.arange(0.875, 7.9, 0.25)
ax.hist([people_train.stress[whole], people_train.stress[~whole]], bins=bins, stacked=True,
        color=[C['group1'], C['group2']], edgecolor=C['surface'], linewidth=0.6,
        label=[f'whole numbers ({whole.sum()} people)', f'decimals ({(~whole).sum()} people)'])
ax.set_xlabel('Stress score')
ax.set_ylabel('Training people')
ax.legend(loc='center right', title='stress recorded as', title_fontsize=8.5)
title(fig, 'Step 2e · The stress column mixes two scoring styles (training people)')
save(fig, 'step2e_stress_scoring')
plt.show()

**Reading 2e.**

* **Sex mismatch.** Two people have a different sex recorded at their two sessions. We cannot tell which is right, so their sex label is hidden from training and grading (Step 3).
* **Stress.** About three-quarters of people have whole-number stress scores and one quarter have decimals. These look like two different questionnaire versions or scoring rules. We cannot convert one into the other without the scale definitions; this is a question for the mentor. We keep the values and remember this when reading the stress results.

In [ ]:
# Step 2f · Do the videos look different from each other? Average brightness of every crop (training videos)
data['brightness'] = frames.mean(axis=(1, 2, 3, 4))               # one number per video: mean pixel value 0..255
data['stress scoring'] = np.where(data.stress % 1 == 0, 'whole numbers', 'decimals')
seen = data[data.split.eq('train')]
brightness = seen.brightness.to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), width_ratios=[1.3, 1], layout='constrained')
axes[0].hist(brightness, bins=40, color=C['group1'], edgecolor=C['surface'], linewidth=0.6)
axes[0].set_xlabel('Mean brightness of the 8 crops (0 = black, 255 = white)')
axes[0].set_ylabel('Training videos')
axes[0].set_title('Lighting varies a lot between videos', fontsize=9.5)
groups = [('whole numbers', C['group1']), ('decimals', C['group2'])]
parts = [seen.loc[seen['stress scoring'] == g, 'brightness'] for g, _ in groups]
box = axes[1].boxplot(parts, positions=[0, 1], widths=0.45, patch_artist=True, showfliers=False,
                      medianprops=dict(color=C['ink'], lw=1.4), whiskerprops=dict(color=C['ink2']),
                      capprops=dict(color=C['ink2']))
for patch, (_, colour) in zip(box['boxes'], groups):
    patch.set(facecolor=colour, alpha=0.55, edgecolor=C['ink2'])
for i, values in enumerate(parts):
    axes[1].text(i + 0.27, values.median(), f'median {values.median():.0f}', va='center', fontsize=8.5, color=C['ink2'])
axes[1].set_xticks([0, 1], [f'stress: {g}' for g, _ in groups])
axes[1].set_xlim(-0.5, 1.9)
axes[1].set_ylabel('Mean crop brightness')
axes[1].set_title('…and the two stress-scoring groups were filmed differently', fontsize=9.5)
title(fig, 'Step 2f · Recording conditions differ between groups of videos (possible "batches"; training videos)')
save(fig, 'step2f_image_brightness')
plt.show()
darkest, brightest = seen.key.iloc[np.argsort(brightness)[:2]].tolist(), seen.key.iloc[np.argsort(brightness)[-2:]].tolist()
plt.close(show_videos(darkest + brightest, 'Step 2f · Darkest and brightest videos (pixelated)', 'step2f_extremes_slides', anonymise=True))
show_videos(darkest + brightest, 'Step 2f · The two darkest and two brightest videos', 'step2f_extremes')
plt.show()

**Reading 2f.**

* Lighting differs strongly between videos.
* The two stress-scoring groups were also filmed under different light. This suggests the data was collected in **batches** (different rooms, periods or settings).
* **The risk:** a model can learn "bright video → batch A → typical label of batch A". That is a **shortcut**, not real physiology.
* **What we do about it:** report it as a caveat. Brightness and contrast augmentation, which changes the lighting of training images at random, was tested against this shortcut in earlier runs. It did not help, so it is not used.

In [ ]:
# Step 2g · Why physiology is hard with this input: 8 snapshots over 60 seconds vs a pulse and a breath
t = np.linspace(0, 60, 6000)
frame_times = np.linspace(0, 59.97, 8)                       # where the 8 crops were taken
pulse = np.sin(2 * np.pi * (75 / 60) * t)                    # 75 beats per minute
breath = np.sin(2 * np.pi * (18 / 60) * t)                   # 18 breaths per minute
fig, axes = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True, layout='constrained')
for ax, wave, label, rate in [(axes[0], pulse, 'Heartbeat (75 per minute)', 75 / 60),
                              (axes[1], breath, 'Breathing (18 per minute)', 18 / 60)]:
    ax.plot(t, wave, color=C['ref'], lw=1)
    ax.scatter(frame_times, np.sin(2 * np.pi * rate * frame_times), s=46, color=C['group1'],
               edgecolor=C['surface'], linewidth=1.4, zorder=3)
    ax.set_yticks([])
    ax.set_title(label, fontsize=9.5)
axes[1].set_xlabel('Seconds into the video  ·  dots = the 8 frames the CNN sees')
title(fig, 'Step 2g · 8 frames, ~8.6 s apart, cannot follow a heartbeat or a breath')
save(fig, 'step2g_time_resolution')
plt.show()

**Reading 2g.** This is the biggest limitation of the face-crop input, and it affects the baseline and the improved CNN equally.

* A heartbeat lasts under a second and a breath a few seconds. The CNN sees one frame every ~8.6 seconds, so it **cannot see pulse or breathing waveforms**.
* For the physiological targets, the CNN can only use what a still face shows: age, build, skin, flushing after exercise and lighting.
* So from the crops alone we expect clear gains for **age, BMI and sex**, and at best small gains for the physiological targets. Step 2h adds the missing time information.

### 2h · The fix we test for 2g: colour signals from every frame (rPPG)

**rPPG** (remote photoplethysmography) measures the pulse with an ordinary camera.

* With every heartbeat, a little more blood flows into the skin, and blood absorbs green light.
* So the skin gets very slightly darker and lighter with every beat: much less than 1%, invisible to the eye, but measurable when many skin pixels are averaged frame after frame.
* Breathing moves the head and changes the light on the face more slowly, a few times per minute.

The module `rppg_signals.py` (in this folder) turned **every frame** of the first minute of each video into numbers. It runs once, in parallel (about 70 minutes; switch `RUN_SIGNALS` in Step 0). This notebook reads its output.

1. **True time of every frame.** The date/time printed in each frame is read digit by digit and looked up in the video's timing file. Every value gets its real time.
   * Where some stamps cannot be read (white digits on bright window blinds), their readable neighbours fix the time.
   * If no stamp at all is readable (6 videos), the frame order is assumed and the video is flagged.
2. **Patches that follow the face.** A face detector (YuNet) finds the eyes, nose and mouth on every frame. Three patches, the forehead and both cheeks, are placed relative to the eyes and move with the face.
3. **Skin colour.** The average red, green and blue of the skin pixels in each patch, so 9 numbers per frame. The positions of the face landmarks are kept too.
   * When the room light gives skin a colour cast outside the skin-colour rule, the patch uses every pixel that is neither too dark nor overexposed.
   * A patch covered by hair stays empty, and that video uses its other patches.
4. **A regular clock.** The values are interpolated onto 30 samples per second (1,800 per video), using the true frame times.

From these signals we compute two rates.

* **Pulse.** The **POS** method (Wang et al., 2017) mixes R, G and B so that lighting changes cancel and the pulse remains. A **spectrum** shows how strong each rhythm is. The highest peak between 42 and 180 per minute is the pulse estimate.
* **Breathing.** We use the up-down **speed** of the head, and the speed of colour change of the skin (speed, not position, so slow drifts do not dominate). The highest peak between 6 and 36 per minute is the breathing estimate.

*Tried and dropped (training people only, before this notebook).*
* Other pulse methods: POS with pre-filtering, CHROM and green-only were all worse than plain POS.
* Breathing from chest movement (optical flow below the face): on 24 training videos its error was 4.2 per minute against 1.1 for the constant guess, with r = +0.2. That is not good enough to justify a second extraction pass.

**The check.** Before trusting these signals, we compare the video estimates with the reference `pulse` and `respiratory` values in `db.csv`, on **training people only**.

In [ ]:
# Step 2h-1 · Colour signals from every frame: load the extracted signals and the extraction log
if RUN_SIGNALS or not SIGNALS.is_file():
    import sys
    sys.path.insert(0, str(HERE))
    import rppg_signals
    rppg_signals.main()                                         # about 70 minutes on 9 processes; writes cache/signals.npz
with np.load(SIGNALS, allow_pickle=False) as z:
    position = {str(k): i for i, k in enumerate(z['keys'])}
    order = [position[k] for k in data.key]                     # same row order as `data` and the face crops
    RGB, LANDMARKS, VALID = z['rgb'][order].astype(np.float64), z['landmarks'][order].astype(np.float64), z['valid'][order]
signal_log = pd.read_csv(SIGNALS.with_name('signals_log.csv')).set_index('key').loc[data.key]
has_signal = signal_log.status.eq('ok').to_numpy()
FS = 30                                                         # samples per second on the regular clock

time_check = signal_log.time_check.fillna('verified')       # (videos from the first extraction pass were all verified)
empty_patch = np.isnan(RGB).all(axis=1).reshape(len(RGB), 3, 3).any(axis=2)      # (videos, 3 patches)
extraction = pd.Series({
    'videos with signals': f'{has_signal.sum()} of {len(has_signal)}',
    'frames kept per video (median, lowest)': f'{signal_log.kept.median():.0f} in 60 s ({signal_log.kept.median() / 60:.1f} per second), '
                                             f'lowest {signal_log.kept.min():.0f}',
    'time stamp read on every frame': f"{time_check.eq('verified').sum()} videos",
    'some stamps unreadable, filled from neighbouring frames': f"{time_check.str.contains('filled').sum()} videos",
    'no stamp readable (frame order assumed, flagged)': f"{time_check.str.startswith('not').sum()} videos",
    'frames whose stamp could not be read': f'{int(signal_log.unreadable_stamps.sum()):,} of {int(signal_log.decoded.sum()):,} '
                                            f'({signal_log.unreadable_stamps.sum() / signal_log.decoded.sum():.1%})',
    'largest gap between kept frames': f'{signal_log.max_gap_s.max() * 1000:.0f} ms (typical video: {signal_log.max_gap_s.median() * 1000:.0f} ms)',
    'frames with a face found': f'{signal_log.face_found.mean():.1%} (lowest video {signal_log.face_found.min():.0%})',
    'skin share of the patches (forehead, right cheek, left cheek)':
        ', '.join(f'{signal_log[c].mean():.0%}' for c in ['skin_forehead', 'skin_right_cheek', 'skin_left_cheek']),
    'videos where room light hides skin colour (patch uses all usable pixels)':
        f"{signal_log.skin_rule.fillna('').str.contains('all usable').sum()} videos",
    'patches without usable pixels, e.g. hair over the forehead': f'{int(empty_patch.sum())} (forehead {int(empty_patch[:, 0].sum())}, '
                                                                    f'cheeks {int(empty_patch[:, 1:].sum())})',
    'regular-clock samples within 0.1 s of a real frame': f'{VALID[has_signal].mean():.1%}'}, name='extraction log')
display(extraction.to_frame())
print('colour signals:', RGB.shape, '= videos x 1,800 samples (60 s x 30 per second) x 9 (3 patches x R, G, B)')
print('landmarks:', LANDMARKS.shape, '= videos x 1,800 samples x 10 (x, y of eyes, nose tip and mouth corners)')

In [ ]:
# Step 2h-2 · From colour to pulse and breathing: POS, spectra and the estimated rates of every video
PULSE_BAND, BREATH_BAND = (0.7, 3.0), (0.1, 0.6)        # Hz = 42-180 beats and 6-36 breaths per minute


def pos_pulse(rgb, window=1.6):
    """POS (Wang et al., 2017): a pulse wave from one patch's mean R, G, B over time, shape (samples, 3).
    In short windows each colour is divided by its own average; G - B and -2R + G + B cancel most lighting and
    motion changes, and adding them with matched spread leaves the pulse."""
    n = int(window * FS)
    w = np.lib.stride_tricks.sliding_window_view(rgb, n, axis=0)            # (windows, 3, n)
    c = w / w.mean(axis=2, keepdims=True)
    s1, s2 = c[:, 1] - c[:, 2], -2 * c[:, 0] + c[:, 1] + c[:, 2]
    h = s1 + s1.std(1, keepdims=True) / (s2.std(1, keepdims=True) + 1e-12) * s2
    h -= h.mean(1, keepdims=True)
    wave = np.zeros(len(rgb))
    np.add.at(wave, np.arange(n)[None] + np.arange(len(h))[:, None], h)     # overlap-add the windows
    return wave


def spectrum(x, seconds, pad):
    """Power at each frequency (Welch): average over pieces of `seconds` (50% overlap, Hann window, zero-padded)."""
    n = int(seconds * FS)
    pieces = np.lib.stride_tricks.sliding_window_view(x, n)[::max(n // 2, 1)]
    pieces = (pieces - pieces.mean(1, keepdims=True)) * np.hanning(n)
    return np.fft.rfftfreq(n * pad, 1 / FS), (np.abs(np.fft.rfft(pieces, n * pad, axis=1)) ** 2).mean(0)


def in_band(f, p, lo_hi):
    keep = (f >= lo_hi[0]) & (f <= lo_hi[1])
    return f[keep], p[keep] / p[keep].sum()                                  # scaled to sum 1: the spectrum's shape


def binned(f, p, lo_hi, width):
    """Sum a spectrum into fixed bins (the model's input has the same length for every video)."""
    edges = np.arange(lo_hi[0], lo_hi[1] + 1e-9, width)
    return np.array([p[(f >= a) & (f < b)].sum() for a, b in zip(edges[:-1], edges[1:])])


def video_signals(rgb, marks):
    """One video -> what the signal branch may use (3 spectrum shapes + 4 quality numbers) and the estimated rates.
    Patches without data (e.g. hair over the forehead) are skipped."""
    patches = [k for k in range(3) if np.isfinite(rgb[:, 3 * k:3 * k + 3]).all()]
    pulse = [in_band(*spectrum(pos_pulse(rgb[:, 3 * k:3 * k + 3]), seconds=20, pad=4), PULSE_BAND) for k in patches]
    f, pulse_shape = pulse[0][0], np.mean([p for _, p in pulse], 0)          # average of the patches
    patch_rates = [60 * f[np.argmax(p)] for _, p in pulse]
    eye = np.median(np.hypot(marks[:, 2] - marks[:, 0], marks[:, 3] - marks[:, 1]))
    height = marks[:, 1::2].mean(1) / eye                                    # head height in eye distances
    head_speed = np.diff(height, prepend=height[0])
    green = np.mean([rgb[:, 3 * k + 1] / rgb[:, 3 * k + 1].mean() for k in patches], 0)   # skin green, patch average
    colour_speed = np.diff(green, prepend=1.0)
    fb, head_shape = in_band(*spectrum(head_speed, seconds=60, pad=8), BREATH_BAND)
    _, colour_shape = in_band(*spectrum(colour_speed, seconds=60, pad=8), BREATH_BAND)
    features = np.concatenate([binned(f, pulse_shape, PULSE_BAND, 0.05), binned(fb, head_shape, BREATH_BAND, 1 / 60),
                               binned(fb, colour_shape, BREATH_BAND, 1 / 60),
                               [pulse_shape.max(), np.std(patch_rates), np.log(np.mean(head_speed ** 2)),
                                np.log(np.mean(colour_speed ** 2))]])
    rates = {'video pulse': 60 * f[np.argmax(pulse_shape)], 'patch disagreement': float(np.std(patch_rates)),
             'video breathing (head)': 60 * fb[np.argmax(head_shape)], 'video breathing (colour)': 60 * fb[np.argmax(colour_shape)]}
    return features, rates


results = [video_signals(RGB[i], LANDMARKS[i]) if has_signal[i] else (None, {}) for i in range(len(data))]
SIGNAL_FEATURES = np.full((len(data), len(next(f for f, _ in results if f is not None))), np.nan)
for i, (features, _) in enumerate(results):
    if features is not None:
        SIGNAL_FEATURES[i] = features
rates = pd.DataFrame([r for _, r in results], index=data.index)
rates[['pulse', 'respiratory', 'step', 'split']] = data[['pulse', 'respiratory', 'step', 'split']]
print(f'{SIGNAL_FEATURES.shape[1]} signal numbers per video = 46 pulse-spectrum bins (3 per minute) '
      '+ 2 x 30 breathing-spectrum bins (1 per minute) + 4 quality numbers')
display(rates.head(4).round(1))

In [ ]:
# Step 2h-3 · Two example training videos: the clearest and the least clear pulse (patches agree / disagree most)
def pulse_band_only(x):
    """For display only: keep the 42-180 per minute part of a signal. (The raw colour also carries a fast flicker
    from the room lights, which the camera turns into a fake 6-10 Hz rhythm; it lies outside the pulse band.)"""
    f = np.fft.rfftfreq(len(x), 1 / FS)
    spectrum_ = np.fft.rfft(x - x.mean())
    spectrum_[(f < PULSE_BAND[0]) | (f > PULSE_BAND[1])] = 0
    return np.fft.irfft(spectrum_, len(x))


train_ok = rates[rates.split.eq('train') & np.isfinite(rates['video pulse']) & ~empty_patch.any(axis=1)]
examples = [train_ok['patch disagreement'].idxmin(), train_ok['patch disagreement'].idxmax()]
fig, axes = plt.subplots(2, 3, figsize=(14, 6.2), layout='constrained')
for row, i in zip(axes, examples):
    t = np.arange(10 * FS) / FS + 20                                         # 10 seconds, from 20 s to 30 s
    green = pulse_band_only(RGB[i, :, 1] / RGB[i, :, 1].mean())[20 * FS:30 * FS]
    wave = pulse_band_only(pos_pulse(RGB[i, :, 0:3]))[20 * FS:30 * FS]
    row[0].plot(t, green / green.std(), color=C['group1'], lw=1.3)
    row[0].plot(t, wave / wave.std() - 4.5, color=C['ink2'], lw=1.3)
    row[0].set_title(f"{data.key[i].replace('_FullHDwebcam_', ' · ')}: forehead green (top), POS wave (bottom)", fontsize=9)
    row[0].set_xlabel('seconds'); row[0].set_yticks([])
    for k, (patch, colour) in enumerate(zip(['forehead', 'right cheek', 'left cheek'], [C['group1'], C['aqua'], C['yellow']])):
        f, p = in_band(*spectrum(pos_pulse(RGB[i, :, 3 * k:3 * k + 3]), seconds=20, pad=4), PULSE_BAND)
        row[1].plot(60 * f, p / p.max(), color=colour, lw=1.4, label=patch)
    row[1].axvline(data.pulse[i], color=C['ink'], ls='--', lw=1.2)
    row[1].set_title(f"Pulse spectrum: video {rates.loc[i, 'video pulse']:.0f}, reference {data.pulse[i]:g} per min (dashed)", fontsize=9)
    row[1].set_xlabel('beats per minute'); row[1].set_yticks([])
    eye = np.median(np.hypot(LANDMARKS[i, :, 2] - LANDMARKS[i, :, 0], LANDMARKS[i, :, 3] - LANDMARKS[i, :, 1]))
    height = LANDMARKS[i, :, 1::2].mean(1) / eye
    f, p = in_band(*spectrum(np.diff(height, prepend=height[0]), seconds=60, pad=8), BREATH_BAND)
    row[2].plot(60 * f, p / p.max(), color=C['group1'], lw=1.4)
    row[2].axvline(data.respiratory[i], color=C['ink'], ls='--', lw=1.2)
    row[2].set_title(f"Breathing spectrum (head speed): video {rates.loc[i, 'video breathing (head)']:.0f}, "
                     f"reference {data.respiratory[i]:g} per min", fontsize=9)
    row[2].set_xlabel('breaths per minute'); row[2].set_yticks([])
axes[0, 1].legend(loc='upper right', fontsize=8)
title(fig, 'Step 2h · What the signals look like: a clear pulse (top) and an unclear one (bottom)')
save(fig, 'step2h_signal_examples')
plt.show()

In [ ]:
# Step 2h-4 · The check: do the video rates agree with the reference values? (training people only)
train_rates = rates[rates.split.eq('train') & np.isfinite(rates['video pulse'])]
checks = []
for estimate, reference, subset in [('video pulse', 'pulse', 'all'), ('video pulse', 'pulse', 'before'),
                                    ('video pulse', 'pulse', 'after'), ('video breathing (head)', 'respiratory', 'all'),
                                    ('video breathing (colour)', 'respiratory', 'all')]:
    part = train_rates if subset == 'all' else train_rates[train_rates.step.eq(subset)]
    checks.append({'video estimate': estimate, 'reference': reference, 'sessions': subset, 'recordings': len(part),
                   'video MAE': (part[estimate] - part[reference]).abs().mean(),
                   'constant MAE': (part[reference] - train_rates[reference].median()).abs().mean(),
                   'correlation r': part[[estimate, reference]].corr().iloc[0, 1],
                   'video mean': part[estimate].mean(), 'reference mean': part[reference].mean()})
check_table = pd.DataFrame(checks)
display(check_table.round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), layout='constrained')
for ax, (estimate, reference, lo, hi) in zip(axes, [('video pulse', 'pulse', 40, 185), ('video breathing (head)', 'respiratory', 5, 37),
                                                    ('video breathing (colour)', 'respiratory', 5, 37)]):
    for step, colour in [('before', C['group1']), ('after', C['group2'])]:
        part = train_rates[train_rates.step.eq(step)]
        ax.scatter(part[reference], part[estimate], s=11, alpha=0.5, color=colour, edgecolor='none', label=f'{step} exercise')
    ax.plot([lo, hi], [lo, hi], color=C['ref'], lw=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(f'reference {reference} in db.csv (per minute)'); ax.set_ylabel(f'{estimate} (per minute)')
    row = check_table[(check_table['video estimate'] == estimate) & check_table.sessions.eq('all')].iloc[0]
    ax.set_title(f"MAE {row['video MAE']:.1f} vs constant {row['constant MAE']:.1f}  ·  r = {row['correlation r']:+.2f}", fontsize=9.5)
axes[0].legend(loc='upper left', fontsize=8)
title(fig, 'Step 2h · Video rates vs the reference values (training people; grey line = perfect agreement)')
save(fig, 'step2h_signal_check')
plt.show()

**Reading 2h.**

* **Extraction worked on every video.**
  * All 1,200 videos were processed. 1,181 have a verified time on every frame, 13 had gaps filled from readable neighbours, and 6 could not be verified.
  * On average 99.8% of the 1,800 clock samples have data, and a face was found on at least 97% of the frames of every video.
* **The pulse is partly visible, mainly at rest.** On the 840 training recordings:
  * **Before exercise**, 43% of video estimates are within 5 beats per minute of the reference, and the correlation is r = +0.23.
  * **When the three patches agree** (within 3 bpm of each other, 41% of recordings), the estimate is much better at rest: error 7.3 bpm, r = +0.56.
  * **After exercise** it seems to fail: only 12% within 5 bpm. The video estimates average about 75 bpm both before and after, while the reference rises from 82 to 102. Either the camera misses the fast post-exercise pulse, or the reference was measured at a different moment than the first minute of video. Step 2h-5 answers this with the synced finger pulse.
  * Measured against `db.csv`, the video pulse is worse than the constant guess (error 21.8 vs 14.8 bpm).
* **Breathing is not recovered.** Neither head movement nor skin colour tracks the reference respiratory rate: r ≈ 0, with errors of 8–9 breaths per minute against 1.3 for the constant guess. The reference values barely vary (Step 2a), which makes this target hard anyway.
* **What we do with it.** Instead of feeding these single-number estimates to the model, we give it the full spectra (110 numbers per recording: pulse band, head-breathing band, colour-breathing band and 4 quality numbers). The model can then learn which parts are reliable. Step 13 tests this as an extra input for the 7 physiological targets.
* **Pulse is not one of our 10 targets.** It is used here only to check that the signals contain heart activity.

### 2h-5 · The synced finger pulse: what was the pulse *during* the video?

The dataset also has a **contact PPG**: a finger sensor recorded during the video. Its `ppg_sync` files hold one finger-pulse value per video frame. `fetch_ppg_sync.py` downloaded them for the 1,200 frontal videos (78 MB, hash-checked; see `data/README.md`), and `rppg_signals.py ppg` put them on the same 30-per-second clock as the colour signals.

* **Alignment.** The finger pulse lines up with the first video frame. The finger wave trails the face wave by about 0.07 s, which is the normal delay for the pulse to reach the hand.
* **Method.** The finger pulse rate comes from exactly the same spectrum method and the same first minute as the video pulse.
* **What it is used for.** It is not a model input and not a target. It only answers *when* `db.csv`'s pulse was measured, and how good the video pulse really is.

In [ ]:
# Step 2h-5 · Finger pulse during the video vs db.csv vs the video estimate (training people only)
if RUN_SIGNALS or not PPG_SYNC.is_file():
    import sys
    sys.path.insert(0, str(HERE))
    import rppg_signals
    rppg_signals.build_ppg_cache()                       # needs data/ppg_sync/ and the raw timing files; a few seconds
with np.load(PPG_SYNC, allow_pickle=False) as z:
    position = {str(k): i for i, k in enumerate(z['keys'])}
    order = [position[k] for k in data.key]
    PPG = z['ppg'][order].astype(np.float64)                   # (1200 videos, 1800 samples): the video's first minute
    PPG_LAST = z['ppg_last_minute'][order].astype(np.float64)  # the video's last minute (videos last about 3 minutes)


def pulse_rate(x):
    """The same method as the video pulse. NaN when the finger signal is flat (no finger on the sensor)."""
    if np.std(x) < 1e-6:
        return np.nan
    f, p = in_band(*spectrum(x, seconds=20, pad=4), PULSE_BAND)
    return 60 * f[np.argmax(p)]


rates['finger pulse'] = [pulse_rate(x) for x in PPG]
rates['finger pulse, last minute'] = [pulse_rate(x) for x in PPG_LAST]
print(f"finger signal flat (no finger on the sensor) in {int(np.isnan(rates['finger pulse']).sum())} of {len(rates)} videos: left out")
with_finger = rates[rates.split.eq('train') & np.isfinite(rates['video pulse']) & np.isfinite(rates['finger pulse'])]
display(with_finger.groupby('step')[['pulse', 'finger pulse', 'finger pulse, last minute', 'video pulse']].mean()
        .rename(columns={'pulse': 'db.csv pulse'}).loc[['before', 'after']].round(1))

compare = []
for a, b, label in [('finger pulse', 'pulse', 'finger pulse during the video vs db.csv'),
                    ('video pulse', 'finger pulse', 'video pulse vs finger pulse'),
                    ('video pulse', 'pulse', 'video pulse vs db.csv')]:
    for step in ('before', 'after'):
        part = with_finger[with_finger.step.eq(step)]
        compare.append({'comparison': label, 'session': step, 'recordings': len(part),
                        f'mean of first': part[a].mean(), 'mean of second': part[b].mean(),
                        'MAE (bpm)': (part[a] - part[b]).abs().mean(), 'within 5 bpm': ((part[a] - part[b]).abs() <= 5).mean(),
                        'correlation r': part[[a, b]].corr().iloc[0, 1]})
compare = pd.DataFrame(compare).set_index(['comparison', 'session'])
display(compare.round(2))
video_gap = with_finger['video pulse'] - with_finger['finger pulse']
agree = (video_gap.abs() <= 5).mean()
miss = video_gap.abs() > 5
print(f"video pulse within 5 bpm of the finger pulse: {agree:.0%} of {len(with_finger)} training recordings "
      f"(within 5 bpm of db.csv: {((with_finger['video pulse'] - with_finger['pulse']).abs() <= 5).mean():.0%})")
print(f"the {miss.sum()} misses: {(video_gap[miss] < 0).mean():.0%} too low (median video {with_finger['video pulse'][miss].median():.0f} "
      f"vs finger {with_finger['finger pulse'][miss].median():.0f} bpm); "
      f"{((with_finger['video pulse'] - with_finger['finger pulse'] / 2).abs()[miss] <= 5).mean():.0%} near half the finger rate")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), width_ratios=[1, 1, 1.5], layout='constrained')
for ax, (x, y, xl, yl) in zip(axes[:2], [('pulse', 'finger pulse', 'db.csv pulse', 'finger pulse during the video'),
                                         ('finger pulse', 'video pulse', 'finger pulse during the video', 'video pulse (colour signals)')]):
    for step, colour in [('before', C['group1']), ('after', C['group2'])]:
        part = with_finger[with_finger.step.eq(step)]
        ax.scatter(part[x], part[y], s=11, alpha=0.5, color=colour, edgecolor='none', label=f'{step} exercise')
    ax.plot([40, 185], [40, 185], color=C['ref'], lw=1)
    ax.set_xlim(40, 185); ax.set_ylim(40, 185)
    ax.set_xlabel(f'{xl} (per minute)'); ax.set_ylabel(f'{yl} (per minute)')
after = compare.loc[('finger pulse during the video vs db.csv', 'after')]
axes[0].set_title(f"After exercise: db.csv says {after['mean of second']:.0f}, the finger {after['mean of first']:.0f}", fontsize=9.5)
axes[1].set_title(f'Video vs finger: within 5 bpm for {agree:.0%} of recordings', fontsize=9.5)
axes[0].legend(loc='upper left', fontsize=8)
def wave_match(i):
    """How closely the forehead's video pulse wave follows the finger wave over the first minute (correlation)."""
    a, b = pulse_band_only(pos_pulse(RGB[i, :, 0:3])), pulse_band_only(PPG[i])
    return np.corrcoef(a[2 * FS:-2 * FS], b[2 * FS:-2 * FS])[0, 1]


agreeing = [i for i in with_finger.index[(with_finger['video pulse'] - with_finger['finger pulse']).abs() <= 2]
            if np.isfinite(RGB[i, :, 0:3]).all()]
i = max(agreeing, key=lambda k: abs(wave_match(k)))                  # the training video whose waves match best
t = np.arange(10 * FS) / FS + 20
video_wave = pulse_band_only(pos_pulse(RGB[i, :, 0:3]))[20 * FS:30 * FS]
finger_wave = pulse_band_only(PPG[i])[20 * FS:30 * FS]
sign = np.sign(wave_match(i)) or 1.0
axes[2].plot(t, finger_wave / finger_wave.std(), color=C['ink2'], lw=1.4, label='finger PPG')
axes[2].plot(t, sign * video_wave / video_wave.std() - 4, color=C['group1'], lw=1.4,
             label='video POS wave' + (' (sign flipped)' if sign < 0 else ''))
axes[2].set_yticks([]); axes[2].set_xlabel('seconds')
axes[2].legend(loc='upper right', fontsize=8, ncols=2)
axes[2].set_title(f"{data.key[i].replace('_FullHDwebcam_', ' · ')}: the same 10 seconds (best-matching training video, "
                  f"r = {abs(wave_match(i)):.2f} over the minute)", fontsize=9.5)
title(fig, 'Step 2h-5 · The finger pulse recorded during the video (training people)')
save(fig, 'step2h_finger_pulse')
plt.show()

**Reading 2h-5.**

* **`db.csv`'s after-exercise pulse was not measured during the video.**
  * Before exercise, the finger pulse during the video agrees with `db.csv` (about 6 bpm apart on average, r = 0.76).
  * After exercise, `db.csv` says about 101 bpm, but the finger sensor shows about 85 during the video's first minute, and about the same (86) in its last minute.
  * So the `db.csv` value was most likely taken at another moment, probably right after the exercise and before filming.
* **The video pulse is much better than Step 2h-4 suggested.** Against the finger pulse it is within 5 bpm for about 7 in 10 recordings, before *and* after exercise. The "failure after exercise" was mostly a timing mismatch in the reference, not a camera problem.
* **The remaining misses are mostly far too low.** In those, the spectrum picked a slow rhythm near the bottom of the search band (median about 50 bpm), probably head movement or drifting light, instead of the pulse. About a quarter of misses sit near half the true rate, and a few far too high. These misses are what keep the correlation low.
* **3 recordings have a flat finger signal** (no finger on the sensor) and are left out.
* **What this means for the targets.**
  * The other after-exercise labels (respiratory rate, saturation, temperature) may also have been measured before filming. There is no synced sensor to check them, so this is a question for the mentor.
  * If so, part of their exercise effect may already have faded by the time of filming. That is one possible reason why no model picks up the exercise effect (Step 10). It is a possibility, not something we can test here.
* **Nothing needs retraining.** Pulse is not a target, and the colour signals are unchanged. A future step could tune the colour-signal extraction against this finger pulse, then re-test the signal branch.

**Summary of the data problems and what we do about them**

| Problem (evidence) | Why it matters | What this pipeline does |
|---|---|---|
| Labels that barely vary (2a) | Little to learn; a constant guess is strong | Always report skill **vs the constant guess** |
| Noisy labels (2b) | Caps achievable accuracy | Huber loss; honest per-target reporting |
| Skewed age (2a) | Few older faces to learn from | Report errors by age group (Step 10) |
| Extreme values (2d) | Can dominate a squared-error loss | Plausibility rules (Step 3) + Huber loss |
| Sex mismatch (2e) | Wrong labels | Hide those two people's sex label |
| Two stress scales (2e) | Mixed meaning | Keep, flag, ask the mentor |
| Lighting batches (2f) | Shortcut learning | A caveat; brightness/contrast augmentation was tested and did not help |
| 8 still frames (2g) | No pulse or breathing visible | Colour signals from every frame (2h), used for the physiological targets in Step 13 |
| After-exercise pulse measured at another moment (2h-5) | Other after-exercise labels may not describe the video either | Flag it; ask the mentor when each label was measured |

## Step 3 · Data cleaning

Cleaning here means **explicit, logged rules**. Two copies of the labels exist after this step:

* `clean`: labels the model **learns from**. Implausible values are hidden (set to "missing").
* `data`: the labels as recorded. These are used for **grading**, so the improved model is graded on exactly the same labels as the baseline.

A hidden ("masked") label is simply skipped by the loss. The recording itself and the person's other labels are kept.

In [ ]:
# Step 3 · Cleaning rules, applied to a copy used for TRAINING; grading keeps the recorded labels
PLAUSIBLE = {                    # broad physical limits; anything outside is almost certainly a recording error
    'saturation': (70, 100), 'temperature': (34.0, 42.0), 'glycated_hemoglobin': (3.0, 20.0),
    'cholesterol': (1.5, 15.0), 'respiratory': (5, 60), 'rigidity': (0, None), 'stress': (0, 10),
    'age': (10, 110), 'bmi': (10, 80)}
clean = data.copy()
cleaning_log = []


def log_rule(rule, recordings, action):
    cleaning_log.append({'rule': rule, 'recordings affected': int(recordings), 'action': action})


# Rule 1 · structure: one key per recording, two recordings per person, one split per person
assert clean.key.is_unique
assert clean.groupby('patient_id').size().eq(2).all()
assert clean.groupby('patient_id').split.nunique().eq(1).all()
log_rule('Each person has exactly 2 frontal recordings in one split', 0, 'checked, nothing to fix')

# Rule 2 · missing values
log_rule('Missing target values', clean[TASKS].isna().sum().sum(), 'none to handle')

# Rule 3 · recorded sex must agree between the two sessions; otherwise hide it (do not guess)
clean['sex_label'] = clean.sex.map({'F': 0.0, 'M': 1.0})
mismatch = clean.groupby('patient_id').sex.transform('nunique').gt(1)
clean.loc[mismatch, 'sex_label'] = np.nan
log_rule('Recorded sex differs between sessions', mismatch.sum(), 'sex label hidden (2 people)')

# Rule 4 · physically implausible values are hidden from training
for t, (lo, hi) in PLAUSIBLE.items():
    bad = pd.Series(False, index=clean.index)
    if lo is not None:
        bad |= clean[t] < lo
    if hi is not None:
        bad |= clean[t] > hi
    if bad.any():
        log_rule(f'{NAME[t]} outside {lo}–{hi} {UNIT[t]}', bad.sum(),
                 f"hidden from training (values: {', '.join(f'{v:g}' for v in sorted(clean.loc[bad, t].unique()))}; "
                 f"split: {', '.join(sorted(clean.loc[bad, 'split'].unique()))})")
        clean.loc[bad, t] = np.nan
log_rule('All other target values inside their plausible ranges', 0, 'kept')

# Rule 5 · the images: every video has a detected face in all 8 frames, and no crop is (almost) black
too_dark = int((frames.mean(axis=(2, 3, 4)) < 10).sum())
log_rule('Videos without a face in every frame', int((~detected.all(axis=1)).sum()), 'none, nothing removed')
log_rule('Crops that are almost black (mean < 10)', too_dark, 'none, nothing removed' if too_dark == 0 else 'kept, flagged')

data['sex_label'] = clean['sex_label']        # grading uses the same sex rule as the baseline did
display(pd.DataFrame(cleaning_log).set_index('rule'))
coverage = pd.DataFrame({'usable labels before': data[TARGETS].notna().sum(),
                         'usable labels after': clean[TARGETS].notna().sum()})
coverage.loc['sex'] = [data.sex.notna().sum(), clean.sex_label.notna().sum()]
coverage.index = coverage.index.map(NAME)
display(coverage.T)

**Reading 3.** The table is **technically clean**: no missing values, no duplicates, and every video has a face in all 8 frames. Only a handful of labels needed hiding. The real problems are the ones from Step 2 (noise, imbalance, batches and still frames), and no amount of cleaning can remove those. They are handled by the loss, the augmentation and honest evaluation.

## Step 4 · Preparing tensors

Neural networks learn best when every numerical target is on a similar scale. We **standardise** each target: subtract the training mean and divide by the training standard deviation. Afterwards a value of +1 means "one standard deviation above the average training person".

* All statistics come from **training people only**. Using validation or test people here would leak information about them into the model.
* The 110 signal numbers from Step 2h are standardised the same way (`SIG`). A video without signals would get 0, the training average.
* The **constant guess** used for every comparison is the training median, the same as in the baseline.
* We also pick **90 training people as a "check" group**. We measure the model on them without augmentation, exactly like validation, so the two curves can be compared fairly to spot overfitting.

In [ ]:
# Step 4 · Tensors and training-only statistics
idx = {s: np.flatnonzero(data.split.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
check_people = np.random.default_rng(0).choice(data.loc[idx['train'], 'patient_id'].unique(), 90, replace=False)
idx['train_check'] = np.flatnonzero(data.patient_id.isin(check_people).to_numpy())

MU = clean.loc[idx['train'], TARGETS].mean().to_numpy()           # training mean   (after cleaning)
SD = clean.loc[idx['train'], TARGETS].std().to_numpy()            # training spread (after cleaning)
CONSTANT = data.loc[idx['train'], TARGETS].median()              # the constant guess = training median
X = torch.from_numpy(frames).permute(0, 1, 4, 2, 3).contiguous()  # (1200, 8, 3, 112, 112) uint8, 0..255
Y = torch.tensor((clean[TARGETS].to_numpy(float) - MU) / SD, dtype=torch.float32)   # standardised; NaN = hidden
S = torch.tensor(clean.sex_label.to_numpy(float), dtype=torch.float32)               # 1 = M, 0 = F, NaN = hidden
SIG_MU = np.nanmean(SIGNAL_FEATURES[idx['train']], 0)             # the 110 signal numbers of Step 2h, standardised
SIG_SD = np.nanstd(SIGNAL_FEATURES[idx['train']], 0)              # with training people only, like the targets
SIG_SD[SIG_SD < 1e-8] = 1.0
SIG = torch.tensor(np.nan_to_num((SIGNAL_FEATURES - SIG_MU) / SIG_SD), dtype=torch.float32)   # no signal -> 0 = average

display(pd.DataFrame({'training mean': MU, 'training SD': SD, 'constant guess (median)': CONSTANT.to_numpy()},
                     index=[NAME[t] for t in TARGETS]).round(3))
print('video rows per group:', {k: len(v) for k, v in idx.items()})
print('X:', tuple(X.shape), X.dtype, '| Y:', tuple(Y.shape), '| S:', tuple(S.shape), '| SIG (signal numbers):', tuple(SIG.shape))

## Step 5 · Data augmentation

Augmentation means **changing the training images at random** in ways that should not change the answer. The model never sees exactly the same input twice, so it learns the face rather than memorising pixels. It is applied to **training only**; validation and test images are never changed.

| Change | Why it is safe for these targets | Why it helps |
|---|---|---|
| Random 4 of the 8 frames | Every frame shows the same person and session | More variety and twice as fast |
| Mirror left↔right | A mirrored face has the same age, BMI, sex… | Doubles the variety of faces |
| Shift up to 8 pixels | The face detector's box is never perfectly centred | Robust to imperfect crops |

We do not change brightness or colours: brightness/contrast changes did not help in earlier tests (Step 2f), and skin colour might carry real information.

In [ ]:
# Step 5 · Augmentation functions (training only) and a picture of what they do
def pick_frames(x, k):
    """Keep k random frames of each video (x: videos, 8, 3, H, W)."""
    if k >= x.shape[1]:
        return x
    choice = torch.argsort(torch.rand(x.shape[0], x.shape[1]), dim=1)[:, :k]
    return x[torch.arange(x.shape[0])[:, None], choice]


def augment(x, shift=8):
    """Random mirror and shift; the same change for all frames of one video."""
    b, f, c, h, w = x.shape
    flip = torch.rand(b) < 0.5
    x = x.clone()
    x[flip] = x[flip].flip(-1)                                           # mirror left <-> right
    padded = F.pad(x.flatten(0, 1), (shift, shift, shift, shift), mode='replicate').view(b, f, c, h + 2 * shift, w + 2 * shift)
    out = torch.empty_like(x)
    for i in range(b):                                                   # shift by cropping a moved window
        dy, dx = np.random.randint(0, 2 * shift + 1, size=2)
        out[i] = padded[i, :, :, dy:dy + h, dx:dx + w]
    return out


def augmentation_figure(anonymise):
    set_seed(3)
    video = X[idx['train'][:1]].float().div(255)                       # one training video, 8 frames
    rows = [('original (4 of 8 frames)', video[:, :4])]
    for n in range(3):
        rows.append((f'augmented example {n + 1}', augment(pick_frames(video, 4))))
    fig, axes = plt.subplots(len(rows), 4, figsize=(7.2, 7.6), layout='constrained')
    for r, (label, clip) in enumerate(rows):
        for c in range(4):
            image = (clip[0, c].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            axes[r, c].imshow(pixelate(image) if anonymise else image)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].grid(False)
            for spine in axes[r, c].spines.values():
                spine.set_visible(False)
        axes[r, 0].set_ylabel(label, fontsize=8.5, rotation=0, ha='right', va='center')
    title(fig, 'Step 5 · Same video, randomly mirrored and shifted')
    save(fig, 'step5_augmentation_slides' if anonymise else 'step5_augmentation')
    return fig


plt.close(augmentation_figure(anonymise=True))
augmentation_figure(anonymise=False)
plt.show()

## Step 6 · The model: frozen DINOv2 + task-specific heads

One network looks at the 8 face crops of a video and predicts all 10 targets (the **multi-task design**):

```
8 face crops ──► DINOv2 on every crop ──► average over the 8 crops ──► 384 numbers per video
                                                                          │
                                        ┌─────────────────────────────────┤ dropout 30%
                                        ▼                  ▼              ▼
                                 head: saturation   head: temperature  …  head: sex   (10 heads)
```

* **The backbone: DINOv2 ViT-S/14**, a *vision transformer* made by Meta. It learned to describe images **without labels**, from 142 million photos.
  * **How it sees an image.** It cuts the image into 14 × 14-pixel patches and lets every patch compare itself with every other patch (*self-attention*). A CNN instead slides small filters over the image. Our 112-pixel crops are enlarged to 224 pixels (16 × 16 patches), the size DINOv2 was trained on.
  * **Its output** is a summary token: 384 numbers per frame.
  * **It is frozen.** Its 22 million numbers are never changed: with only 420 training people, retraining them would memorise those people. Starting from a model trained elsewhere is called **transfer learning**.
* **The heads.** Each task has its own small head, 384 → 64 → 1 with a ReLU in between (*task-specific heads*). Only the heads learn.
* **The loss.** Each numerical task uses a **Huber loss** on the standardised value, and sex uses **binary cross-entropy**. The 10 task losses are averaged with **equal weight**. Hidden labels (Step 3) are skipped.
* **An optional signal branch,** used in Step 13: the 110 signal numbers of Step 2h go through one layer (110 → 32 numbers) and are joined to the 384 before the heads.

**Why this design?** Earlier versions of this notebook compared it with four CNN backbones and with other settings, on the validation people and in cross-validation:
* the backbones were the baseline's small CNN, ResNet-18 from scratch, and ResNet-18 with ImageNet weights, frozen or fine-tuned;
* the settings were one linear layer per task, learned task weights, and brightness/contrast augmentation.

Frozen DINOv2 with task-specific heads did best. The losing alternatives were removed on 19 October 2026 to keep the notebook short; their numbers are in the project plan.

In [ ]:
# Step 6a · Building blocks: DINOv2, the multi-task model and the losses
class DINOv2Small(nn.Module):
    """Meta's DINOv2 ViT-S/14, a vision transformer (not a CNN) -> 384 numbers per frame.
    Each frame is enlarged from 112 to 224 pixels (16 x 16 patches of 14 pixels, the size DINOv2 was trained on).
    The summary ('class') token after the final layer norm describes the frame."""
    width = 384

    def __init__(self):
        super().__init__()
        self.vit = Dinov2Model.from_pretrained(DINOV2, local_files_only=True)   # pretrained weights load here

    def forward(self, x):                          # x: colour-scaled frames, (frames, 3, 112, 112)
        x = F.interpolate(x, size=224, mode='bicubic', align_corners=False)
        return self.vit(pixel_values=x).pooler_output


class MultiTaskCNN(nn.Module):
    """Shared backbone on every frame -> average over frames -> shared vector -> one small head per task (10 outputs).
    Optional signal branch (Step 13): the 110 signal numbers of Step 2h -> 32 numbers, joined to the shared vector."""

    def __init__(self, backbone, hidden=64, dropout=0.3, signal_dim=0):
        super().__init__()
        self.backbone = backbone
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))   # ImageNet colour scaling,
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))    # which DINOv2 uses too
        self.dropout = nn.Dropout(dropout)
        self.signal = nn.Sequential(nn.Linear(signal_dim, 32), nn.ReLU()) if signal_dim else None
        width = backbone.width + (32 if signal_dim else 0)
        self.heads = nn.ModuleDict({task: nn.Sequential(nn.Linear(width, hidden), nn.ReLU(), nn.Linear(hidden, 1))
                                    for task in TASKS})             # a small private layer for every task

    def encode(self, x):                           # x: (videos, frames, 3, 112, 112) with values 0..1
        """The backbone part: every frame -> numbers, then the average over the frames of each video."""
        videos, n_frames = x.shape[:2]
        x = ((x.flatten(0, 1) - self.mean) / self.std).contiguous(memory_format=torch.channels_last)
        return self.backbone(x).view(videos, n_frames, -1).mean(1)

    def forward(self, x, signals=None, encoded=None):
        """encoded: the output of encode(x), when it was computed before (a frozen backbone gives the same numbers every time)."""
        shared = self.encode(x) if encoded is None else encoded
        if self.signal is not None:
            shared = torch.cat([shared, self.signal(signals)], dim=1)       # join what the face and the signals show
        shared = self.dropout(shared)
        return torch.cat([self.heads[task](shared) for task in TASKS], dim=1)   # (videos, 10)


def task_losses(out, y_num, y_sex):
    """One loss per task, skipping hidden labels: Huber for the 9 standardised numbers, cross-entropy for sex."""
    losses = []
    for j in range(len(TARGETS)):
        ok = ~torch.isnan(y_num[:, j])
        losses.append(F.huber_loss(out[ok, j], y_num[ok, j], delta=SETTINGS['huber_delta']) if ok.any() else out.sum() * 0)
    ok = ~torch.isnan(y_sex)
    losses.append(F.binary_cross_entropy_with_logits(out[ok, 9], y_sex[ok]) if ok.any() else out.sum() * 0)
    return torch.stack(losses)


def build_model(cfg):
    """The final model: frozen DINOv2 + task-specific heads, plus the signal branch if cfg['signals'] (Step 13)."""
    model = MultiTaskCNN(DINOv2Small(), SETTINGS['head_hidden'], SETTINGS['dropout'],
                         signal_dim=SIG.shape[1] if cfg.get('signals') else 0)
    model = model.to(DEVICE, memory_format=torch.channels_last)      # GPU (or CPU); channels-last is faster on both
    for p in model.backbone.parameters():
        p.requires_grad = False                                      # frozen: DINOv2 is never changed
    model.frozen_key = 'dinov2-pretrained'                           # its output can be reused (Step 6b)
    return model, [{'params': [p for n, p in model.named_parameters() if not n.startswith('backbone.')],
                    'lr': SETTINGS['lr_heads']}]


FINAL_CFG = dict(signals=False)      # the final set-up of Steps 8-12: face crops only (Step 13 adds the signals)

check = DINOv2Small()
print(f'DINOv2 ViT-S/14 loaded from {DINOV2.relative_to(PROJECT)}: {sum(p.numel() for p in check.parameters()):,} parameters, '
      f'{len(check.vit.encoder.layer)} transformer blocks')
del check

**Step 6b, the training function, also saves time and prevents two mistakes** (added 16 October 2026; the results stay bit-for-bit identical):

* **A frozen backbone's output is computed once.**
  * After every epoch the model is measured on the check and validation videos, without augmentation.
  * A frozen backbone gives exactly the same numbers for them every time, so they are computed once and reused by every run and seed.
  * For DINOv2 this was 45% of every epoch. The training videos still go through the backbone each time, because their augmentation changes.
* **Saved runs are reused only with the same settings.** Each run stores the settings it was trained with. If you change one (say `dropout`), that run is trained again instead of being silently reused.
* **Only the trained parts are saved.** For a frozen backbone, `best.pt` holds just the heads, about 1 MB instead of 89 MB. The backbone comes back from its original, hash-checked weight file.
* **The screen stays on while training.** With the screen off, this laptop holds the GPU at 33 W, and training runs about 4× slower.

In [ ]:
# Step 6b · Training and measuring (one function used by every run in this notebook)
import contextlib
import ctypes

ENCODED = {}          # (frozen backbone, group of videos) -> its output; computed once, reused by every run and seed
TRAINING_SETTINGS = ['epochs', 'patience', 'batch_videos', 'frames_train', 'lr_heads', 'weight_decay', 'dropout',
                     'head_hidden', 'huber_delta']   # the SETTINGS a saved run depends on


@torch.no_grad()
def encoded_videos(model, part, batch):
    """A frozen backbone's output for one group of videos, computed once in the same batches as below.
    Same weights, same pictures (no augmentation), same batch shapes -> exactly the same numbers every time."""
    key = (model.frozen_key, part, batch)
    if key not in ENCODED:
        rows = idx[part]
        ENCODED[key] = torch.cat([model.encode(X[torch.from_numpy(rows[i:i + batch])].float().div_(255).to(DEVICE))
                                  for i in range(0, len(rows), batch)])
    return ENCODED[key]


@torch.no_grad()
def predict(model, part, batch=32):
    """All 8 frames of every video in idx[part], no augmentation.
    Returns numbers in original units, P(male) and the raw outputs."""
    model.eval()
    rows = idx[part]
    cached = encoded_videos(model, part, batch)
    outputs = []
    for i in range(0, len(rows), batch):
        r = torch.from_numpy(rows[i:i + batch])
        signals = SIG[r].to(DEVICE) if model.signal is not None else None
        outputs.append(model(None, signals, encoded=cached[i:i + batch]).cpu())
    out = torch.cat(outputs).numpy()
    return out[:, :9] * SD + MU, 1 / (1 + np.exp(-out[:, 9])), out


@contextlib.contextmanager
def keep_screen_on():
    """While training, ask Windows to keep the screen (and the PC) on. With the screen off, this laptop holds the GPU
    at 33 W and training is about 4x slower. The request ends with the block. Does nothing on other systems."""
    windows = os.name == 'nt'
    if windows:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001 | 0x00000002)   # continuous | system | display
    try:
        yield
    finally:
        if windows:
            ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)                          # back to normal


def equal_loss(out, rows):
    """The selection score: all 10 task losses averaged with equal weight (lower = better)."""
    r = torch.from_numpy(rows)
    return task_losses(torch.from_numpy(out), Y[r], S[r]).mean().item()


def score(rows, reg, prob):
    """Errors against the RECORDED labels, averaged per person first; skill = 1 - MAE / MAE of the constant guess."""
    ids = data.patient_id.to_numpy()[rows]
    result = {}
    for j, t in enumerate(TARGETS):
        y = data[t].to_numpy(float)[rows]
        mae = participant_mae(y, reg[:, j], ids)
        result[f'mae_{t}'] = mae
        result[f'skill_{t}'] = 1 - mae / participant_mae(y, np.full(len(y), CONSTANT[t]), ids)
    y = data.sex_label.to_numpy(float)[rows]
    ok = ~np.isnan(y)
    guess = (prob[ok] >= 0.5).astype(float)
    result['sex_accuracy'] = float(np.mean(guess == y[ok]))
    result['sex_balanced_accuracy'] = float(np.mean([np.mean(guess[y[ok] == c] == c) for c in (0, 1)]))
    result['mean_skill'] = float(np.mean([result[f'skill_{t}'] for t in TARGETS]))
    return result


def run_name(cfg, seed):
    """The folder name of a saved run. It spells out the final set-up, as when the runs were first saved;
    '_rppg' marks the runs with the 110 signal numbers (Step 13)."""
    return f"dinov2-pretrained-frozen_mlp-heads_equal-weights_geometric-aug_lr0{'_rppg' if cfg.get('signals') else ''}_seed{seed}"


def train_run(cfg, seed):
    """Train one configuration with one seed; reuse the saved run if it exists with the same settings (unless RETRAIN)."""
    name = run_name(cfg, seed)
    settings = {k: SETTINGS[k] for k in TRAINING_SETTINGS} | {'max_batches': MAX_BATCHES}
    folder = RUNS / name
    if not RETRAIN and all((folder / f).is_file() for f in ('history.csv', 'best.pt', 'summary.json')):
        summary = json.loads((folder / 'summary.json').read_text(encoding='utf-8'))
        if summary.get('settings') == settings:
            history = pd.read_csv(folder / 'history.csv')
            print(f"saved run reused: {name} (trained in {summary['minutes']:.1f} min on {summary.get('device', 'cpu').upper()}). "
                  f"Its epoch log:")
            for r in history.itertuples():
                print(f"epoch {r.epoch:2d} | train objective {r.train_objective:.3f} | check loss {r.train_check_loss:.3f} | "
                      f"validation loss {r.validation_loss:.3f} | validation mean skill {r.validation_mean_skill:+.1%}"
                      f"{'  ← kept (best)' if r.epoch == summary['best_epoch'] else ''} | {r.seconds:.0f}s")
            return dict(name=name, folder=folder, history=history, summary=summary)
        changed = {k: (summary.get('settings', {}).get(k), v) for k, v in settings.items() if summary.get('settings', {}).get(k) != v}
        print(f'saved run {name} used other settings {changed} (saved, now): training it again')
    folder.mkdir(parents=True, exist_ok=True)
    with keep_screen_on():
        return _train(cfg, seed, name, folder, settings)


def _train(cfg, seed, name, folder, settings):
    set_seed(seed)
    model, groups = build_model(cfg)
    optimizer = torch.optim.AdamW(groups, weight_decay=SETTINGS['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SETTINGS['epochs'])   # smoothly lower the LR
    train_rows = torch.from_numpy(idx['train'])
    rows, best_loss, best_epoch, waited, started = [], np.inf, 0, 0, time.time()
    for epoch in range(1, SETTINGS['epochs'] + 1):
        t0 = time.time()
        model.train()
        model.backbone.eval()                                        # the frozen backbone stays in evaluation mode
        objective = []
        for b, batch in enumerate(train_rows[torch.randperm(len(train_rows))].split(SETTINGS['batch_videos'])):
            if MAX_BATCHES is not None and b >= MAX_BATCHES:
                break
            x = augment(pick_frames(X[batch], SETTINGS['frames_train']).float().div_(255))
            signals = SIG[batch].to(DEVICE) if model.signal is not None else None
            loss = task_losses(model(x.to(DEVICE), signals), Y[batch].to(DEVICE), S[batch].to(DEVICE)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            objective.append(loss.item())
        scheduler.step()
        record = {'epoch': epoch, 'train_objective': float(np.mean(objective))}
        for part in ('train_check', 'validation'):
            reg, prob, out = predict(model, part)
            record[f'{part}_loss'] = equal_loss(out, idx[part])
            record.update({f'{part}_{k}': v for k, v in score(idx[part], reg, prob).items()})
        record['seconds'] = time.time() - t0
        rows.append(record)
        improved = record['validation_loss'] < best_loss
        if improved:
            best_loss, best_epoch, waited = record['validation_loss'], epoch, 0
            # Save the best epoch, not the last. The frozen backbone never changes, so only the trained parts are saved
            # (about 1 MB instead of 89 MB); load_model() takes the backbone from the original weight file.
            torch.save({k: v.cpu() for k, v in model.state_dict().items() if not k.startswith('backbone.')}, folder / 'best.pt')
        else:
            waited += 1
        print(f"epoch {epoch:2d} | train objective {record['train_objective']:.3f} | check loss {record['train_check_loss']:.3f} | "
              f"validation loss {record['validation_loss']:.3f} | validation mean skill {record['validation_mean_skill']:+.1%}"
              f"{'  ← best so far' if improved else ''} | {record['seconds']:.0f}s", flush=True)
        if waited >= SETTINGS['patience']:
            print(f'early stopping: no better validation loss for {waited} epochs')
            break
    history = pd.DataFrame(rows)
    history.to_csv(folder / 'history.csv', index=False)
    best = history.loc[history.epoch == best_epoch].iloc[0]
    summary = dict(name=name, config=cfg, seed=seed, best_epoch=int(best_epoch), best_val_loss=float(best_loss),
                   best_val_mean_skill=float(best['validation_mean_skill']),
                   best_val_sex_balanced_accuracy=float(best['validation_sex_balanced_accuracy']),
                   epochs_run=int(len(history)), minutes=(time.time() - started) / 60, device=str(DEVICE), settings=settings,
                   trainable_parameters=int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
                   total_parameters=int(sum(p.numel() for p in model.parameters())))
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"finished {name}: best epoch {best_epoch}, validation loss {best_loss:.4f}, {summary['minutes']:.1f} min")
    return dict(name=name, folder=folder, history=history, summary=summary)


def load_model(run):
    """Rebuild the model and load its best epoch. The frozen backbone comes from its original (hash-checked) weight file."""
    model, _ = build_model(run['summary']['config'])
    result = model.load_state_dict(torch.load(run['folder'] / 'best.pt', map_location=DEVICE, weights_only=True), strict=False)
    assert not result.unexpected_keys, result.unexpected_keys
    assert all(k.startswith('backbone.') for k in result.missing_keys), result.missing_keys[:5]
    return model.eval()

## Step 7 · The layers of the final model

The table lists every block of the final model, with:

* the **shape** of the data after the block, for one frame (for DINOv2: *tokens × numbers per token*);
* the number of **parameters**, the numbers inside the model, and whether they are trained.

The settings chosen by us (the *hyperparameters*: learning rate, epochs, batch size, dropout, head size and so on) are listed in `SETTINGS` in Step 0.

In [ ]:
# Step 7 · Layer-by-layer table of the final model (shapes for one 112x112 frame)
model, _ = build_model(FINAL_CFG)
model.eval()
count = lambda module: sum(p.numel() for p in module.parameters())
vit = model.backbone.vit                                   # a transformer: each frame becomes 257 tokens of 384 numbers
with torch.no_grad():
    tokens = vit(pixel_values=torch.zeros(1, 3, 224, 224, device=DEVICE)).last_hidden_state
shape = f'{tokens.shape[1]} tokens × {tokens.shape[2]}'
layer_table = pd.DataFrame([
    {'block': 'enlarge to 224 px, cut into 16 × 16 patches of 14 px, + 1 summary token',
     'output shape (one frame)': shape, 'parameters': count(vit.embeddings), 'trained?': 'no (frozen)'},
    {'block': f'{len(vit.encoder.layer)} transformer blocks (self-attention + MLP)',
     'output shape (one frame)': shape, 'parameters': count(vit.encoder), 'trained?': 'no (frozen)'},
    {'block': 'final layer norm, keep the summary token',
     'output shape (one frame)': str(tokens.shape[2]), 'parameters': count(vit.layernorm), 'trained?': 'no (frozen)'},
    {'block': 'average over the 8 frames', 'output shape (one frame)': f'{model.backbone.width} per video',
     'parameters': 0, 'trained?': '—'},
    {'block': f'dropout {SETTINGS["dropout"]:.0%} + 10 task heads ({model.backbone.width} → {SETTINGS["head_hidden"]} → 1 each)',
     'output shape (one frame)': '10 numbers per video', 'parameters': count(model.heads), 'trained?': 'yes'},
]).set_index('block')
display(layer_table)
print(f"total parameters: {count(model):,}, of which trained (the heads): {count(model.heads):,}")
del model, vit

## Step 8 · Final training: 3 seeds

The final configuration is trained with **3 different random seeds**. A seed controls the random starting values, the order of the videos and the augmentation. The final prediction is the **average of the 3 models** (a small *ensemble*), which is steadier than any single run. Runs that already exist with identical settings are reused.

Each run keeps the epoch with the **best validation loss** and stops early if 5 epochs pass without improvement.

In [ ]:
# Step 8a · Train (or load) the final model with 3 seeds
final_runs = []
for seed in SETTINGS['seeds']:
    print(f'\n=== final configuration, seed {seed} ===')
    final_runs.append(train_run(FINAL_CFG, seed))

In [ ]:
# Step 8b · Loss curves of the final runs: seen people (check group) vs new people (validation)
fig, axes = plt.subplots(1, len(final_runs), figsize=(4.2 * len(final_runs), 3.9), sharey=True, layout='constrained', squeeze=False)
for ax, run in zip(axes[0], final_runs):
    h, best = run['history'], run['summary']['best_epoch']
    ax.plot(h.epoch, h.train_check_loss, color=C['spec'], alpha=0.45)
    ax.plot(h.epoch, h.validation_loss, color=C['spec'])
    ax.axvline(best, color=C['ref'], lw=1)
    ax.text(h.epoch.iloc[-1] + 0.3, h.train_check_loss.iloc[-1], 'training people', fontsize=8.5, color=C['ink2'], va='center')
    ax.text(h.epoch.iloc[-1] + 0.3, h.validation_loss.iloc[-1], 'validation people', fontsize=8.5, color=C['ink2'], va='center')
    ax.set_xlim(0.5, h.epoch.max() + 4.5)
    ax.set_xticks(range(1, int(h.epoch.max()) + 1, 2))
    ax.set_title(f"Seed {run['summary']['seed']}  ·  {run['summary']['epochs_run']} epochs, {run['summary']['minutes']:.0f} min")
    ax.set_xlabel('Epoch')
for ax, run in zip(axes[0], final_runs):                  # label the kept epoch at the top, once the y-range is final
    ax.text(run['summary']['best_epoch'] - 0.3, ax.get_ylim()[1], f"kept: epoch {run['summary']['best_epoch']}",
            ha='right', va='top', fontsize=8.5, color=C['ink2'])
axes[0, 0].set_ylabel('Equal-weight loss over 10 tasks')
title(fig, 'Step 8 · Loss curves: the gap between seen and new people shows how much the model memorises')
save(fig, 'step8_loss_curves')
plt.show()
display(pd.DataFrame([{k: run['summary'][k] for k in ('seed', 'best_epoch', 'epochs_run', 'best_val_loss', 'minutes')}
                      for run in final_runs]).set_index('seed').round(3))

In [ ]:
# Step 8c · Per-task learning curves (average of the 3 seeds): every task has its own story
curves = pd.concat([run['history'].assign(seed=run['summary']['seed']) for run in final_runs])
mean_curve = curves.groupby('epoch').mean(numeric_only=True)
fig, axes = plt.subplots(2, 5, figsize=(15, 6.2), layout='constrained')
for ax, task in zip(axes.flat, TASKS):
    if task == 'sex':
        ax.plot(mean_curve.index, 100 * mean_curve['train_check_sex_balanced_accuracy'], color=C['spec'], alpha=0.45)
        ax.plot(mean_curve.index, 100 * mean_curve['validation_sex_balanced_accuracy'], color=C['spec'])
        ax.set_ylabel('Balanced accuracy %')
    else:
        ax.plot(mean_curve.index, 100 * mean_curve[f'train_check_skill_{task}'], color=C['spec'], alpha=0.45)
        ax.plot(mean_curve.index, 100 * mean_curve[f'validation_skill_{task}'], color=C['spec'])
        ax.axhline(0, color=C['ref'], lw=1)
        ax.set_ylabel('Skill vs constant %')
    ax.set_title(NAME[task], fontsize=9.5)
    ax.set_xlabel('Epoch')
fig.legend(handles=[Line2D([], [], color=C['spec'], alpha=0.45, label='training people (check group)'),
                    Line2D([], [], color=C['spec'], label='validation people (new)'),
                    Line2D([], [], color=C['ref'], lw=1, label='constant guess (0%)')],
           loc='outside lower center', ncols=3)
title(fig, 'Step 8 · Per-task curves: appearance tasks learn; physiological tasks stay near the constant guess')
save(fig, 'step8_task_curves')
plt.show()

**Reading Step 8.**

* **Loss curves.**
  * The validation loss is best early (epochs 3, 7 and 5) and then drifts, while the training-people loss keeps falling. The heads have about 250,000 numbers, so they start memorising the training people after a few epochs.
  * Early stopping caught this in every seed and kept the best epoch. Averaging the 3 seeds smooths what is left.
* **Per-task curves.**
  * Age, BMI and sex rise quickly on new people and then level off; age reaches about +55% on validation.
  * Saturation, temperature, HbA1c, cholesterol, respiratory rate, stiffness and stress stay around 0%, the constant guess, while their training-people curves keep rising. That gap is memorisation of person-specific noise, and early stopping limits it.

## Step 9 · Validation check: overfitting and predictions

The final model is the average of the 3 seeds. Before touching the test people, we check it on **training** and **validation** people:

* **Overfitting check.** We compare the error on the 420 training people (no augmentation) with the 90 validation people. A large gap means the model memorised training faces.
* **Predicted vs true.** Do the predictions follow the true values, or do they collapse to the middle?

In [ ]:
# Step 9a · Ensemble predictions (average of the 3 seeds) on training, validation and test videos
final_models = [load_model(run) for run in final_runs]
ensemble = {}
for part in ('train', 'validation', 'test'):
    preds = [predict(m, part) for m in final_models]
    ensemble[part] = (np.mean([p[0] for p in preds], axis=0), np.mean([p[1] for p in preds], axis=0))
train_scores = score(idx['train'], *ensemble['train'])
val_scores = score(idx['validation'], *ensemble['validation'])

# Compare the ERRORS themselves (MAE). Skill would mix in how spread out each group's labels are:
# e.g. the training group happens to contain more older people, which changes its constant-guess error.
gap = pd.DataFrame({'training MAE': [train_scores[f'mae_{t}'] for t in TARGETS],
                    'validation MAE': [val_scores[f'mae_{t}'] for t in TARGETS]}, index=TARGETS)
gap['validation ÷ training'] = gap['validation MAE'] / gap['training MAE']
gap = gap.sort_values('validation ÷ training')
in_sd = gap[['training MAE', 'validation MAE']].div(pd.Series(SD, index=TARGETS).loc[gap.index], axis=0)   # common scale
fig, ax = plt.subplots(figsize=(9.5, 4.8), layout='constrained')
y = np.arange(len(gap))
ax.hlines(y, in_sd['training MAE'], in_sd['validation MAE'], color=C['spec'], alpha=0.3, lw=2)
ax.scatter(in_sd['training MAE'], y, s=46, color=C['spec'], alpha=0.45, edgecolor=C['surface'], linewidth=1.4, zorder=3,
           label='training people (seen)')
ax.scatter(in_sd['validation MAE'], y, s=46, color=C['spec'], edgecolor=C['surface'], linewidth=1.4, zorder=3,
           label='validation people (new)')
for yi, ratio in zip(y, gap['validation ÷ training']):
    ax.text(max(in_sd.iloc[yi]) + 0.012, yi, f'{ratio:.2f}×', va='center', fontsize=8.5, color=C['ink2'])
ax.set_yticks(y, [NAME[t] for t in gap.index])
ax.set_xlabel("Mean absolute error in units of the target's training SD  ·  label = validation ÷ training")
ax.set_xlim(0, in_sd.to_numpy().max() * 1.15)
ax.legend(loc='lower right')
title(fig, f"Step 9 · Overfitting check: errors on new people are at most {gap['validation ÷ training'].max() - 1:.0%} higher than on seen people")
save(fig, 'step9_overfitting_gap')
plt.show()
display(gap.rename(index=NAME).round(3))
print(f"sex balanced accuracy: training {train_scores['sex_balanced_accuracy']:.1%} | validation {val_scores['sex_balanced_accuracy']:.1%}")

In [ ]:
# Step 9b · Predicted vs true on validation, improved CNN vs baseline CNN
base_val = pd.read_csv(BASELINE / 'validation_predictions.csv', dtype={'patient_id': str}).set_index('key').loc[data.key[idx['validation']]]
fig, axes = plt.subplots(3, 3, figsize=(11, 11), layout='constrained')
for ax, (j, t) in zip(axes.flat, enumerate(TARGETS)):
    truth = data[t].to_numpy(float)[idx['validation']]
    ours, theirs = ensemble['validation'][0][:, j], base_val[f'pred_{t}'].to_numpy(float)
    values = np.r_[truth, ours, theirs]
    pad = (values.max() - values.min()) * 0.05
    lim = (values.min() - pad, values.max() + pad)
    ax.plot(lim, lim, color=C['ref'], lw=1)
    ax.scatter(truth, theirs, s=14, color=C['cnn'], alpha=0.6, edgecolor='none')
    ax.scatter(truth, ours, s=14, color=C['spec'], alpha=0.6, edgecolor='none')
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect('equal')
    ax.set_title(f'{NAME[t]} ({UNIT[t]})')
    ax.text(0.97, 0.03, f'spread kept\nimproved {np.std(ours) / np.std(truth):.0%} · baseline {np.std(theirs) / np.std(truth):.0%}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, color=C['ink2'])
for ax in axes[-1]:
    ax.set_xlabel('True value')
for ax in axes[:, 0]:
    ax.set_ylabel('Predicted value')
fig.legend(handles=[Line2D([], [], marker='o', ls='', color=C['spec'], label='Improved CNN'),
                    Line2D([], [], marker='o', ls='', color=C['cnn'], label='Baseline CNN'),
                    Line2D([], [], color=C['ref'], lw=1, label='perfect prediction')], loc='outside lower center', ncols=3)
title(fig, 'Step 9 · Predicted vs true on validation people')
save(fig, 'step9_predicted_vs_true')
plt.show()

**Reading Step 9.**

* **Overfitting chart.** Every dot pair compares the error on people the model has seen (faint) with new people (solid). A model that memorised its training people would have a much smaller error on them.
* **The final model barely memorises.** Its errors on new people are at most 26% higher than on seen people (temperature), and for age and HbA1c they are even slightly lower.
  * Age: 1.55 years on new people vs 1.63 on seen people.
  * BMI: 2.41 vs 2.18 kg/m².
  * Sex balanced accuracy: 96.9% vs 99.4%.
  * Compare the earlier specialist pipeline, whose BMI model had an error of 0.37 on seen people and 2.36 on new people.
* **Why errors, not skill?** Skill also depends on how spread out each group's labels are; the training group, for example, contains more older people. Comparing the errors directly avoids that trap.
* **Spread kept** (in the scatter plots) is the spread of the predictions divided by the spread of the true values.
  * Close to 0% means the model mostly predicts the middle value. That is what happens for the physiological targets, for both models.
  * Age keeps all of its spread (101%, vs 81% for the baseline), BMI 56% (vs 41%). The improved model follows the older validation people much better (scatter plot for age).

## Step 10 · Test evaluation: scored once

Everything is now frozen: the model choice, the settings and the 3 trained seeds. The **90 test people** are scored once. We compare three things on the *same* people:

* the **improved CNN**;
* the **baseline CNN** (its saved test predictions);
* the **constant guess** (training median).

**How to read the numbers.**

* **Skill** = 1 − MAE<sub>model</sub> / MAE<sub>constant</sub>. 0% means no better than the constant guess; higher is better. Skill has no units, so all targets share one axis.
* The **95% intervals** come from a *participant bootstrap*. We resample the 90 test people 2,000 times and see how much the score moves.
* Sex is a classification task, so it is reported separately.

*Note: these test people were already used in earlier project experiments, so this is a fair comparison between models, not an untouched final confirmation.*

In [ ]:
# Step 10a · Test metrics with participant-bootstrap intervals, improved vs baseline vs constant
base_test = pd.read_csv(BASELINE / 'test_predictions.csv', dtype={'patient_id': str}).set_index('key').loc[data.key[idx['test']]]
ids = data.patient_id.to_numpy()[idx['test']]
rng = np.random.default_rng(20261015)
people_list = np.unique(ids)
draws = rng.integers(0, len(people_list), (SETTINGS['bootstrap'], len(people_list)))


def per_person_error(truth, pred):
    return pd.Series(np.abs(truth - pred), index=ids).groupby(level=0).mean().loc[people_list].to_numpy()


rows = []
for j, t in enumerate(TARGETS):
    truth = data[t].to_numpy(float)[idx['test']]
    assert np.allclose(truth, base_test[f'true_{t}'].to_numpy(float)), 'baseline labels differ'
    e_ours = per_person_error(truth, ensemble['test'][0][:, j])
    e_base = per_person_error(truth, base_test[f'pred_{t}'].to_numpy(float))
    e_const = per_person_error(truth, np.full(len(truth), CONSTANT[t]))
    boot = lambda e: 1 - e[draws].mean(1) / e_const[draws].mean(1)
    rows.append({'target': t, 'improved MAE': e_ours.mean(), 'baseline MAE': e_base.mean(), 'constant MAE': e_const.mean(),
                 'improved skill %': 100 * (1 - e_ours.mean() / e_const.mean()),
                 'improved low %': 100 * np.quantile(boot(e_ours), 0.025), 'improved high %': 100 * np.quantile(boot(e_ours), 0.975),
                 'baseline skill %': 100 * (1 - e_base.mean() / e_const.mean()),
                 'baseline low %': 100 * np.quantile(boot(e_base), 0.025), 'baseline high %': 100 * np.quantile(boot(e_base), 0.975),
                 'improved − baseline MAE, 95% interval': '[{:+.3f}, {:+.3f}]'.format(*np.quantile((e_ours - e_base)[draws].mean(1), [0.025, 0.975]))})
test_table = pd.DataFrame(rows).set_index('target')

val_order = sorted(TARGETS, key=lambda t: val_scores[f'skill_{t}'])        # order fixed by VALIDATION, not test
fig, axes = plt.subplots(2, 1, figsize=(9.5, 6.6), height_ratios=[2.2, 7], layout='constrained')
for ax, (panel, members) in zip(axes, [('Appearance targets', ['age', 'bmi']),
                                       ('Physiological targets', [t for t in TARGETS if t not in ('age', 'bmi')])]):
    order = [t for t in val_order if t in members]
    for model, colour, offset in [('improved', C['spec'], 0.16), ('baseline', C['cnn'], -0.16)]:
        part = test_table.loc[order]
        yy = np.arange(len(order)) + offset
        ax.hlines(yy, part[f'{model} low %'], part[f'{model} high %'], color=colour, lw=2)
        ax.scatter(part[f'{model} skill %'], yy, s=46, color=colour, edgecolor=C['surface'], linewidth=1.4, zorder=3,
                   label=f"{model.capitalize()} CNN: dot = test skill, line = 95% interval")
    ax.axvline(0, color=C['ink2'], lw=1)
    ax.set_yticks(range(len(order)), [NAME[t] for t in order])
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.set_title(panel)
    ax.annotate('0 = constant guess', (0, len(order) - 0.4), xytext=(4, -2), textcoords='offset points',
                color=C['ink2'], fontsize=8.5, va='top')
axes[1].set_xlabel('Test error reduction vs constant guess (%)  ·  left of 0 = worse than always guessing the training median')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncols=2)
clear = [NAME[t] for t in TARGETS if test_table.loc[t, 'improved low %'] > 0]
title(fig, 'Step 10 · Test: the improved CNN clearly beats the constant guess for '
      + (' and '.join(clear) if clear else 'no target') + '; physiological targets stay near 0')
save(fig, 'step10_test_skill')
plt.show()
display(test_table.rename(index=NAME).round(3))

In [ ]:
# Step 10b · Recorded sex on the test people: accuracy, balanced accuracy, AUC and ROC curves
def auc(y, p):
    """Area under the ROC curve via ranks (same value as scikit-learn's roc_auc_score)."""
    ranks = pd.Series(p).rank().to_numpy()
    n1, n0 = y.sum(), len(y) - y.sum()
    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def roc_points(y, p):
    order = np.argsort(-p)
    tpr = np.r_[0, np.cumsum(y[order]) / y.sum()]
    fpr = np.r_[0, np.cumsum(1 - y[order]) / (len(y) - y.sum())]
    return fpr, tpr


truth = data.sex_label.to_numpy(float)[idx['test']]
ok = ~np.isnan(truth)
majority = float(data.loc[idx['train']].drop_duplicates('patient_id').sex_label.dropna().mode()[0])
probabilities = {'Improved CNN': ensemble['test'][1], 'Baseline CNN': base_test['pred_prob_male'].to_numpy(float)}
sex_rows = []
fig, ax = plt.subplots(figsize=(5.4, 5), layout='constrained')
for label, p in probabilities.items():
    y, pp = truth[ok], p[ok]
    guess = (pp >= 0.5).astype(float)
    sex_rows.append({'model': label, 'accuracy %': 100 * np.mean(guess == y),
                     'balanced accuracy %': 100 * np.mean([np.mean(guess[y == c] == c) for c in (0, 1)]),
                     'AUC': auc(y, pp), 'wrong recordings': f'{int((guess != y).sum())} of {len(y)}'})
    fpr, tpr = roc_points(y, pp)
    ax.plot(fpr, tpr, color=C['spec'] if label.startswith('Improved') else C['cnn'], label=f"{label} (AUC {auc(y, pp):.3f})")
sex_rows.append({'model': 'Constant guess (majority: F)', 'accuracy %': 100 * np.mean(truth[ok] == majority),
                 'balanced accuracy %': 50.0, 'AUC': 0.5, 'wrong recordings': f'{int((truth[ok] != majority).sum())} of {ok.sum()}'})
ax.plot([0, 1], [0, 1], color=C['ref'], lw=1)
ax.set_xlabel('False-positive rate')
ax.set_ylabel('True-positive rate')
ax.legend(loc='lower right')
title(fig, 'Step 10 · Recorded sex (test): ROC curves')
save(fig, 'step10_sex_roc')
plt.show()
sex_table = pd.DataFrame(sex_rows).set_index('model')
display(sex_table.round(3))

In [ ]:
# Step 10c · Who gets the error? Age by age group, and saving every test number to CSV
age_rows = []
for band, in_group in [('18–24', lambda a: (a >= 18) & (a <= 24)), ('older than 24', lambda a: a > 24)]:
    rows_in = idx['test'][in_group(data.age.to_numpy()[idx['test']])]
    true_age, people_ids = data.age.to_numpy(float)[rows_in], data.patient_id.to_numpy()[rows_in]
    position_in_test = np.searchsorted(idx['test'], rows_in)
    age_rows.append({'age group': band, 'people': len(np.unique(people_ids)),
                     'improved MAE (years)': participant_mae(true_age, ensemble['test'][0][position_in_test, TARGETS.index('age')], people_ids),
                     'baseline MAE (years)': participant_mae(true_age, base_test['pred_age'].to_numpy(float)[position_in_test], people_ids),
                     'constant MAE (years)': participant_mae(true_age, np.full(len(true_age), CONSTANT['age']), people_ids)})
age_table = pd.DataFrame(age_rows).set_index('age group')
display(age_table.round(2))
age_table.to_csv(RESULTS / 'test_age_groups.csv')

predictions = data.loc[idx['test'], ['key', 'patient_id', 'step']].reset_index(drop=True)
for j, t in enumerate(TARGETS):
    predictions[f'true_{t}'] = data[t].to_numpy()[idx['test']]
    predictions[f'pred_{t}'] = ensemble['test'][0][:, j]
predictions['true_sex'] = data.sex_label.to_numpy(float)[idx['test']]
predictions['pred_prob_male'] = ensemble['test'][1]
predictions.to_csv(RESULTS / 'test_predictions.csv', index=False)
test_table.to_csv(RESULTS / 'test_metrics.csv')
sex_table.to_csv(RESULTS / 'test_sex_metrics.csv')
pd.DataFrame([{**{'part': 'train'}, **train_scores}, {**{'part': 'validation'}, **val_scores}]).to_csv(RESULTS / 'train_validation_scores.csv', index=False)
print('Saved: test_predictions.csv, test_metrics.csv, test_sex_metrics.csv, test_age_groups.csv, train_validation_scores.csv in',
      RESULTS.relative_to(PROJECT))

**Reading Step 10** (the saved test run).

* **Age: a clear gain.** The improved model's test error is 1.63 years, against 2.61 for the baseline and 3.48 for the constant guess: skill +53% vs +25%.
  * Its 95% interval (+26% to +68%) excludes zero; the baseline's does not.
  * The direct improved-minus-baseline difference is −1.52 to −0.55 years, which also excludes zero. Unlike the first run of this notebook, the gain over the baseline is larger than the noise of 90 test people.
* **BMI.** 2.81 vs 2.98 kg/m² (skill +18% vs +13%). Both intervals exclude zero, but the difference between the models ([−0.46, +0.10]) does not, so this is at most a small gain.
* **Physiological targets.** The improved model does not beat the constant guess on any of them: every interval includes zero.
  * The improved model is usually closer to 0, so it is less harmful than the baseline: saturation −5% vs −9%, and stiffness −3% vs −7%.
  * Stiffness is the one physiological target where the improved-vs-baseline difference excludes zero, and it is still not better than the constant guess.
* **Recorded sex.** Accuracy is 97.8% (4 of 180 recordings wrong) vs 97.2% for the baseline (5 wrong); AUC 0.997 vs 0.994. That is practically equal; both are close to the ceiling.
* **Age groups.**
  * **18–24-year-olds (83 people):** error 1.28 years, against 2.15 for the baseline and 1.35 for the constant guess. Unlike the baseline and the first run, it is slightly better than guessing "20" for young people.
  * **The 7 older test people:** 5.75 years, against 8.10 for the baseline. That is better, but still large: the training data has few older faces (Step 2a).

## Step 11 · Summary

In [ ]:
# Step 11 · One summary table: what the improved CNN achieves vs the baseline and the constant guess
summary = test_table[['improved skill %', 'baseline skill %', 'improved MAE', 'baseline MAE', 'constant MAE']].copy()
summary['improved interval excludes 0'] = (test_table['improved low %'] > 0) | (test_table['improved high %'] < 0)
summary.index = summary.index.map(NAME)
display(summary.round(2))
total_minutes = sum(run['summary']['minutes'] for run in final_runs)
devices = ' + '.join(sorted({run['summary'].get('device', 'cpu').upper() for run in final_runs}))
print(f"Final model: frozen DINOv2 + task-specific heads, {len(final_runs)} seeds, "
      f"{total_minutes:.0f} minutes of training in total (on {devices})")

**What to take away** (the numbers are in the tables above; the full story is in `report/ML_PROCESS.md`):

1. **The improved model is a frozen, pretrained transformer with small trained heads.** Because the backbone is frozen, only the heads learn, so the model memorises much less than a fully trained network. Early stopping handles the rest.
2. **It is clearly better where the face carries information.**
   * Age: test error 1.63 vs 2.61 years (skill +53% vs +25%). The gain over the baseline is beyond the noise of 90 people.
   * BMI: +18% vs +13%, within the noise.
   * Sex: 97.8% vs 97.2% accuracy, both near the ceiling.
3. **Physiological targets stay at the constant-guess level for both models.** This is mainly an *input* limitation: 8 still frames, about 8.6 s apart, cannot show a pulse or a breath (Step 2g).
4. **Colour signals from every frame (rPPG) can show the pulse.** It is visible at rest when the face patches agree (Step 2h); breathing was not recovered. Step 13 gives these signals to the physiological targets.
5. **Overfitting is under control.** Errors on new people are at most 26% higher than on seen people.
6. **Honest evaluation.** Every choice used validation people only. The 90 test people were scored once for this run (Steps 12d and 13d score them again, only to compare finished models).

**Next steps** (proposals, not run):

* **Physiology needs a different input.** Two options:
  * a synchronised contact PPG next to the video, to settle whether the reference pulse was measured at the same time as the video;
  * the planned ECG, PPG and BIA data.
* **Fine-tuning the last DINOv2 blocks** with a small learning rate, guarded against memorisation.
* **More older participants,** or age-balanced sampling.
* **Cross-validation** over all 510 non-test people, instead of one 90-person validation group: **done in Step 12** below.
* **The colour signals with 3 seeds,** so that gains of a few points can be told apart from chance: **done in Step 13** below, together with a time-series network on the full traces.

## Step 12 · Cross-validation: the final set-up judged on 510 people instead of 90

Steps 8–10 judged the final set-up on **one group of 90 validation people**, only 5 of them older than 24. Scores on 90 people wobble by chance. **Cross-validation** judges the same set-up on all **510 non-test people** (the old training and validation groups together):

```
510 non-test people, dealt into 5 groups of 102 (both sessions of a person stay together)

round 1:   [ JUDGED ][ train  ][ train  ][ train  ][ train  ]
round 2:   [ train  ][ JUDGED ][ train  ][ train  ][ train  ]
round 3:   [ train  ][ train  ][ JUDGED ][ train  ][ train  ]
round 4:   [ train  ][ train  ][ train  ][ JUDGED ][ train  ]
round 5:   [ train  ][ train  ][ train  ][ train  ][ JUDGED ]
```

* Each round trains on 408 people and judges the other 102, exactly as Step 8 trained on 420 and judged 90. The settings are the same, and each round keeps the epoch with the best loss on its judged group, like Step 6b's early stopping.
* After 5 rounds, **every one of the 510 people has been judged once**, by a model that never saw them. That includes **48 people older than 24**, instead of 5.
* The **90 test people stay locked away** until Step 12d.

| Part | Question |
|---|---|
| 12a | The 5 groups |
| 12b | A shortcut for the frozen DINOv2, and a check that it gives the same answers |
| 12c | The final set-up with 3 seeds in all 5 rounds; then, for each target, the heads or a simpler Ridge model? |
| 12d | The final model, trained on all 510 people; one more look at the test people |
| 12e | The errors with and without cross-validation, side by side |

**The rules, fixed before running**

1. **Per target:** the final set-up (frozen DINOv2 + task-specific heads) is the default for every target. **Ridge** replaces it for one target only if its error is lower **and** the 95% interval of the paired difference over the 510 people excludes zero.
2. **Final model:** the chosen option for each target, retrained on all 510 people with 3 seeds. Nobody is left to judge, so training stops at the median kept epoch of its cross-validation rounds. The test people are scored once more **for information**; nothing is chosen on them.

**Run time.** About 15 minutes the first time on this laptop's GPU: about 5 to compute the DINOv2 numbers (Step 12b), 3 for the 18 training rounds and 8 for Ridge. The rounds are saved in `results/cv/` and reused afterwards, like the runs of Step 8, so *Run All* stays fast. `RETRAIN = True` trains them again.

### 12a · The 5 groups

The 510 people are dealt into 5 groups of 102, like cards: first all people aged 18–24 recorded as F, then those recorded as M, then the older ones in the same way. So every group gets about the same number of older people and the same mix of recorded sex. Both sessions of a person always land in the same group.

Each round learns its own scaling and constant guess (Step 4) from its 408 training people only, so nothing about the judged people leaks into their model.

In [ ]:
# Step 12a · Deal the 510 non-test people into 5 groups, balanced by age group and recorded sex
CV_DIR = RESULTS / 'cv'                                            # everything of Step 12 is saved here
CV_RUNS = CV_DIR / 'runs'
CV_RUNS.mkdir(parents=True, exist_ok=True)
N_FOLDS = 5
NONTEST = np.flatnonzero(data.split.ne('test').to_numpy())          # the 1,020 videos of the 510 non-test people
people_cv = data.loc[NONTEST].groupby('patient_id').agg(age=('age', 'first'), sex=('sex', 'first'), split=('split', 'first'))
people_cv['age group'] = np.where(people_cv.age > 24, 'older than 24', '18–24')
deal = np.random.default_rng(2026)
order = [p for _, group in people_cv.groupby(['age group', 'sex']) for p in deal.permutation(group.index.to_numpy())]
people_cv.loc[order, 'fold'] = np.arange(len(order)) % N_FOLDS      # like dealing cards, one kind of person after another
people_cv['fold'] = people_cv.fold.astype(int)
FOLD = data.patient_id.map(people_cv.fold)                          # group of every video (NaN for the test people)
FOLD_ROWS = {k: np.flatnonzero(FOLD.eq(k).to_numpy()) for k in range(N_FOLDS)}         # the people judged in round k
FIT_ROWS = {k: np.setdiff1d(NONTEST, FOLD_ROWS[k]) for k in range(N_FOLDS)}            # the people trained on in round k
FOLD_HASH = hashlib.sha256(people_cv.fold.sort_index().to_csv().encode()).hexdigest()[:12]   # saved rounds must match it
people_cv[['fold']].to_csv(CV_DIR / 'folds.csv')
assert not people_cv.index.isin(data.patient_id.to_numpy()[idx['test']]).any(), 'a test person is in a group'
assert all(np.intersect1d(data.patient_id.to_numpy()[FOLD_ROWS[k]], data.patient_id.to_numpy()[FIT_ROWS[k]]).size == 0
           for k in range(N_FOLDS)), 'a person is both trained on and judged in one round'


def fit_statistics(rows):
    """What Step 4 learned from the 420 training people, learned from any group of people instead:
    target mean and spread (cleaned labels), the constant guess (median of the recorded labels) and the signal scaling."""
    mu = clean.loc[rows, TARGETS].mean().to_numpy()
    sd = clean.loc[rows, TARGETS].std().to_numpy()
    sig_mu = np.nanmean(SIGNAL_FEATURES[rows], 0)
    sig_sd = np.nanstd(SIGNAL_FEATURES[rows], 0)
    sig_sd[sig_sd < 1e-8] = 1.0
    return dict(mu=mu, sd=sd, constant=data.loc[rows, TARGETS].median(),
                y=torch.tensor((clean[TARGETS].to_numpy(float) - mu) / sd, dtype=torch.float32),
                sig=torch.tensor(np.nan_to_num((SIGNAL_FEATURES - sig_mu) / sig_sd), dtype=torch.float32))


FOLD_STATS = {k: fit_statistics(FIT_ROWS[k]) for k in range(N_FOLDS)}   # each round learns its scaling from its own 408 people
ORIGINAL_STATS = fit_statistics(idx['train'])                           # the 420 training people of Steps 4–10
assert np.array_equal(ORIGINAL_STATS['mu'], MU) and np.array_equal(ORIGINAL_STATS['sd'], SD)
CV_CONSTANT = np.full((len(data), len(TARGETS)), np.nan)                # each judged person's constant guess comes from
for k in range(N_FOLDS):                                                # the people their round trained on
    CV_CONSTANT[FOLD_ROWS[k]] = FOLD_STATS[k]['constant'].to_numpy()

fold_table = people_cv.groupby('fold').agg(
    people=('age', 'size'), **{'older than 24': ('age', lambda a: int((a > 24).sum()))},
    female=('sex', lambda s: int(s.eq('F').sum())), male=('sex', lambda s: int(s.eq('M').sum())),
    **{'mean age': ('age', 'mean'), 'from the old training group': ('split', lambda s: int(s.eq('train').sum())),
       'from the old validation group': ('split', lambda s: int(s.eq('validation').sum()))})
fold_table.index = [f'group {k + 1}' for k in fold_table.index]
fold_table.loc['all 510'] = fold_table.sum(numeric_only=True)
fold_table.loc['all 510', 'mean age'] = people_cv.age.mean()
fold_table = fold_table.astype({c: int for c in fold_table.columns if c != 'mean age'})
display(fold_table.round(1))

fig, ax = plt.subplots(figsize=(9.5, 3.6), layout='constrained')
for r in range(N_FOLDS):
    for k in range(N_FOLDS):
        judged = r == k
        ax.add_patch(plt.Rectangle((k + 0.04, N_FOLDS - 1 - r + 0.08), 0.92, 0.84, lw=0,
                                   color=C['spec'] if judged else C['grid']))
        ax.text(k + 0.5, N_FOLDS - 0.5 - r, f"judged\n{fold_table.iloc[k]['people']:.0f} people · {fold_table.iloc[k]['older than 24']:.0f} older"
                if judged else 'trained on', ha='center', va='center', fontsize=8.5,
                color='white' if judged else C['ink2'])
ax.set_xlim(0, N_FOLDS)
ax.set_ylim(0, N_FOLDS)
ax.set_xticks(np.arange(N_FOLDS) + 0.5, [f'group {k + 1}' for k in range(N_FOLDS)])
ax.set_yticks(np.arange(N_FOLDS) + 0.5, [f'round {r}' for r in range(N_FOLDS, 0, -1)])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)
title(fig, f'Step 12a · 5 rounds: every one of the 510 non-test people is judged once, by a model that never saw them')
save(fig, 'step12a_cross_validation_rounds')
plt.show()
print(f'{len(people_cv)} people, {len(NONTEST)} videos; groups saved to {(CV_DIR / "folds.csv").relative_to(PROJECT)} (id {FOLD_HASH})')
print('The 90 test people are in no group.')

### 12b · A shortcut for the frozen DINOv2, and a check that it gives the same answers

Cross-validation trains 5 times as many models as Step 8. The frozen DINOv2 turns the same picture into the same 384 numbers every time. Step 6b already reused these numbers for the validation and test videos. For the training videos it could not, because augmentation changes their pictures in every epoch.

**The shortcut.**
* DINOv2's numbers are computed once for the unchanged frames of all 1,200 videos (used for judging).
* For every non-test video they are also computed for **8 random augmented versions**, made with Step 5's `augment()` (mirror and shift).
* Each training step then takes one of the 8 versions of each video and 4 random frames of it, as before. A run takes seconds instead of 1–2 minutes.
* The numbers are saved in `cache/dinov2_features.npz` (about 5 minutes to compute once).

**What changes.** Each video now has 8 fixed versions instead of a fresh random one in every epoch.

**The check.** The shortcut is run on the **original split** (420 training / 90 validation people) with the final set-up's 3 seeds, and compared with Step 8's runs. Single runs cannot match exactly, because the random draws differ. Averaged over the 3 seeds, the two should agree **within the seed-to-seed range** of Step 8's own runs.

In [ ]:
# Step 12b-1 · The shortcut: DINOv2's numbers for every frame, computed once (unchanged frames + 8 augmented versions)
FEATURE_CACHE = HERE / 'cache' / 'dinov2_features.npz'
AUG_VERSIONS = 8                                                     # augmented versions saved per non-test video
FEATURE_TAG = f'dinov2-ae1e99fc_{AUG_VERSIONS}-versions_shift8_strength0.15'   # the saved file's tag (weight file, versions
                                                                                # and the Step 5 settings when it was made)
encoder, _ = build_model(FINAL_CFG)
encoder.eval()


@torch.no_grad()
def dinov2_numbers(rows, version=None, batch=32):
    """DINOv2's 384 numbers for each of the 8 frames of the videos in rows: (videos, 8, 384).
    version=None: the frames as they are (for judging). version=v: augmented version v, made with Step 5's augment()
    (mirror + shift). Each version is a fixed random draw, the same on every rerun."""
    if version is not None:
        set_seed(1000 + version)
    parts = []
    for i in range(0, len(rows), batch):
        x = X[torch.from_numpy(rows[i:i + batch])].float().div_(255)
        if version is not None:
            x = augment(x)
        videos = x.shape[0]
        x = ((x.flatten(0, 1).to(DEVICE) - encoder.mean) / encoder.std).contiguous(memory_format=torch.channels_last)
        parts.append(encoder.backbone(x).view(videos, 8, -1).cpu())      # the same steps as MultiTaskCNN.encode()
    return torch.cat(parts).numpy()


def load_or_compute_features():
    keys, nontest_keys = data.key.to_numpy().astype(str), data.key.to_numpy()[NONTEST].astype(str)
    if FEATURE_CACHE.is_file():
        with np.load(FEATURE_CACHE, allow_pickle=False) as z:
            if str(z['tag']) == FEATURE_TAG and np.array_equal(z['keys'], keys) and np.array_equal(z['nontest_keys'], nontest_keys):
                print(f'saved DINOv2 numbers reused: {FEATURE_CACHE.relative_to(PROJECT)}')
                return {name: z[name] for name in ('plain', 'geometric')}
    started = time.time()
    with keep_screen_on():
        plain = dinov2_numbers(np.arange(len(data)))                              # all 1,200 videos, frames unchanged
        geometric = np.stack([dinov2_numbers(NONTEST, v).astype(np.float16) for v in range(AUG_VERSIONS)])   # half precision
    np.savez(FEATURE_CACHE, tag=np.array(FEATURE_TAG), keys=keys, nontest_keys=nontest_keys, plain=plain, geometric=geometric)
    frames_done = plain.shape[0] * 8 + AUG_VERSIONS * len(NONTEST) * 8
    print(f'computed DINOv2 numbers for {frames_done:,} frames in {(time.time() - started) / 60:.1f} min; '
          f'saved {FEATURE_CACHE.relative_to(PROJECT)} ({FEATURE_CACHE.stat().st_size / 1e6:.0f} MB)')
    return dict(plain=plain, geometric=geometric)


FEATURES = load_or_compute_features()
PLAIN_MEAN = torch.from_numpy(FEATURES['plain']).to(DEVICE).mean(1)                 # (1200, 384): encode() of all 8 frames
VERSIONS = torch.from_numpy(FEATURES['geometric']).to(DEVICE)                       # (8 versions, 1020 videos, 8 frames, 384)
NONTEST_POS = torch.full((len(data),), -1, dtype=torch.long)
NONTEST_POS[torch.from_numpy(NONTEST)] = torch.arange(len(NONTEST))
NONTEST_POS = NONTEST_POS.to(DEVICE)                                               # video row -> its place in VERSIONS
for part in ('validation', 'test'):                                                # same numbers as Step 6b's?
    step6b = ENCODED.get(('dinov2-pretrained', part, 32))
    if step6b is not None:
        print(f'{part} videos: largest difference from the numbers Step 6b computed: '
              f'{(PLAIN_MEAN[torch.from_numpy(idx[part]).to(DEVICE)] - step6b).abs().max().item():.1e}')
print('saved numbers:', {k: tuple(v.shape) for k, v in FEATURES.items()}, '= (versions,) videos x 8 frames x 384')

In [ ]:
# Step 12b-2 · Step 6b's training, for any group of people: train on one group, judge another after every epoch
class SavedFeatures(nn.Module):
    """Stands in for the frozen DINOv2 when its numbers come from the saved file: no pictures go through the model."""
    width = DINOv2Small.width


def build_judged_model(cfg):
    """build_model() of Step 6a, with the saved DINOv2 numbers in place of DINOv2 itself."""
    model = MultiTaskCNN(SavedFeatures(), SETTINGS['head_hidden'], SETTINGS['dropout'],
                         signal_dim=SIG.shape[1] if cfg.get('signals') else 0).to(DEVICE)
    return model, [{'params': list(model.parameters()), 'lr': SETTINGS['lr_heads']}]


@torch.no_grad()
def raw_outputs(model, rows, stats, batch=32):
    """Like predict() of Step 6b: all 8 frames of every video in rows, no augmentation; the 10 raw outputs."""
    model.eval()
    parts = []
    for i in range(0, len(rows), batch):
        r = torch.from_numpy(rows[i:i + batch])
        signals = stats['sig'][r].to(DEVICE) if model.signal is not None else None
        parts.append(model(None, signals, encoded=PLAIN_MEAN[r.to(DEVICE)]).cpu())
    return torch.cat(parts).numpy()


def fit(cfg, seed, fit_rows, stats, judge_rows=None, keep_epoch=None):
    """The training loop of Step 6b (same optimiser, schedule, batches, frames and augmentation) on any group of people.
    It always runs the full schedule: the raw outputs for judge_rows after every epoch are returned, so early stopping
    can be applied afterwards exactly as Step 6b does it. keep_epoch: return the model after that epoch instead."""
    set_seed(seed)
    model, groups = build_judged_model(cfg)
    optimizer = torch.optim.AdamW(groups, weight_decay=SETTINGS['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SETTINGS['epochs'])
    rows = torch.from_numpy(fit_rows)
    assert not np.isin(fit_rows, idx['test']).any(), 'a test video in the training rows'
    outputs = []
    for epoch in range(1, SETTINGS['epochs'] + 1):
        model.train()
        for b, batch in enumerate(rows[torch.randperm(len(rows))].split(SETTINGS['batch_videos'])):
            if MAX_BATCHES is not None and b >= MAX_BATCHES:
                break
            signals = stats['sig'][batch].to(DEVICE) if model.signal is not None else None
            version = torch.randint(AUG_VERSIONS, (len(batch),))      # one of the 8 saved augmented versions of each video,
            chosen_frames = torch.argsort(torch.rand(len(batch), 8), dim=1)[:, :SETTINGS['frames_train']]   # 4 random frames
            place = NONTEST_POS[batch.to(DEVICE)]
            shared = VERSIONS[version.to(DEVICE)[:, None], place[:, None], chosen_frames.to(DEVICE)].float().mean(1)
            loss = task_losses(model(None, signals, encoded=shared), stats['y'][batch].to(DEVICE), S[batch].to(DEVICE)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        if judge_rows is not None:
            outputs.append(raw_outputs(model, judge_rows, stats))
        if epoch == keep_epoch:
            return model.eval()
    return outputs


def judged_loss(out, rows, stats):
    """Step 6b's selection score (all 10 task losses averaged with equal weight) on the judged people."""
    r = torch.from_numpy(rows)
    return task_losses(torch.from_numpy(out), stats['y'][r], S[r]).mean().item()


def early_stopping_epoch(losses, patience=SETTINGS['patience']):
    """The epoch Step 6b's early stopping keeps: the best so far, stopping after `patience` epochs without improvement."""
    best, best_epoch, waited = np.inf, 0, 0
    for epoch, loss in enumerate(losses, 1):
        if loss < best:
            best, best_epoch, waited = loss, epoch, 0
        else:
            waited += 1
            if waited >= patience:
                break
    return best_epoch


def judged_run(cfg, seed, group):
    """group 0..4: train on the 408 people outside that group, judge the group (one round of cross-validation).
    group 'original': train on the 420 training people and judge the 90 validation people, like Step 8.
    Saved in results/cv/runs/ and reused when the settings and the groups are the same (like train_run of Step 6b)."""
    if group == 'original':
        fit_rows, judge_rows, stats, tag, groups_id = idx['train'], idx['validation'], ORIGINAL_STATS, 'original-split', 'participant_split.csv'
    else:
        fit_rows, judge_rows, stats, tag, groups_id = FIT_ROWS[group], FOLD_ROWS[group], FOLD_STATS[group], f'round{group + 1}', FOLD_HASH
    name = f'{run_name(cfg, seed)}_{tag}'
    settings = {k: SETTINGS[k] for k in TRAINING_SETTINGS} | {'max_batches': MAX_BATCHES, 'groups': groups_id, 'input': FEATURE_TAG}
    folder = CV_RUNS / name
    if not RETRAIN and (folder / 'summary.json').is_file() and (folder / 'outputs.npy').is_file():
        summary = json.loads((folder / 'summary.json').read_text(encoding='utf-8'))
        if summary['settings'] == settings:
            return dict(summary=summary, outputs=np.load(folder / 'outputs.npy'), rows=judge_rows, stats=stats)
    started = time.time()
    with keep_screen_on():
        outputs = np.stack(fit(cfg, seed, fit_rows, stats, judge_rows)).astype(np.float32)
    losses = [judged_loss(out, judge_rows, stats) for out in outputs]
    best = early_stopping_epoch(losses)
    summary = dict(name=name, config=cfg, seed=seed, group=str(group), settings=settings, losses=losses, best_epoch=best,
                   best_loss=losses[best - 1], minutes=(time.time() - started) / 60, device=str(DEVICE))
    folder.mkdir(parents=True, exist_ok=True)
    np.save(folder / 'outputs.npy', outputs)
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"  trained {name}: kept epoch {best}, judged loss {losses[best - 1]:.4f}, {summary['minutes'] * 60:.0f} s", flush=True)
    return dict(summary=summary, outputs=outputs, rows=judge_rows, stats=stats)


def cv_runs(cfg, seed):
    """All 5 rounds of one set-up and seed."""
    return [judged_run(cfg, seed, k) for k in range(N_FOLDS)]


def predictions_of(runs_by_seed):
    """Predictions in original units for the judged videos, from the kept epoch of each run; seeds averaged like Step 9a.
    For 5 rounds this gives one out-of-round prediction for every non-test video."""
    reg = np.full((len(data), len(TARGETS)), np.nan)
    prob = np.full(len(data), np.nan)
    for k in range(len(runs_by_seed[0])):
        rows, regs, probs = runs_by_seed[0][k]['rows'], [], []
        for runs in runs_by_seed:
            run = runs[k]
            out = run['outputs'][run['summary']['best_epoch'] - 1]
            regs.append(out[:, :9] * run['stats']['sd'] + run['stats']['mu'])
            probs.append(1 / (1 + np.exp(-out[:, 9])))
        reg[rows], prob[rows] = np.mean(regs, 0), np.mean(probs, 0)
    return reg, prob

In [ ]:
# Step 12b-3 · The check: on the original split (420 training / 90 validation people), does the shortcut give the answers
# of Step 8? Single runs differ by their random draws, so the fair comparison is the average over the 3 seeds.
check_rows, check_runs = [], {}
for run in final_runs:
    seed = run['summary']['seed']
    new = check_runs[seed] = judged_run(FINAL_CFG, seed, 'original')
    check_rows.append({'seed': seed, 'Step 8: validation loss': run['summary']['best_val_loss'],
                       'shortcut: validation loss': new['summary']['best_loss'],
                       'difference': new['summary']['best_loss'] - run['summary']['best_val_loss'],
                       'kept epoch (Step 8 → shortcut)': f"{run['summary']['best_epoch']} → {new['summary']['best_epoch']}",
                       'seconds (Step 8 → shortcut)': f"{run['summary']['minutes'] * 60:.0f} → {new['summary']['minutes'] * 60:.0f}"})
check_table = pd.DataFrame(check_rows).set_index('seed')
display(check_table.round(4))
seed_range = np.ptp(check_table['Step 8: validation loss'])
average_difference = check_table['difference'].mean()
print(f"averaged over the {len(check_table)} seeds the shortcut differs by {average_difference:+.4f}; Step 8's own seeds span "
      f"{seed_range:.4f} -> " + ('within that range: Step 12 uses the shortcut' if abs(average_difference) <= seed_range
                                 else 'LARGER than that range: treat the results of Step 12 with care'))

shortcut_scores = score(idx['validation'], *[p[idx['validation']] for p in predictions_of([[r] for r in check_runs.values()])])
compare = pd.DataFrame({'Steps 8–9 (slow)': val_scores, 'shortcut': shortcut_scores}).loc[
    ['mae_age', 'mae_bmi', 'skill_age', 'skill_bmi', 'mean_skill', 'sex_balanced_accuracy']]
compare.index = ['Age MAE (years)', 'BMI MAE (kg/m²)', 'Age skill', 'BMI skill', 'Mean skill (9 numbers)', 'Sex balanced accuracy']
print('Final set-up, average of 3 seeds, the same 90 validation people:')
display(compare.round(3))

**Reading 12b** (the saved run, 19 October 2026).

* **The saved numbers are DINOv2's own.** For the validation and test videos, they equal the numbers Step 6b computed (difference exactly 0).
* **The shortcut agrees with the slow way.**
  * Single runs differ by +0.0017 to +0.0035. One run says little, because the random draws differ: Step 8's own 3 seeds already span 0.005 (0.305 to 0.310).
  * Averaged over the 3 seeds, the difference is +0.0024, inside that range.
  * The 3-seed average scores the 90 validation people almost the same: age error 1.575 vs 1.550 years, BMI 2.393 vs 2.410 kg/m², sex balanced accuracy 96.9% for both.
  * **A possible small bias.** All 3 runs score a little worse with the shortcut, and two of them keep an earlier epoch. With 3 runs this can be chance.
* **A run now takes about 13 seconds instead of 1–2 minutes.** That is what makes cross-validation cheap.

### 12c · Step 8 again in all 5 rounds, and for each target: the heads or Ridge?

**Step 8 again.** The final set-up is trained with 3 seeds in all 5 rounds, and the 3 seeds are averaged (Step 9a). This gives one prediction for every non-test person, from a model that never saw them.

**One choice per target (rule 1).** Steps 8–10 use one model for all 10 targets. A simpler model could still be better for one particular target, so rule 1 checks each target separately against **Ridge**:
* **Ridge** is a straight-line model on the same 384 DINOv2 numbers, with a penalty that keeps the line from bending to noise. It has no heads and no epochs. Each target gets its own penalty, chosen inside the training people of each round. For sex it is logistic regression, the yes/no version.
* **The test:** Ridge replaces the heads for a target only if its error is lower and the 95% interval of the paired difference excludes zero. "Paired" means both are compared on the same 510 people, person by person.

In [ ]:
# Step 12c-1 · Step 8 again: the final set-up with 3 seeds in all 5 rounds, scored on all 510 non-test people
CV_IDS = data.patient_id.to_numpy()[NONTEST]
CV_PEOPLE = np.unique(CV_IDS)
cv_draws = np.random.default_rng(20261019).integers(0, len(CV_PEOPLE), (SETTINGS['bootstrap'], len(CV_PEOPLE)))


def person_errors(truth, pred, ids=CV_IDS, people=CV_PEOPLE):
    """Absolute error averaged within each person first (both sessions), in a fixed order of people."""
    return pd.Series(np.abs(truth - pred), index=ids).groupby(level=0).mean().loc[people].to_numpy()


def cv_errors(reg):
    """Per-person errors of the model and of the constant guess, for every target (510 people each)."""
    return {t: (person_errors(data[t].to_numpy(float)[NONTEST], reg[NONTEST, j]),
                person_errors(data[t].to_numpy(float)[NONTEST], CV_CONSTANT[NONTEST, j])) for j, t in enumerate(TARGETS)}


def sex_counts(prob, rows=NONTEST):
    """Per person: right and all recordings among F, then among M (hidden labels count for neither)."""
    y = data.sex_label.to_numpy(float)[rows]
    guess = (prob[rows] >= 0.5).astype(float)
    table = pd.DataFrame({'id': data.patient_id.to_numpy()[rows], 'F right': (y == 0) & (guess == 0), 'F': y == 0,
                          'M right': (y == 1) & (guess == 1), 'M': y == 1})
    return table.groupby('id')[['F right', 'F', 'M right', 'M']].sum().loc[np.unique(table.id)].to_numpy(float)


def balanced_accuracy(counts):
    """Mean of the share right among F and among M recordings; counts: (..., people, 4)."""
    total = counts.sum(-2)
    return 0.5 * (total[..., 0] / total[..., 1] + total[..., 2] / total[..., 3])


def cv_score(reg, prob):
    """score() of Step 6b for all 510 people: per-person MAE on the RECORDED labels, skill against each person's constant
    guess (the median of the people their round trained on), and sex accuracy, balanced accuracy and AUC per recording."""
    result = {}
    for t, (e, e0) in cv_errors(reg).items():
        result[f'mae_{t}'], result[f'skill_{t}'] = e.mean(), 1 - e.mean() / e0.mean()
    y = data.sex_label.to_numpy(float)[NONTEST]
    ok = ~np.isnan(y)
    guess = (prob[NONTEST][ok] >= 0.5).astype(float)
    result['sex_accuracy'] = float(np.mean(guess == y[ok]))
    result['sex_balanced_accuracy'] = float(balanced_accuracy(sex_counts(prob)))
    result['sex_auc'] = auc(y[ok], prob[NONTEST][ok])
    result['mean_skill'] = float(np.mean([result[f'skill_{t}'] for t in TARGETS]))
    return result


cv_final = [cv_runs(FINAL_CFG, seed) for seed in SETTINGS['seeds']]
cv_final_pred = predictions_of(cv_final)
kept_epochs = [run['summary']['best_epoch'] for runs in cv_final for run in runs]
print(f'{len(kept_epochs)} rounds (3 seeds x 5 rounds); kept epochs {min(kept_epochs)} to {max(kept_epochs)}, '
      f'median {np.median(kept_epochs):g}')

In [ ]:
# Step 12c-2 · Ridge in the same 5 rounds; then, for each target, the heads (the default) or Ridge (rule 1)
RIDGE_ALPHAS = np.logspace(-1, 5, 25)          # penalties tried: larger = flatter, more cautious predictions
LOGISTIC_LAMBDAS = np.logspace(-2, 4, 19)
PLAIN_NP = PLAIN_MEAN.cpu().numpy().astype(np.float64)      # the 8-frame DINOv2 numbers of every video (Step 12b)
RIDGE_FILE = CV_DIR / 'ridge_predictions.npz'


def ridge_path(Xf, y, Xn, alphas):
    """Ridge regression for several penalties at once (via SVD). The 384 numbers are standardised on the fitting rows."""
    m, s = Xf.mean(0), Xf.std(0) + 1e-12
    U, sv, Vt = np.linalg.svd((Xf - m) / s, full_matrices=False)
    Uty = U.T @ (y - y.mean())
    return np.array([((Xn - m) / s) @ (Vt.T @ (sv / (sv ** 2 + a) * Uty)) + y.mean() for a in alphas])


def logistic(Xf, y, Xn, lam, iters=100):
    """Logistic regression with an L2 penalty (Newton steps); returns P(male) for the rows of Xn."""
    m, s = Xf.mean(0), Xf.std(0) + 1e-12
    A, An = np.c_[(Xf - m) / s, np.ones(len(Xf))], np.c_[(Xn - m) / s, np.ones(len(Xn))]
    penalty = lam * np.eye(A.shape[1])
    penalty[-1, -1] = 0.0                                    # the intercept is not penalised
    w = np.zeros(A.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(A @ w)))
        step = np.linalg.solve((A.T * (p * (1 - p))) @ A + penalty, A.T @ (p - y) + penalty @ w)
        w -= step
        if np.abs(step).max() < 1e-9:
            break
    return 1 / (1 + np.exp(-(An @ w)))


def ridge_fit_predict(fit_rows, new_rows):
    """Ridge per number and logistic regression for sex on the frozen DINOv2 numbers: no heads, no epochs.
    Each target's penalty is chosen by 5-fold person-grouped CV inside fit_rows (per-person MAE; log-loss for sex)."""
    Xf, Xn, ids = PLAIN_NP[fit_rows], PLAIN_NP[new_rows], data.patient_id.to_numpy()[fit_rows]
    people = np.unique(ids)
    inner = pd.Series(np.random.default_rng(0).permutation(len(people)) % 5, index=people).loc[ids].to_numpy()
    reg = np.zeros((len(new_rows), len(TARGETS)))
    for j, t in enumerate(TARGETS):
        y = clean[t].to_numpy(float)[fit_rows]
        ok = ~np.isnan(y)
        cv = np.zeros(len(RIDGE_ALPHAS))
        for g in range(5):
            a, h = ok & (inner != g), ok & (inner == g)
            cv += [participant_mae(y[h], p, ids[h]) for p in ridge_path(Xf[a], y[a], Xf[h], RIDGE_ALPHAS)]
        reg[:, j] = ridge_path(Xf[ok], y[ok], Xn, [RIDGE_ALPHAS[np.argmin(cv)]])[0]
    y = clean.sex_label.to_numpy(float)[fit_rows]
    ok = ~np.isnan(y)
    cv = []
    for lam in LOGISTIC_LAMBDAS:
        loss = 0.0
        for g in range(5):
            a, h = ok & (inner != g), ok & (inner == g)
            p = np.clip(logistic(Xf[a], y[a], Xf[h], lam), 1e-6, 1 - 1e-6)
            loss += -np.mean(y[h] * np.log(p) + (1 - y[h]) * np.log(1 - p))
        cv.append(loss)
    return reg, logistic(Xf[ok], y[ok], Xn, LOGISTIC_LAMBDAS[int(np.argmin(cv))])


def ridge_cv_predictions():
    """Ridge in the same 5 rounds (saved in results/cv/; reused when the groups and the DINOv2 numbers are the same)."""
    tag = f'{FOLD_HASH}_{FEATURE_TAG}'
    if not RETRAIN and RIDGE_FILE.is_file():
        with np.load(RIDGE_FILE, allow_pickle=False) as z:
            if str(z['tag']) == tag:
                return z['reg'], z['prob']
    reg, prob = np.full((len(data), len(TARGETS)), np.nan), np.full(len(data), np.nan)
    for k in range(N_FOLDS):
        reg[FOLD_ROWS[k]], prob[FOLD_ROWS[k]] = ridge_fit_predict(FIT_ROWS[k], FOLD_ROWS[k])
    np.savez(RIDGE_FILE, tag=np.array(tag), reg=reg, prob=prob)
    return reg, prob


DEFAULT = 'DINOv2 + task-specific heads (3 seeds)'
RIDGE = 'Ridge on the DINOv2 numbers'
OPTIONS = {DEFAULT: dict(cfg=FINAL_CFG, runs=cv_final, pred=cv_final_pred),
           RIDGE: dict(cfg=None, runs=None, pred=ridge_cv_predictions())}

default_errors, ridge_errors = cv_errors(cv_final_pred[0]), cv_errors(OPTIONS[RIDGE]['pred'][0])
default_sex, ridge_sex = sex_counts(cv_final_pred[1]), sex_counts(OPTIONS[RIDGE]['pred'][1])
choice_rows, CV_CHOICE = [], {}
for t in TASKS:
    if t == 'sex':                                   # higher balanced accuracy is better
        d, r = balanced_accuracy(default_sex), balanced_accuracy(ridge_sex)
        low, high = np.quantile(balanced_accuracy(ridge_sex[cv_draws]) - balanced_accuracy(default_sex[cv_draws]), [0.025, 0.975])
        ridge_better = r > d and low > 0
        row = {'heads (default)': f'{100 * d:.1f}% balanced accuracy', 'Ridge': f'{100 * r:.1f}%',
               'Ridge − heads, 95% interval': f'{100 * (r - d):+.1f} [{100 * low:+.1f}, {100 * high:+.1f}] points'}
    else:                                            # lower MAE is better
        (e, e0), er = default_errors[t], ridge_errors[t][0]
        low, high = np.quantile((er - e)[cv_draws].mean(1), [0.025, 0.975])
        ridge_better = er.mean() < e.mean() and high < 0
        row = {'heads (default)': f'MAE {e.mean():.3f} (skill {100 * (1 - e.mean() / e0.mean()):+.1f}%)',
               'Ridge': f'MAE {er.mean():.3f} (skill {100 * (1 - er.mean() / e0.mean()):+.1f}%)',
               'Ridge − heads, 95% interval': f'{er.mean() - e.mean():+.3f} [{low:+.3f}, {high:+.3f}] {UNIT[t]}'}
    CV_CHOICE[t] = RIDGE if ridge_better else DEFAULT
    choice_rows.append({'target': NAME[t], **row, 'chosen': 'Ridge' if ridge_better else 'heads'})
choice_table = pd.DataFrame(choice_rows).set_index('target')
display(choice_table)
switched = {t: o for t, o in CV_CHOICE.items() if o != DEFAULT}
print('Rule 1 result:', 'the heads stay for every target.' if not switched else
      'Ridge for ' + ', '.join(NAME[t] for t in switched) + '; the heads for every other target.')
choice_table.to_csv(CV_DIR / 'per_target_choice.csv')

In [ ]:
# Step 12c-3 · The final set-up measured two ways: one validation group (Steps 8–9) vs cross-validation
val_ids = data.patient_id.to_numpy()[idx['validation']]
val_people = np.unique(val_ids)
val_draws = np.random.default_rng(20261019).integers(0, len(val_people), (SETTINGS['bootstrap'], len(val_people)))
cv_same = cv_final_pred                                          # the same set-up (FINAL_CFG) in cross-validation


def interval(e, e0, draws):
    boot = 1 - e[draws].mean(1) / e0[draws].mean(1)
    return 100 * (1 - e.mean() / e0.mean()), 100 * np.quantile(boot, 0.025), 100 * np.quantile(boot, 0.975)


two_ways = []
for j, t in enumerate(TARGETS):
    truth = data[t].to_numpy(float)[idx['validation']]
    one = interval(person_errors(truth, ensemble['validation'][0][:, j], val_ids, val_people),
                   person_errors(truth, np.full(len(truth), CONSTANT[t]), val_ids, val_people), val_draws)
    cv = interval(*cv_errors(cv_same[0])[t], cv_draws)
    two_ways.append({'target': t, 'one group: skill %': one[0], 'one group: low': one[1], 'one group: high': one[2],
                     'cross-validation: skill %': cv[0], 'cross-validation: low': cv[1], 'cross-validation: high': cv[2]})
two_ways = pd.DataFrame(two_ways).set_index('target')
two_ways['interval width ratio'] = ((two_ways['one group: high'] - two_ways['one group: low'])
                                    / (two_ways['cross-validation: high'] - two_ways['cross-validation: low']))
prob_val = np.full(len(data), np.nan)
prob_val[idx['validation']] = ensemble['validation'][1]
val_counts, cv_counts = sex_counts(prob_val, idx['validation']), sex_counts(cv_same[1])
sex_two_ways = pd.DataFrame({
    'balanced accuracy %': [100 * balanced_accuracy(val_counts), 100 * balanced_accuracy(cv_counts)],
    'low %': [100 * np.quantile(balanced_accuracy(val_counts[val_draws]), 0.025), 100 * np.quantile(balanced_accuracy(cv_counts[cv_draws]), 0.025)],
    'high %': [100 * np.quantile(balanced_accuracy(val_counts[val_draws]), 0.975), 100 * np.quantile(balanced_accuracy(cv_counts[cv_draws]), 0.975)],
    'people': [len(val_people), len(CV_PEOPLE)]}, index=['one validation group', 'cross-validation'])

fig, ax = plt.subplots(figsize=(9.5, 5.6), layout='constrained')
order = two_ways['cross-validation: skill %'].sort_values().index
for way, colour, offset, people in [('one group', C['ref'], 0.17, len(val_people)), ('cross-validation', C['group1'], -0.17, len(CV_PEOPLE))]:
    yy = np.arange(len(order)) + offset
    part = two_ways.loc[order]
    ax.hlines(yy, part[f'{way}: low'], part[f'{way}: high'], color=colour, lw=2)
    ax.scatter(part[f'{way}: skill %'], yy, s=46, color=colour, edgecolor=C['surface'], linewidth=1.4, zorder=3,
               label=f"{'one validation group' if way == 'one group' else 'cross-validation'}: {people} people")
ax.axvline(0, color=C['ink2'], lw=1)
ax.set_yticks(range(len(order)), [NAME[t] for t in order])
ax.set_xlabel('Skill vs the constant guess (%) · dot = estimate, line = 95% interval (participant bootstrap)')
ax.legend(loc='lower right')
title(fig, f"Step 12c · The same set-up measured two ways: on 510 people the intervals are "
      f"{two_ways['interval width ratio'].median():.1f}× narrower")
save(fig, 'step12c_one_group_vs_cross_validation')
plt.show()
shown = two_ways.rename(index=NAME)
display(shown.round(1))
display(sex_two_ways.round(1))

age_rows = []
for way, rows, pred_age, constant in [('one validation group', idx['validation'], ensemble['validation'][0][:, TARGETS.index('age')],
                                       np.full(len(idx['validation']), CONSTANT['age'])),
                                      ('cross-validation', NONTEST, cv_same[0][NONTEST, TARGETS.index('age')],
                                       CV_CONSTANT[NONTEST, TARGETS.index('age')])]:
    age, ids_ = data.age.to_numpy(float)[rows], data.patient_id.to_numpy()[rows]
    for band, member in [('18–24', (age >= 18) & (age <= 24)), ('older than 24', age > 24)]:
        age_rows.append({'measured on': way, 'age group': band, 'people': len(np.unique(ids_[member])),
                         'model MAE (years)': participant_mae(age[member], pred_age[member], ids_[member]),
                         'constant MAE (years)': participant_mae(age[member], constant[member], ids_[member])})
cv_age_table = pd.DataFrame(age_rows).set_index(['measured on', 'age group'])
cv_age_table['skill %'] = 100 * (1 - cv_age_table['model MAE (years)'] / cv_age_table['constant MAE (years)'])
print('Age by age group:')
display(cv_age_table.round(2))

**Reading 12c** (the saved run).

* **The final set-up, cross-validated** (3 seeds, 510 people):
  * age +54% (error 1.94 years), BMI +25% (2.36 kg/m²), sex 98.6% balanced accuracy;
  * the physiological targets stay within about ±2% of the constant guess, except temperature (−8%) and saturation (+5%).
  * **Temperature** is below zero because of how the constant guess works. 41% of the recordings are exactly 36.6 °C, the median. The model predicts about 36.57 °C for everyone, close to the average, and on absolute error the median guess wins. Its predictions do not follow the true values (correlation −0.03).
  * Saturation is the only physiological target whose interval just excludes zero (+0.9% to +9.1%). With nine targets, one result like this can be chance, and on the test people (Step 10) saturation was −5%. So it is not counted as a finding.
* **The same model, measured two ways (the chart).** The estimates hardly move, but the 95% intervals are about 2.4 times narrower (the median over the 9 targets).
  * Age: +54% on one group (+19% to +69%), +54% by cross-validation (+44% to +60%).
  * BMI: +24% (+11% to +34%) vs +25% (+20% to +30%).
  * Sex: 96.9% (91.9% to 100%) vs 98.6% (97.4% to 99.6%).
* **Age by age group is less flattering on 510 people.**
  * **Older than 24: 48 people instead of 5.** Their error is 8.2 years, not 5.8. The skill is still large (+75%), because guessing 20 for a 60-year-old is very wrong.
  * **18–24 (462 people):** the error is 1.28 years, about the same as on one group (1.30). But always guessing the median, 20, is better here (1.16): for young people the model is about 10% worse than the constant guess.
  * So the age skill comes from recognising that someone is older, not from telling a 19-year-old from a 23-year-old.
* **Rule 1 makes one switch: BMI → Ridge.**
  * Ridge on the same DINOv2 numbers beats the heads: 2.26 vs 2.36 kg/m² (skill +28% vs +25%). The 95% interval of the difference, −0.18 to −0.02, excludes zero.
  * Possible reasons, not tested here: BMI changes gradually with the shape of the face, which a penalised straight line captures well. The heads, in contrast, stop at the epoch that suits all ten tasks together.
  * For age, Ridge is far worse (+22% vs +54%): a straight line cannot follow the cluster of 18–24-year-olds and the few older people at the same time.

### 12d · The final model on all 510 people, and the test people once more

The chosen option for each target is trained again on **all 510 non-test people**, with 3 seeds. That is 90 more people than the model of Steps 8–10 saw, including 5 more people older than 24. Nobody is left to judge it, so it stops at the median kept epoch of its cross-validation rounds.

Then it is scored on the same 90 test people as Step 10.

*This is the second scoring of these test people in this notebook, and they were also used in earlier project experiments. So this is a fair comparison of two models on the same people, not a fresh confirmation, and nothing is chosen on it. The two models also differ in two ways at once: the choices (from cross-validation) and the training people (510 instead of 420).*

In [ ]:
# Step 12d-1 · The final model of Step 12: the chosen option for every target, retrained on all 510 non-test people
ALL_STATS = fit_statistics(NONTEST)                  # scaling and constant guess from all 510 people
FINAL_DIR = CV_DIR / 'final'


def kept_epoch(runs_by_seed):
    """No one is left to judge when training on all 510 people, so the epoch is the median kept epoch of the rounds."""
    return int(round(np.median([run['summary']['best_epoch'] for runs in runs_by_seed for run in runs])))


def final_model(cfg, seed, epochs):
    """Train on all 510 non-test people and stop after `epochs`. Saved in results/cv/final/ and reused with the same settings;
    only the trained parts exist, because the saved DINOv2 numbers stand in for the backbone."""
    name = f'{run_name(cfg, seed)}_all-510_epoch{epochs}'
    settings = {k: SETTINGS[k] for k in TRAINING_SETTINGS} | {'max_batches': MAX_BATCHES, 'people': FOLD_HASH,
                                                              'input': FEATURE_TAG}
    folder = FINAL_DIR / name
    if not RETRAIN and (folder / 'model.pt').is_file() and (folder / 'summary.json').is_file():
        if json.loads((folder / 'summary.json').read_text(encoding='utf-8'))['settings'] == settings:
            model, _ = build_judged_model(cfg)
            result = model.load_state_dict(torch.load(folder / 'model.pt', map_location=DEVICE, weights_only=True), strict=False)
            assert not result.unexpected_keys and all(k.startswith('backbone.') for k in result.missing_keys)
            return model.eval()
    started = time.time()
    with keep_screen_on():
        model = fit(cfg, seed, NONTEST, ALL_STATS, keep_epoch=epochs)
    folder.mkdir(parents=True, exist_ok=True)
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, folder / 'model.pt')
    (folder / 'summary.json').write_text(json.dumps(dict(name=name, config=cfg, seed=seed, epochs=epochs, settings=settings,
                                                         minutes=(time.time() - started) / 60, device=str(DEVICE)), indent=2),
                                         encoding='utf-8')
    print(f'  trained {name}: {(time.time() - started):.0f} s', flush=True)
    return model


def final_ridge_predictions():
    """Ridge refitted on all 510 people (penalties chosen inside them), predicting the test videos. Saved like the rest."""
    file, tag = FINAL_DIR / 'ridge_all-510_test_predictions.npz', f'{FOLD_HASH}_{FEATURE_TAG}'
    if not RETRAIN and file.is_file():
        with np.load(file, allow_pickle=False) as z:
            if str(z['tag']) == tag:
                return z['reg'], z['prob']
    reg, prob = ridge_fit_predict(NONTEST, idx['test'])
    file.parent.mkdir(parents=True, exist_ok=True)
    np.savez(file, tag=np.array(tag), reg=reg, prob=prob)
    return reg, prob


needed = list(dict.fromkeys(CV_CHOICE.values()))                     # every option chosen for at least one target
test_by_option, final_rows = {}, []
for option in needed:
    spec = OPTIONS[option]
    if spec['cfg'] is None:                                           # Ridge: refit on all 510 people (penalties chosen inside)
        test_by_option[option] = final_ridge_predictions()
        final_rows.append({'option': option, 'used for': ', '.join(NAME[t] for t, o in CV_CHOICE.items() if o == option),
                           'trained on': '510 people', 'seeds': '—', 'stopped after epoch': '—'})
        continue
    epochs = kept_epoch(spec['runs'])
    regs, probs = [], []
    for seed in SETTINGS['seeds']:
        out = raw_outputs(final_model(spec['cfg'], seed, epochs), idx['test'], ALL_STATS)
        regs.append(out[:, :9] * ALL_STATS['sd'] + ALL_STATS['mu'])
        probs.append(1 / (1 + np.exp(-out[:, 9])))
    test_by_option[option] = (np.mean(regs, 0), np.mean(probs, 0))
    final_rows.append({'option': option, 'used for': ', '.join(NAME[t] for t, o in CV_CHOICE.items() if o == option),
                       'trained on': '510 people', 'seeds': len(SETTINGS['seeds']), 'stopped after epoch': epochs})
cv_test_reg = np.column_stack([test_by_option[CV_CHOICE[t]][0][:, j] for j, t in enumerate(TARGETS)])
cv_test_prob = test_by_option[CV_CHOICE['sex']][1]
display(pd.DataFrame(final_rows).set_index('option'))

In [ ]:
# Step 12d-2 · The test people once more: Step 12's model vs Step 10's model (for information; nothing is chosen here)
ids = data.patient_id.to_numpy()[idx['test']]
people_list = np.unique(ids)
draws = np.random.default_rng(20261015).integers(0, len(people_list), (SETTINGS['bootstrap'], len(people_list)))  # Step 10a's resamples
rows = []
for j, t in enumerate(TARGETS):
    truth = data[t].to_numpy(float)[idx['test']]
    e_old = person_errors(truth, ensemble['test'][0][:, j], ids, people_list)
    e_new = person_errors(truth, cv_test_reg[:, j], ids, people_list)
    e_const = person_errors(truth, np.full(len(truth), CONSTANT[t]), ids, people_list)    # Step 10's constant guess
    boot = lambda e: 100 * (1 - e[draws].mean(1) / e_const[draws].mean(1))
    rows.append({'target': t, 'Step 10 MAE': e_old.mean(), 'Step 12 MAE': e_new.mean(), 'constant MAE': e_const.mean(),
                 'Step 10 skill %': 100 * (1 - e_old.mean() / e_const.mean()),
                 'Step 10 low %': np.quantile(boot(e_old), 0.025), 'Step 10 high %': np.quantile(boot(e_old), 0.975),
                 'Step 12 skill %': 100 * (1 - e_new.mean() / e_const.mean()),
                 'Step 12 low %': np.quantile(boot(e_new), 0.025), 'Step 12 high %': np.quantile(boot(e_new), 0.975),
                 'Step 12 − Step 10 MAE, 95% interval': '[{:+.3f}, {:+.3f}]'.format(*np.quantile((e_new - e_old)[draws].mean(1), [0.025, 0.975]))})
cv_test_table = pd.DataFrame(rows).set_index('target')

fig, axes = plt.subplots(2, 1, figsize=(9.5, 6.6), height_ratios=[2.2, 7], layout='constrained')
for ax, (panel, members) in zip(axes, [('Appearance targets', ['age', 'bmi']),
                                       ('Physiological targets', [t for t in TARGETS if t not in ('age', 'bmi')])]):
    order = [t for t in val_order if t in members]                 # Step 10's order (fixed by validation, not test)
    for model, colour, offset, label in [('Step 12', C['group1'], 0.16, 'Step 12: cross-validated choices, trained on 510 people'),
                                         ('Step 10', C['spec'], -0.16, 'Step 10: one validation group, trained on 420 people')]:
        part = cv_test_table.loc[order]
        yy = np.arange(len(order)) + offset
        ax.hlines(yy, part[f'{model} low %'], part[f'{model} high %'], color=colour, lw=2)
        ax.scatter(part[f'{model} skill %'], yy, s=46, color=colour, edgecolor=C['surface'], linewidth=1.4, zorder=3, label=label)
    ax.axvline(0, color=C['ink2'], lw=1)
    ax.set_yticks(range(len(order)), [NAME[t] for t in order])
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.set_title(panel)
axes[1].set_xlabel('Test skill vs the constant guess (%) · dot = estimate, line = 95% interval')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncols=1)
title(fig, 'Step 12d · The same 90 test people: Step 12 vs Step 10')
save(fig, 'step12d_test_comparison')
plt.show()
display(cv_test_table.rename(index=NAME).round(3))

truth = data.sex_label.to_numpy(float)[idx['test']]
ok = ~np.isnan(truth)
sex_rows = []
for label, p in [('Step 10 model', ensemble['test'][1]), ('Step 12 model', cv_test_prob)]:
    guess = (p[ok] >= 0.5).astype(float)
    sex_rows.append({'model': label, 'accuracy %': 100 * np.mean(guess == truth[ok]),
                     'balanced accuracy %': 100 * np.mean([np.mean(guess[truth[ok] == c] == c) for c in (0, 1)]),
                     'AUC': auc(truth[ok], p[ok]), 'wrong recordings': f'{int((guess != truth[ok]).sum())} of {ok.sum()}'})
cv_sex_table = pd.DataFrame(sex_rows).set_index('model')
display(cv_sex_table.round(3))

age_rows = []
for band, member in [('18–24', lambda a: (a >= 18) & (a <= 24)), ('older than 24', lambda a: a > 24)]:
    inside = member(data.age.to_numpy()[idx['test']])
    true_age, people_ids = data.age.to_numpy(float)[idx['test']][inside], ids[inside]
    age_rows.append({'age group': band, 'people': len(np.unique(people_ids)),
                     'Step 10 MAE (years)': participant_mae(true_age, ensemble['test'][0][inside, TARGETS.index('age')], people_ids),
                     'Step 12 MAE (years)': participant_mae(true_age, cv_test_reg[inside, TARGETS.index('age')], people_ids),
                     'constant MAE (years)': participant_mae(true_age, np.full(len(true_age), CONSTANT['age']), people_ids)})
cv_test_age = pd.DataFrame(age_rows).set_index('age group')
display(cv_test_age.round(2))

cv_predictions = data.loc[idx['test'], ['key', 'patient_id', 'step']].reset_index(drop=True)
for j, t in enumerate(TARGETS):
    cv_predictions[f'true_{t}'] = data[t].to_numpy()[idx['test']]
    cv_predictions[f'pred_{t}'] = cv_test_reg[:, j]
cv_predictions['true_sex'] = truth
cv_predictions['pred_prob_male'] = cv_test_prob
cv_predictions.to_csv(CV_DIR / 'test_predictions.csv', index=False)
cv_test_table.to_csv(CV_DIR / 'test_metrics.csv')
cv_sex_table.to_csv(CV_DIR / 'test_sex_metrics.csv')
cv_test_age.to_csv(CV_DIR / 'test_age_groups.csv')
pd.DataFrame({option: cv_score(*spec['pred']) for option, spec in OPTIONS.items()}).T.to_csv(CV_DIR / 'cross_validation_scores.csv')
print('Saved in', CV_DIR.relative_to(PROJECT), ': folds.csv, per_target_choice.csv, cross_validation_scores.csv, '
      'test_predictions.csv, test_metrics.csv, test_sex_metrics.csv, test_age_groups.csv')

**Reading 12d** (the saved run; for information only, nothing was chosen here).

* **On the same 90 test people, the Step 12 model beats the Step 10 model for age and BMI.**
  * **Age:** 1.50 vs 1.63 years (skill +57% vs +53%); the difference, −0.23 to −0.02 years, excludes zero. Both models have the same set-up, so the gain comes from the 90 extra training people and the stopping epoch taken from cross-validation.
  * **BMI:** 2.45 vs 2.81 kg/m² (skill +28% vs +18%); the difference, −0.60 to −0.13, excludes zero. About half comes from the 90 extra people (the same heads trained on 510 people reach 2.63) and half from Ridge (2.45). Ridge was chosen on the 510 non-test people, and the test people agree.
  * **Sex:** the same 4 of 180 recordings are wrong (AUC 0.998 vs 0.997).
  * **Physiological targets:** no real change; every difference interval includes zero. Cholesterol is now slightly below the constant guess (−7%; interval −14% to −1%).
* **Age groups on test:** the gain is in the 18–24 group (1.13 vs 1.28 years; the constant guess 1.35). For the 7 older test people the two models are close (5.93 vs 5.75 years).
* **How much to trust this.** The test people were used before (Step 10 and earlier experiments), and they were looked at only after every choice was fixed. So this is a fair comparison of two finished models, not a new confirmation. The cross-validated numbers of 12e, from 510 people, are the firmer estimate of how well the method works.

### 12e · Summary: the errors with and without cross-validation

The table puts the errors side by side: the same set-up judged on one validation group of 90 people (without cross-validation) and on all 510 non-test people (with it), and the two finished models on the same 90 test people.

In [ ]:
# Step 12e · The errors side by side: without cross-validation (Steps 6–10) and with it (Step 12)
cv_default_scores = cv_score(*cv_final_pred)
error_rows = []
for t in TARGETS:
    error_rows.append({
        'target': f'{NAME[t]} ({UNIT[t]})',
        'without CV: 90 validation people': val_scores[f'mae_{t}'],
        'constant guess, same 90': val_scores[f'mae_{t}'] / (1 - val_scores[f'skill_{t}']),
        'with CV: all 510 non-test people': cv_default_scores[f'mae_{t}'],
        'constant guess, same 510': cv_default_scores[f'mae_{t}'] / (1 - cv_default_scores[f'skill_{t}']),
        'test: Step 10 (without CV)': cv_test_table.loc[t, 'Step 10 MAE'],
        'test: Step 12 (with CV)': cv_test_table.loc[t, 'Step 12 MAE'],
        'test: constant guess': cv_test_table.loc[t, 'constant MAE'],
        'test: Step 12 − Step 10, 95% interval': cv_test_table.loc[t, 'Step 12 − Step 10 MAE, 95% interval']})
error_table = pd.DataFrame(error_rows).set_index('target')
print('Mean absolute error (lower = better). The first four columns use the SAME set-up (DINOv2 + task-specific heads, 3 seeds)\n'
      'on people it never saw; the test columns compare the two finished models on the same 90 test people.')
display(error_table.round(3))
if CV_CHOICE['bmi'] != DEFAULT:
    print(f"BMI with the option rule 3 chose ({CV_CHOICE['bmi']}): cross-validated error "
          f"{cv_score(*OPTIONS[CV_CHOICE['bmi']]['pred'])['mae_bmi']:.3f} kg/m²; this is what Step 12's final model uses.")

sex_errors = pd.DataFrame({
    'without CV: 90 validation people': [100 * val_scores['sex_accuracy'], 100 * val_scores['sex_balanced_accuracy']],
    'with CV: all 510 non-test people': [100 * cv_default_scores['sex_accuracy'], 100 * cv_default_scores['sex_balanced_accuracy']],
    'test: Step 10 (without CV)': cv_sex_table.loc['Step 10 model', ['accuracy %', 'balanced accuracy %']].to_numpy(),
    'test: Step 12 (with CV)': cv_sex_table.loc['Step 12 model', ['accuracy %', 'balanced accuracy %']].to_numpy()},
    index=['accuracy %', 'balanced accuracy %'])
print('Recorded sex (higher = better):')
display(sex_errors.round(1))
error_table.to_csv(CV_DIR / 'error_comparison.csv')

**Reading the error table** (the saved run).

* **Without vs with cross-validation, same set-up (the first four columns).** For most targets the error barely changes. Two differences are about *who* is judged, not about the model:
  * **Age: 1.55 years without CV, 1.94 with CV.** The 510 people include 48 older people (the 90 validation people only 5), and older ages are harder to guess. The constant guess gets worse in the same way (3.40 → 4.18 years). Measured against it, the model is just as good: skill +54% both ways.
  * **Stiffness: 2.40 vs 2.25.** Here the 510 people happen to be easier for the model and the constant guess alike (2.45 → 2.26).
  * So raw errors from different groups of people cannot be compared directly. Compare each with the constant guess of the **same** people (or use the test columns).
* **The two finished models on the same 90 test people (the test columns).** This is the direct comparison:
  * **Age: 1.63 → 1.50 years** and **BMI: 2.81 → 2.45 kg/m²** with cross-validation. Both intervals exclude zero.
  * **Every other target:** the same error within the noise; every interval includes zero.
  * **Sex:** the same 4 of 180 recordings are wrong.

**What cross-validation changed** (Step 12, saved run of 19 October 2026):

1. **The numbers are firmer.** The same model's skill estimates barely moved, but their 95% intervals are about 2.4 times narrower: age +54% (+44% to +60%), BMI +25% (+20% to +30%), sex 98.6% balanced accuracy.
2. **Age is clearer and less flattering.** With 48 older people instead of 5, the error for people over 24 is 8.2 years, not 5.8. For 18–24-year-olds the model is no better than always guessing 20 (about 10% worse across the 510 people). The age skill comes from recognising older faces.
3. **One target changed model: BMI now uses Ridge** on the same DINOv2 numbers (rule 1).
4. **The final model, trained on all 510 people, is better on the test people for age and BMI:** 1.50 vs 1.63 years, and 2.45 vs 2.81 kg/m². Sex and the physiological targets did not change.
5. **The physiological targets stay at the constant-guess level**, now measured on 510 people. Only saturation has a small positive estimate (+5%), which the test people did not confirm.
6. **Both parts stay in the notebook.**
   * Steps 6–11 give the first, single-split result, and they contain the building blocks that Step 12 reuses.
   * Step 12 is the stronger check. Its final model rests on more training people, and the test people agree that it is better.

When the backbone and setting comparisons were still part of this notebook, cross-validation also repeated them on the 510 people and chose the same backbone and settings as the single validation group (the numbers are in the plan).

**Next steps** (proposals, not run):
* For age: more people older than 24, or age-balanced sampling. Cross-validation shows the weak spot clearly: 8.2 years of error for the older group.
* The rest of Step 11's list still applies, above all a different input for the physiological targets. Step 13 below gives them the 1,800 frames.

## Step 13 · The 1,800 frames for the physiological targets

The models of Steps 8–12 use the 8 face pictures only. On 19 October 2026 the student decided that **the 7 physiological targets** (oxygen saturation, temperature, HbA1c, cholesterol, respiratory rate, arterial stiffness and stress) **also use the 1,800 frames**, because a pulse or a breath shows only in a continuous recording, never in 8 still pictures. Age, BMI and recorded sex keep Step 12's choices.

Three ways of giving the model the 1,800 frames are compared:

| Way | What the model gets from the 1,800 frames |
|---|---|
| Face pictures + 110 summary numbers | Step 2h's 110 numbers per video (pulse and breathing spectra, 4 quality numbers) through the 110 → 32 signal branch of Step 6a |
| Face pictures + time-series network | **new:** the traces themselves, 15 × 1,800 samples per video, read by a network that learns its own filters (13a–13b) |
| Time-series network alone | the same network without the face pictures: what do the traces show on their own? |

**The rules, fixed before running**

1. **Everything as in Step 12:** the same 5 groups, training settings and early stopping, and 3 seeds. Scaling is learned from each round's 408 people only. The test people stay locked away until 13d.
2. **The 7 physiological targets use the 1,800 frames.** This is the student's decision, not a result. Of the three ways, the final model uses the one with the **highest mean cross-validated skill over these 7 targets**.
3. **The difference from Step 12's model** (without the 1,800 frames) is reported for every target with its 95% interval. It decides nothing.
4. **The final model** is retrained on all 510 people (3 seeds, stopped at the median kept epoch). The test people are scored once more, for information.

**Run time.** About 13 minutes the first time on this laptop's GPU (45 rounds of 13–18 seconds). The rounds are saved in `results/cv/` and reused afterwards, like those of Step 12.

### 13a · What the time-series network reads

The 110 summary numbers of Step 2h describe each video by its spectra: how strong each rhythm is over the whole minute. That throws away the order in time and the shape of each beat. The time-series network gets the traces themselves: **15 traces of 1,800 samples per video** (60 s × 30 per second), made from the signals that Step 2h extracted from every frame.

* **9 colour traces:** red, green and blue of the forehead and both cheeks. Each is the change relative to its own average, so skin tone is removed.
* **3 pulse waves:** Step 2h-2's POS wave of each patch. POS is a fixed recipe that turns the three colours into a pulse.
* **3 head movements:** up/down, left/right and nearer/farther, measured in eye distances. Breathing moves the head a little.

Each video is prepared on its own, without labels or other videos:

1. **Slow changes removed.** The 15-second moving average is subtracted. Drifting light and posture go; breathing (2–10 s per breath) and the pulse stay.
2. **Fast flicker removed.** Everything faster than 4 per second is cut. The room lights add a fake 6–10 Hz rhythm (Step 2h-3), which held about a third of the power of the colour traces.
3. **The same size for every video.** Each group of 3 traces is divided by its typical size in that video, so a video with a lot of movement does not drown the others. The 5 group sizes are kept as 5 extra numbers. Rare spikes are capped at 8 times the typical size.

In [ ]:
# Step 13a · The time-series network's input: 15 traces of 1,800 samples per video, made from the signals of Step 2h
TRACE_NAMES = ([f'{patch} {colour}' for patch in ('forehead', 'right cheek', 'left cheek') for colour in ('red', 'green', 'blue')]
               + [f'pulse wave (POS), {patch}' for patch in ('forehead', 'right cheek', 'left cheek')]
               + ['head up/down', 'head left/right', 'head nearer/farther'])
TRACE_GROUPS = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11], [12, 13, 14]]   # the traces of a group keep their relative sizes
SLOW_S, FAST_HZ, CLIP = 15, 4.0, 8       # remove changes slower than ~15 s and faster than 4 per second; cap at 8x the typical size
MIRRORED = [0, 1, 2, 6, 7, 8, 3, 4, 5, 9, 11, 10, 12, 13, 14]                  # a mirrored face: the two cheeks swap places
MIRROR_SIGN = torch.tensor([-1.0 if name == 'head left/right' else 1.0 for name in TRACE_NAMES])   # and left/right flips


def remove_slow_changes(x, seconds=SLOW_S):
    """Subtract the moving average over `seconds` (centred; shorter at the ends) along axis 1 (time).
    Drifting light and posture go; breathing (2-10 s per breath) and the pulse stay."""
    half = int(seconds * FS) // 2
    c = np.concatenate([np.zeros_like(x[:, :1]), np.cumsum(x, axis=1)], axis=1)
    i = np.arange(x.shape[1])
    lo, hi = np.maximum(i - half, 0), np.minimum(i + half + 1, x.shape[1])
    return x - (c[:, hi] - c[:, lo]) / (hi - lo)[None, :, None]


def video_traces():
    """All 1,200 videos -> (videos, 15 traces, 1,800 samples) and the typical size of each trace group (videos, 5).
    Each video is prepared on its own (no labels, no other videos), so nothing here is learned from the training people."""
    with np.errstate(invalid='ignore'):
        colour = RGB / RGB.mean(axis=1, keepdims=True) - 1                   # each patch colour relative to its own average
    pulse = np.zeros((len(data), RGB.shape[1], 3))                            # Step 2h-2's POS wave of each patch
    for i in range(len(data)):
        for k in range(3):
            if np.isfinite(RGB[i, :, 3 * k:3 * k + 3]).all():                 # (a patch hidden by hair stays 0)
                pulse[i, :, k] = pos_pulse(RGB[i, :, 3 * k:3 * k + 3])
    eye = np.hypot(LANDMARKS[:, :, 2] - LANDMARKS[:, :, 0], LANDMARKS[:, :, 3] - LANDMARKS[:, :, 1])
    typical_eye = np.median(eye, axis=1, keepdims=True)                       # head movement measured in eye distances
    motion = np.stack([LANDMARKS[:, :, 1::2].mean(2) / typical_eye, LANDMARKS[:, :, 0::2].mean(2) / typical_eye,
                       np.log(eye / typical_eye)], axis=2)
    x = np.nan_to_num(remove_slow_changes(np.concatenate([colour, pulse, motion], axis=2))).transpose(0, 2, 1)
    spectrum_ = np.fft.rfft(x, axis=2)
    spectrum_[:, :, np.fft.rfftfreq(x.shape[2], 1 / FS) > FAST_HZ] = 0        # the room lights' 6-10 Hz flicker goes
    x = np.fft.irfft(spectrum_, x.shape[2], axis=2)
    size = np.zeros((len(x), len(TRACE_GROUPS)))
    for g, members in enumerate(TRACE_GROUPS):
        size[:, g] = 1.4826 * np.median(np.abs(x[:, members]), axis=(1, 2))    # typical size (= the SD for a bell curve)
        x[:, members] /= np.where(size[:, g] > 0, size[:, g], 1.0)[:, None, None]
    return np.clip(x, -CLIP, CLIP).astype(np.float32), size


started = time.time()
TRACES_NP, TRACE_SIZE = video_traces()
assert np.isfinite(TRACES_NP).all()
TRACES = torch.from_numpy(TRACES_NP).to(DEVICE)                              # (1200, 15, 1800) on the GPU, about 130 MB
LOG_SIZE = np.log(np.where(TRACE_SIZE > 0, TRACE_SIZE, np.nan))              # how strong each group was: 5 numbers per video


def with_trace_sizes(stats, rows):
    """A Step 12 statistics dict plus the 5 trace sizes, standardised with the given (fitting) people only."""
    m, s = np.nanmean(LOG_SIZE[rows], 0), np.nanstd(LOG_SIZE[rows], 0)
    return dict(stats, size=torch.tensor(np.nan_to_num((LOG_SIZE - m) / s), dtype=torch.float32).to(DEVICE))


TRACE_STATS = {k: with_trace_sizes(FOLD_STATS[k], FIT_ROWS[k]) for k in range(N_FOLDS)}   # each round: its own 408 people
ALL_TRACE_STATS = with_trace_sizes(ALL_STATS, NONTEST)                                    # the final model: all 510

i = examples[0]                                         # Step 2h-3's training video with the clearest pulse
fig, axes = plt.subplots(3, 2, figsize=(14, 6.4), width_ratios=[2.2, 1], sharey='row', layout='constrained')
for row, (channel, colour, what) in zip(axes, [(1, C['group1'], 'forehead green: skin colour (pulse, breathing, light)'),
                                               (9, C['spec'], 'pulse wave (POS) of the forehead'),
                                               (12, C['aqua'], 'head up/down: movement (breathing, posture)')]):
    for ax, (a, b) in zip(row, [(0, 60), (20, 30)]):
        t = np.arange(a * FS, b * FS) / FS
        ax.plot(t, TRACES_NP[i, channel, a * FS:b * FS], color=colour, lw=0.9 if b - a > 10 else 1.4)
        ax.set_xlim(a, b)
    row[0].set_title(what, fontsize=9.5)
    row[1].set_title('the same, seconds 20-30', fontsize=9.5)
    row[0].set_ylabel('typical sizes')
axes[-1, 0].set_xlabel('seconds')
axes[-1, 1].set_xlabel('seconds')
title(fig, f"Step 13a · What the time-series network reads: 3 of the 15 traces of one training video "
      f"({data.key[i].replace('_FullHDwebcam_', ' · ')}), 1,800 samples each")
save(fig, 'step13a_traces')
plt.show()

clipped = np.abs(TRACES_NP[NONTEST]) >= CLIP
print(f'traces: {tuple(TRACES.shape)} = videos x {len(TRACE_NAMES)} traces x 1,800 samples '
      f'({TRACES_NP.nbytes / 1e6:.0f} MB, made in {time.time() - started:.0f} s)')
print(f'samples capped at {CLIP}x the typical size: {clipped.mean():.2%} (non-test videos); '
      f'patches without skin (left at 0): {int((TRACE_SIZE == 0).sum())}')
display(pd.DataFrame({'group': ['forehead colour', 'right cheek colour', 'left cheek colour', 'pulse waves (POS)', 'head movement'],
                      'traces': [', '.join(TRACE_NAMES[c] for c in members) for members in TRACE_GROUPS]}).set_index('group'))

### 13b · The time-series network, and a check that it can read a pulse

```
15 traces x 1,800 samples
   │  6 blocks: 1-D convolution over 7 samples → BatchNorm → ReLU → halve the length
   │  (each block looks at a longer stretch of time; the last one at about 15 seconds)
   ▼
32 patterns x 28 moments ──► average and largest value over the minute, + the 5 trace sizes
   ▼
32 numbers per video   (in place of the 32 from the 110 summary numbers)
   ▼
joined to the 384 DINOv2 numbers of the face (or used alone) ──► the same 10 task-specific heads
```

* **A 1-D convolution** is the time version of a CNN's picture filters. A small pattern of 7 samples (a quarter of a second) slides along the traces and measures how well each moment matches it. The network learns 32 such patterns, then patterns of patterns.
* **About 42,000 trained numbers.** The network is small on purpose: each round has only about 800 training videos.
* **Training as in Step 12.** The optimiser, learning rate, epochs, batches and early stopping are the same, and the network and the heads are trained together. The face part uses the saved DINOv2 numbers (12b). The face pictures are mirrored at random, and so are the traces: the two cheeks swap places.

**The check (13b-2).** If the network could not read a pulse at all, a poor result on the 7 targets would say nothing about the 1,800 frames. So the network is first trained on one thing a minute of video certainly contains: the **pulse during the video**, as measured by the synced finger sensor (Step 2h-5).
* This is done in the same 5 rounds, so every recording is judged by a network that never saw that person.
* The finger pulse is used **only for this check**. It is not one of the 10 targets, and it is never an input.

In [ ]:
# Step 13b · The time-series network, and Step 12b-2's training with it in place of the 110 -> 32 signal branch
class TraceNet(nn.Module):
    """15 traces x 1,800 samples -> 32 numbers. Six blocks of [1-D convolution over 7 samples + BatchNorm + ReLU + halve
    the length]: each block sees a longer stretch of time, the last about 15 seconds (several breaths, many heartbeats).
    Then the average and the largest value of each of the 32 channels over the minute, joined to the 5 trace sizes."""
    blocks, hidden, kernel, width = 6, 32, 7, 32

    def __init__(self, channels=len(TRACE_NAMES)):
        super().__init__()
        layers = []
        for b in range(self.blocks):
            layers += [nn.Conv1d(channels if b == 0 else self.hidden, self.hidden, self.kernel, padding=self.kernel // 2, bias=False),
                       nn.BatchNorm1d(self.hidden), nn.ReLU(), nn.MaxPool1d(2)]
        self.features = nn.Sequential(*layers)
        self.out = nn.Sequential(nn.Linear(2 * self.hidden + len(TRACE_GROUPS), self.width), nn.ReLU())

    def forward(self, traces, size):              # traces: (videos, 15, samples); size: (videos, 5)
        h = self.features(traces)                  # (videos, 32, 28): 32 learned patterns, 28 moments of the minute
        return self.out(torch.cat([h.mean(2), h.amax(2), size], dim=1))


class TraceModel(nn.Module):
    """MultiTaskCNN's heads (Step 6a) on [the 384 saved DINOv2 numbers (faces=True) or nothing] + the 32 of TraceNet.
    The face part is always the saved numbers of the frozen DINOv2 (Step 12b), the backbone of the final model."""

    def __init__(self, cfg):
        super().__init__()
        self.faces = cfg['faces']
        self.signal = TraceNet()
        width = (DINOv2Small.width if self.faces else 0) + TraceNet.width
        hidden = SETTINGS['head_hidden']
        self.dropout = nn.Dropout(SETTINGS['dropout'])
        self.heads = nn.ModuleDict({task: nn.Sequential(nn.Linear(width, hidden), nn.ReLU(), nn.Linear(hidden, 1))
                                    for task in TASKS})

    def forward(self, traces, size, encoded=None):
        parts = ([encoded] if self.faces else []) + [self.signal(traces, size)]
        shared = self.dropout(torch.cat(parts, dim=1))        # join what the face and the traces show
        return torch.cat([self.heads[task](shared) for task in TASKS], dim=1)   # (videos, 10)


def trace_inputs(rows, stats, mirror=False):
    """Traces and sizes of the videos in rows (a tensor). mirror=True mirrors a random half of them (training only)."""
    rows = rows.to(DEVICE)
    traces, size = TRACES[rows], stats['size'][rows]
    if mirror:
        flip = (torch.rand(len(rows)) < 0.5).to(DEVICE)
        traces = torch.where(flip[:, None, None], traces[:, MIRRORED] * MIRROR_SIGN.to(DEVICE)[None, :, None], traces)
    return traces, size


@torch.no_grad()
def trace_outputs(model, rows, stats, batch=32):
    """Like raw_outputs() of Step 12b-2: all 8 frames (for faces=True) and the whole minute of traces; the 10 raw outputs."""
    model.eval()
    parts = []
    for i in range(0, len(rows), batch):
        r = torch.from_numpy(rows[i:i + batch])
        parts.append(model(*trace_inputs(r, stats), encoded=PLAIN_MEAN[r.to(DEVICE)] if model.faces else None).cpu())
    return torch.cat(parts).numpy()


def fit_traces(cfg, seed, fit_rows, stats, judge_rows=None, keep_epoch=None):
    """fit() of Step 12b-2 with TraceModel: the same optimiser, schedule, batches, saved face versions and frames,
    plus the traces of every video (mirrored at random). Returns the judged outputs of every epoch, or the model."""
    set_seed(seed)
    model = TraceModel(cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=SETTINGS['lr_heads'], weight_decay=SETTINGS['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SETTINGS['epochs'])
    rows = torch.from_numpy(fit_rows)
    assert not np.isin(fit_rows, idx['test']).any(), 'a test video in the training rows'
    outputs = []
    for epoch in range(1, SETTINGS['epochs'] + 1):
        model.train()
        for b, batch in enumerate(rows[torch.randperm(len(rows))].split(SETTINGS['batch_videos'])):
            if MAX_BATCHES is not None and b >= MAX_BATCHES:
                break
            shared = None
            if model.faces:       # as fit(): one of the 8 saved augmented versions of each video, and 4 random frames of it
                version = torch.randint(AUG_VERSIONS, (len(batch),))
                chosen_frames = torch.argsort(torch.rand(len(batch), 8), dim=1)[:, :SETTINGS['frames_train']]
                place = NONTEST_POS[batch.to(DEVICE)]
                shared = VERSIONS[version.to(DEVICE)[:, None], place[:, None], chosen_frames.to(DEVICE)].float().mean(1)
            out = model(*trace_inputs(batch, stats, mirror=True), encoded=shared)
            loss = task_losses(out, stats['y'][batch].to(DEVICE), S[batch].to(DEVICE)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        if judge_rows is not None:
            outputs.append(trace_outputs(model, judge_rows, stats))
        if epoch == keep_epoch:
            return model.eval()
    return outputs


TRACE_TAG = (f'traces-{len(TRACE_NAMES)}x{TRACES.shape[2]}_slow{SLOW_S}s_fast{FAST_HZ:g}Hz_clip{CLIP}_mirror'
             f'_net{TraceNet.blocks}x{TraceNet.hidden}k{TraceNet.kernel}')     # saved runs must match the input and the network


def trace_name(cfg, seed):
    return f"timeseries-net_{'faces-and-traces' if cfg['faces'] else 'traces-only'}_mlp-heads_seed{seed}"


def trace_settings(cfg, people):
    return {k: SETTINGS[k] for k in TRAINING_SETTINGS} | {'max_batches': MAX_BATCHES, 'people': people,
                                                          'input': (FEATURE_TAG + ' + ' if cfg['faces'] else '') + TRACE_TAG}


def trace_run(cfg, seed, group):
    """judged_run() of Step 12b-2 for TraceModel: train on the 408 people outside the group, judge the group.
    Saved in results/cv/runs/ and reused when the settings and the groups are the same."""
    fit_rows, judge_rows, stats = FIT_ROWS[group], FOLD_ROWS[group], TRACE_STATS[group]
    name, settings = f'{trace_name(cfg, seed)}_round{group + 1}', trace_settings(cfg, FOLD_HASH)
    folder = CV_RUNS / name
    if not RETRAIN and (folder / 'summary.json').is_file() and (folder / 'outputs.npy').is_file():
        summary = json.loads((folder / 'summary.json').read_text(encoding='utf-8'))
        if summary['settings'] == settings:
            return dict(summary=summary, outputs=np.load(folder / 'outputs.npy'), rows=judge_rows, stats=stats)
    started = time.time()
    with keep_screen_on():
        outputs = np.stack(fit_traces(cfg, seed, fit_rows, stats, judge_rows)).astype(np.float32)
    losses = [judged_loss(out, judge_rows, stats) for out in outputs]
    best = early_stopping_epoch(losses)
    summary = dict(name=name, config=cfg, seed=seed, group=str(group), settings=settings, losses=losses, best_epoch=best,
                   best_loss=losses[best - 1], minutes=(time.time() - started) / 60, device=str(DEVICE))
    folder.mkdir(parents=True, exist_ok=True)
    np.save(folder / 'outputs.npy', outputs)
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"  trained {name}: kept epoch {best}, judged loss {losses[best - 1]:.4f}, {summary['minutes'] * 60:.0f} s", flush=True)
    return dict(summary=summary, outputs=outputs, rows=judge_rows, stats=stats)


def trace_cv_runs(cfg, seed):
    return [trace_run(cfg, seed, k) for k in range(N_FOLDS)]


def final_trace_model(cfg, seed, epochs):
    """final_model() of Step 12d-1 for TraceModel: all 510 non-test people, stopped after `epochs`; saved in results/cv/final/."""
    name, settings = f'{trace_name(cfg, seed)}_all-510_epoch{epochs}', trace_settings(cfg, FOLD_HASH)
    folder = FINAL_DIR / name
    if not RETRAIN and (folder / 'model.pt').is_file() and (folder / 'summary.json').is_file():
        if json.loads((folder / 'summary.json').read_text(encoding='utf-8'))['settings'] == settings:
            model = TraceModel(cfg).to(DEVICE)
            model.load_state_dict(torch.load(folder / 'model.pt', map_location=DEVICE, weights_only=True))
            return model.eval()
    started = time.time()
    with keep_screen_on():
        model = fit_traces(cfg, seed, NONTEST, ALL_TRACE_STATS, keep_epoch=epochs)
    folder.mkdir(parents=True, exist_ok=True)
    torch.save({k: v.cpu() for k, v in model.state_dict().items()}, folder / 'model.pt')
    (folder / 'summary.json').write_text(json.dumps(dict(name=name, config=cfg, seed=seed, epochs=epochs, settings=settings,
                                                         minutes=(time.time() - started) / 60, device=str(DEVICE)), indent=2),
                                         encoding='utf-8')
    print(f'  trained {name}: {(time.time() - started):.0f} s', flush=True)
    return model


example = TraceModel(dict(faces=True))
layer_rows = [{'part': 'time-series network (TraceNet)', 'trained numbers': sum(p.numel() for p in example.signal.parameters())},
              {'part': '10 task heads on 384 + 32 numbers', 'trained numbers': sum(p.numel() for p in example.heads.parameters())}]
with torch.no_grad():
    shape = tuple(example.signal.features(torch.zeros(1, len(TRACE_NAMES), TRACES.shape[2])).shape)
display(pd.DataFrame(layer_rows).set_index('part'))
print(f'TraceNet: {len(TRACE_NAMES)} traces x {TRACES.shape[2]} samples -> {shape[1]} patterns x {shape[2]} moments '
      f'-> average + largest + 5 sizes -> {TraceNet.width} numbers; input tag {TRACE_TAG}')
del example

In [ ]:
# Step 13b-2 · A check of the network itself: can it read a pulse from the traces?
# The target of this check is the synced finger pulse of Step 2h-5. It is used ONLY here: it is not one of the 10 targets,
# and it is never an input of any model.
PULSE_CHECK = CV_DIR / 'pulse_check.npz'
finger = rates['finger pulse'].to_numpy(float)                       # NaN where the finger signal is flat (Step 2h-5)
pulse_tag = f'{FOLD_HASH}_{TRACE_TAG}_' + json.dumps({k: SETTINGS[k] for k in TRAINING_SETTINGS} | {'max_batches': MAX_BATCHES})


def pulse_check_round(k, seed=0):
    """TraceNet + one output, trained on the finger pulse of round k's 408 people with Step 12's optimiser and schedule
    (all 15 epochs; nothing is chosen here). Returns the judged rows, their predictions and the constant guess."""
    fit_rows, judge_rows = (r[np.isfinite(finger[r])] for r in (FIT_ROWS[k], FOLD_ROWS[k]))
    m, s = finger[fit_rows].mean(), finger[fit_rows].std()
    y = torch.tensor((np.nan_to_num(finger) - m) / s, dtype=torch.float32)
    set_seed(seed)
    net, head = TraceNet().to(DEVICE), nn.Linear(TraceNet.width, 1).to(DEVICE)
    optimizer = torch.optim.AdamW([*net.parameters(), *head.parameters()], lr=SETTINGS['lr_heads'],
                                  weight_decay=SETTINGS['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SETTINGS['epochs'])
    rows = torch.from_numpy(fit_rows)
    for epoch in range(SETTINGS['epochs']):
        net.train()
        for b, batch in enumerate(rows[torch.randperm(len(rows))].split(SETTINGS['batch_videos'])):
            if MAX_BATCHES is not None and b >= MAX_BATCHES:
                break
            out = head(net(*trace_inputs(batch, TRACE_STATS[k], mirror=True)))[:, 0]
            loss = F.huber_loss(out, y[batch].to(DEVICE), delta=SETTINGS['huber_delta'])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
    net.eval()
    with torch.no_grad():
        out = head(net(*trace_inputs(torch.from_numpy(judge_rows), TRACE_STATS[k])))[:, 0].cpu().numpy()
    return judge_rows, out * s + m, np.median(finger[fit_rows])


saved_check = None
if not RETRAIN and PULSE_CHECK.is_file():
    with np.load(PULSE_CHECK, allow_pickle=False) as z:
        if str(z['tag']) == pulse_tag:
            saved_check = z['network'], z['constant']
if saved_check is None:
    saved_check = np.full(len(data), np.nan), np.full(len(data), np.nan)
    with keep_screen_on():
        for k in range(N_FOLDS):
            judged, estimate, constant = pulse_check_round(k)
            saved_check[0][judged], saved_check[1][judged] = estimate, constant
    np.savez(PULSE_CHECK, tag=np.array(pulse_tag), network=saved_check[0], constant=saved_check[1])
network_pulse, constant_pulse = saved_check
ok = np.isfinite(network_pulse)                                       # the non-test recordings with a finger signal
estimates = {'time-series network (learned from the traces)': network_pulse[ok],
             'Step 2h: strongest rhythm in the POS spectrum (no learning)': rates['video pulse'].to_numpy(float)[ok],
             'constant guess (median of the fitting people)': constant_pulse[ok]}
pulse_table = pd.DataFrame({name: {'error (bpm)': np.abs(est - finger[ok]).mean(), 'within 5 bpm': (np.abs(est - finger[ok]) <= 5).mean(),
                                   'correlation r': np.nan if name.startswith('constant') else np.corrcoef(est, finger[ok])[0, 1]}
                            for name, est in estimates.items()}).T

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True, layout='constrained')
for ax, (name, est), short, colour in zip(axes, list(estimates.items())[:2], ['time-series network', 'Step 2h POS spectrum'],
                                         [C['group1'], C['ref']]):
    ax.scatter(finger[ok], est, s=9, alpha=0.45, color=colour, edgecolor='none')
    ax.plot([40, 185], [40, 185], color=C['ink2'], lw=1)
    ax.set_xlim(40, 185); ax.set_ylim(40, 185)
    ax.set_xlabel('finger pulse during the video (per minute)')
    ax.set_title(f"{short}: error {pulse_table.loc[name, 'error (bpm)']:.1f} bpm, "
                 f"{pulse_table.loc[name, 'within 5 bpm']:.0%} within 5 bpm", fontsize=9.5)
axes[0].set_ylabel('pulse read from the video (per minute)')
title(fig, f'Step 13b-2 · The network reads the pulse: {ok.sum()} non-test recordings, each judged in its own round')
save(fig, 'step13b_pulse_check')
plt.show()
display(pulse_table.round(3))

**Reading 13b** (the saved run, 19 October 2026).

* **The network can read a pulse.** It was trained on the finger pulse in Step 12's 5 rounds. For people it never saw, it is off by **4.5 beats per minute** on average (r = 0.87). Always guessing the median is off by 10.8.
* **It also avoids the big misses of Step 2h's recipe.** The strongest rhythm in the POS spectrum is within 5 bpm just as often (71% vs 70%). But when it misses, it misses by a lot: it picks a slow rhythm near 42 bpm, or a fake one near 150 (right panel). So its average error is 11.7 bpm.
* **So the time-series network can read the traces.** If it does not help the 7 targets below, the reason is not that it cannot see a pulse.

### 13c · Cross-validation: the 7 physiological targets with and without the 1,800 frames

Each way is trained with 3 seeds in all 5 rounds, exactly like Step 12c's final set-up. So every non-test person gets a prediction from models that never saw them.
* **The chart:** the cross-validated skill of the 7 physiological targets for all four versions. Grey is Step 12's model without the 1,800 frames; the other three are the ways with them.
* **The first table:** the same skills as numbers, with the mean over the 7 targets that rule 2 uses.
* **The second table:** the chosen way against Step 12's model, compared person by person on the same 510 people.
* **13c-2:** whether the predictions follow the pulse during the video (the finger pulse of Step 2h-5, used as a check only).

In [ ]:
# Step 13c · Cross-validation: three ways to give the model the 1,800 frames, 3 seeds each, in the same 5 rounds as Step 12
PHYSIOLOGY = [t for t in TARGETS if t not in ('age', 'bmi')]             # the 7 targets that get the 1,800 frames
WITHOUT = 'without the 1,800 frames (Step 12)'
WAYS = {'face pictures + 110 summary numbers': (dict(FINAL_CFG, signals=True), cv_runs),
        'face pictures + time-series network': (dict(faces=True), trace_cv_runs),
        'time-series network alone': (dict(faces=False), trace_cv_runs)}
way_runs = {way: [runs_of(cfg, seed) for seed in SETTINGS['seeds']] for way, (cfg, runs_of) in WAYS.items()}
way_pred = {WITHOUT: cv_final_pred} | {way: predictions_of(runs) for way, runs in way_runs.items()}
way_errors = {way: cv_errors(pred[0]) for way, pred in way_pred.items()}

skill_rows = []
for t in PHYSIOLOGY:
    row = {'target': NAME[t]}
    for way, errors in way_errors.items():
        row[f'{way}: skill %'], row[f'{way}: low'], row[f'{way}: high'] = interval(*errors[t], cv_draws)   # Step 12c-3's interval()
    skill_rows.append(row)
way_skill = pd.DataFrame(skill_rows).set_index('target')
mean_skill = {way: way_skill[f'{way}: skill %'].mean() for way in way_errors}
CHOSEN_WAY = max(WAYS, key=mean_skill.get)                                 # the rule, fixed before running

compare_rows = []
for t in PHYSIOLOGY:
    e_without, e_with = way_errors[WITHOUT][t][0], way_errors[CHOSEN_WAY][t][0]
    low, high = np.quantile((e_with - e_without)[cv_draws].mean(1), [0.025, 0.975])
    compare_rows.append({'target': NAME[t], 'MAE without': e_without.mean(), 'MAE with': e_with.mean(),
                         'constant guess MAE': way_errors[WITHOUT][t][1].mean(),
                         'with − without, 95% interval': f'{e_with.mean() - e_without.mean():+.3f} [{low:+.3f}, {high:+.3f}] {UNIT[t]}',
                         'verdict': ('better with the 1,800 frames' if high < 0 else 'worse with the 1,800 frames' if low > 0
                                     else 'no clear difference')})
way_compare = pd.DataFrame(compare_rows).set_index('target')

fig, ax = plt.subplots(figsize=(10, 6.4), layout='constrained')
styles = {WITHOUT: C['ref'], 'face pictures + 110 summary numbers': C['aqua'],
          'face pictures + time-series network': C['group1'], 'time-series network alone': C['cnn']}
order = way_skill[f'{WITHOUT}: skill %'].sort_values().index
for w, (way, colour) in enumerate(styles.items()):
    yy = np.arange(len(order)) + 0.27 - 0.18 * w
    part = way_skill.loc[order]
    ax.hlines(yy, part[f'{way}: low'], part[f'{way}: high'], color=colour, lw=2)
    ax.scatter(part[f'{way}: skill %'], yy, s=40, color=colour, edgecolor=C['surface'], linewidth=1.2, zorder=3,
               label=f"{way} · mean {mean_skill[way]:+.1f}%" + ('  ← chosen' if way == CHOSEN_WAY else ''))
ax.axvline(0, color=C['ink2'], lw=1)
ax.set_yticks(range(len(order)), order)
ax.set_xlabel('Cross-validated skill vs the constant guess (%) · dot = estimate, line = 95% interval (510 people)')
fig.legend(loc='outside lower center', ncols=2)
title(fig, 'Step 13c · The 7 physiological targets on 510 people, without and with the 1,800 frames')
save(fig, 'step13c_physiology_with_1800_frames')
plt.show()

display(way_skill[[f'{way}: skill %' for way in way_errors]].rename(columns=lambda c: c.replace(': skill %', ''))
        .round(1).T.assign(**{'mean of the 7': [round(mean_skill[w], 1) for w in way_errors]}))
print(f'Chosen by the rule (highest mean skill over the 7 physiological targets): {CHOSEN_WAY}. '
      f'Kept epochs (median of {len(way_runs[CHOSEN_WAY]) * N_FOLDS} rounds): {kept_epoch(way_runs[CHOSEN_WAY])}.')
print('With the chosen way vs without the 1,800 frames (paired difference in MAE over the same 510 people):')
display(way_compare.round(3))
way_skill.to_csv(CV_DIR / 'step13_cross_validation_skill.csv')
way_compare.to_csv(CV_DIR / 'step13_with_vs_without.csv')

In [ ]:
# Step 13c-2 · What do the 1,800 frames add? Do the predictions follow the pulse during the video (the finger pulse, a check only)?
with_finger = np.isfinite(finger[NONTEST])
pulse_during = finger[NONTEST][with_finger]
follow_rows = []
for t in ['respiratory', 'rigidity']:
    j = TARGETS.index(t)
    row = {'target': NAME[t], 'recorded label': np.corrcoef(data[t].to_numpy(float)[NONTEST][with_finger], pulse_during)[0, 1]}
    for way, pred in way_pred.items():
        row[f'prediction: {way}'] = np.corrcoef(pred[0][NONTEST, j][with_finger], pulse_during)[0, 1]
    follow_rows.append(row)
follow_table = pd.DataFrame(follow_rows).set_index('target')
print(f'Correlation r with the finger pulse during the video ({with_finger.sum()} non-test recordings; predictions from Step 13c):')
display(follow_table.round(2))

**Reading 13c** (the saved run).

* **Rule 2 chose the 110 summary numbers.** Their mean skill over the 7 targets is −0.1%. The face pictures + time-series network reach −0.7%, the network alone −1.8%, and Step 12's model without the 1,800 frames −0.8%.
* **With the 1,800 frames, two targets are measurably better: the two the physiology points to.**
  * **Respiratory rate:** 1.348 → 1.319 breaths/min. The difference, −0.043 to −0.015, excludes zero. But the model is now only as good as always guessing the median (skill +0.5%, up from −1.7%).
  * **Arterial stiffness:** 2.248 → 2.187 (−0.089 to −0.030). Skill +3.0%, up from +0.3%.
  * With 3 seeds on 510 people, both gains are large enough to tell apart from chance.
* **The other 5 targets do not change;** every interval includes zero. Temperature, HbA1c and cholesterol do not change within a minute, so watching longer shows nothing new about them. For saturation and stress the model finds nothing it can use in the frames.
* **Where the gain comes from (13c-2): the pulse rate.**
  * Both labels are a little higher when the pulse during the video is higher (r = 0.22 for respiratory rate, 0.29 for stiffness).
  * With the summary numbers, the predictions follow the pulse: r = 0.26 and 0.43, against 0.05 and 0.14 without the frames.
  * So the model mainly uses how fast the heart beats, not the breathing itself (Step 2h found no breathing signal) or the shape of each beat.
* **The time-series network did not beat the 110 summary numbers.**
  * Next to the face pictures, its predictions hardly follow the pulse (r = 0.09 and 0.13). With only weak labels to learn from, it did not find the pulse on its own, although it can read one when it is taught (13b-2).
  * The 110 numbers hand the model the pulse spectrum directly.
  * Used alone, the network does follow the pulse a little (r = 0.28 and 0.29). But without the face it loses what the pictures give saturation and HbA1c, so its mean is the lowest.

### 13d · The final model of Step 13, and the test people once more

* **The 7 physiological targets:** the chosen way, retrained on all 510 non-test people with 3 seeds. It stops at the median kept epoch of its cross-validation rounds.
* **Age and recorded sex:** Step 12's final model (frozen DINOv2 + task-specific heads). **BMI:** Step 12's Ridge.

Then the same 90 test people are scored once more, against Step 12's final model.

*As in 12d, this compares two finished models on people who were used before. It is for information only, and nothing is chosen on it.*

In [ ]:
# Step 13d · The final model of Step 13: the chosen way for the 7 physiological targets, Step 12's model for the rest
chosen_cfg = WAYS[CHOSEN_WAY][0]
epochs13 = kept_epoch(way_runs[CHOSEN_WAY])
regs = []
for seed in SETTINGS['seeds']:                     # retrained on all 510 non-test people, stopped at the median kept epoch
    if chosen_cfg.get('signals'):                  # face pictures + 110 summary numbers: final_model() of Step 12d-1
        out = raw_outputs(final_model(chosen_cfg, seed, epochs13), idx['test'], ALL_STATS)
    else:
        out = trace_outputs(final_trace_model(chosen_cfg, seed, epochs13), idx['test'], ALL_TRACE_STATS)
    regs.append(out[:, :9] * ALL_STATS['sd'] + ALL_STATS['mu'])
step13_test_reg = cv_test_reg.copy()               # age and sex from Step 12's model, BMI from Step 12's Ridge
for j, t in enumerate(TARGETS):
    if t in PHYSIOLOGY:
        step13_test_reg[:, j] = np.mean(regs, 0)[:, j]

test_ids = data.patient_id.to_numpy()[idx['test']]
test_people = np.unique(test_ids)
test_draws = np.random.default_rng(20261015).integers(0, len(test_people), (SETTINGS['bootstrap'], len(test_people)))  # Step 10a's
rows = []
for j, t in enumerate(TARGETS):
    if t not in PHYSIOLOGY:
        continue
    truth = data[t].to_numpy(float)[idx['test']]
    e12, e13 = (person_errors(truth, reg[:, j], test_ids, test_people) for reg in (cv_test_reg, step13_test_reg))
    e0 = person_errors(truth, np.full(len(truth), CONSTANT[t]), test_ids, test_people)     # Step 10's constant guess
    s12, s13 = interval(e12, e0, test_draws), interval(e13, e0, test_draws)
    low, high = np.quantile((e13 - e12)[test_draws].mean(1), [0.025, 0.975])
    rows.append({'target': NAME[t], 'Step 12 MAE': e12.mean(), 'Step 13 MAE': e13.mean(), 'constant MAE': e0.mean(),
                 'Step 12 skill %': s12[0], 'Step 13 skill %': s13[0], 'Step 13 low %': s13[1], 'Step 13 high %': s13[2],
                 'Step 13 − Step 12 MAE, 95% interval': f'{e13.mean() - e12.mean():+.3f} [{low:+.3f}, {high:+.3f}] {UNIT[t]}'})
step13_test = pd.DataFrame(rows).set_index('target')
display(pd.DataFrame([{'targets': 'the 7 physiological targets', 'model': CHOSEN_WAY, 'trained on': '510 people',
                       'seeds': len(SETTINGS['seeds']), 'stopped after epoch': epochs13},
                      {'targets': 'age, recorded sex', 'model': DEFAULT, 'trained on': '510 people', 'seeds': len(SETTINGS['seeds']),
                       'stopped after epoch': kept_epoch(cv_final)},
                      {'targets': 'BMI', 'model': CV_CHOICE['bmi'], 'trained on': '510 people', 'seeds': '—',
                       'stopped after epoch': '—'}]).set_index('targets'))
print('The same 90 test people, Step 12 (without the 1,800 frames) vs Step 13 (with them); for information, nothing is chosen here:')
display(step13_test.round(3))

step13_predictions = cv_predictions.copy()          # Step 12d-2's table of the test videos, with Step 13's predictions
for j, t in enumerate(TARGETS):
    step13_predictions[f'pred_{t}'] = step13_test_reg[:, j]
step13_predictions.to_csv(CV_DIR / 'step13_test_predictions.csv', index=False)
step13_test.to_csv(CV_DIR / 'step13_test_metrics.csv')

In [ ]:
# Step 13e · The errors side by side: Step 12 (without the 1,800 frames) and Step 13 (with them)
error_rows = []
for t in PHYSIOLOGY:
    cv, test = way_compare.loc[NAME[t]], step13_test.loc[NAME[t]]
    error_rows.append({'target': f'{NAME[t]} ({UNIT[t]})',
                       'CV: without (Step 12)': cv['MAE without'], 'CV: with (Step 13)': cv['MAE with'],
                       'CV: constant guess': cv['constant guess MAE'],
                       'CV: with − without, 95% interval': cv['with − without, 95% interval'].rsplit(' ', 1)[0],
                       'test: Step 12': test['Step 12 MAE'], 'test: Step 13': test['Step 13 MAE'], 'test: constant guess': test['constant MAE'],
                       'test: Step 13 − Step 12, 95% interval': test['Step 13 − Step 12 MAE, 95% interval'].rsplit(' ', 1)[0]})
step13_errors = pd.DataFrame(error_rows).set_index('target')
print('Mean absolute error of the 7 physiological targets (lower = better). CV = all 510 non-test people, each judged by\n'
      f'models that never saw them; test = the same 90 test people for both models. "With" = {CHOSEN_WAY}.')
display(step13_errors.round(3))
print('Age, BMI and recorded sex are the same as in Step 12: frozen DINOv2 + task-specific heads, Ridge for BMI.')
step13_errors.to_csv(CV_DIR / 'step13_error_comparison.csv')

# Every version of Step 13c, cross-validated: MAE, and in brackets the skill (% smaller than always guessing the median)
order = [CHOSEN_WAY, *[way for way in WAYS if way != CHOSEN_WAY], WITHOUT]
label = {way: f'{way} (final model)' if way == CHOSEN_WAY else way for way in order}
numbers, shown = {}, {}
for t in PHYSIOLOGY:
    e0 = way_errors[WITHOUT][t][1].mean()                      # always guessing the median (the same for every version)
    for way in order:
        e = way_errors[way][t][0].mean()
        numbers.setdefault(f'{label[way]}: MAE', {})[NAME[t]] = e
        numbers.setdefault(f'{label[way]}: skill %', {})[NAME[t]] = 100 * (1 - e / e0)
        shown.setdefault(label[way], {})[f'{NAME[t]} ({UNIT[t]})'] = f'{e:.3f} ({100 * (1 - e / e0):+.1f}%)'
    numbers.setdefault('always guessing the median: MAE', {})[NAME[t]] = e0
    shown.setdefault('always guessing the median', {})[f'{NAME[t]} ({UNIT[t]})'] = f'{e0:.3f}'
print()
print('Every version on the same 510 people: MAE (in brackets: how much smaller the error is than always guessing the '
      'median; skill = 1 - MAE / MAE of the median guess, so negative = worse than guessing):')
display(pd.DataFrame(shown))
pd.DataFrame(numbers).to_csv(CV_DIR / 'step13_mae_all_versions.csv')

**Reading 13d** (the saved run; for information only, nothing was chosen here).

* **On the same 90 test people, respiratory rate and stiffness move the same way as in cross-validation,** but 90 people are too few to be sure:
  * **respiratory rate:** 1.409 → 1.388 breaths/min (skill +0.9% → +2.4%); the difference, −0.065 to +0.022, includes zero;
  * **stiffness:** 1.871 → 1.795 (skill −1.5% → +2.6%); −0.174 to +0.013.
* **Two targets change on the test people only.**
  * **Saturation is better:** 0.850 → 0.806 (−0.079 to −0.013).
  * **Stress is worse:** 1.249 → 1.286 (+0.010 to +0.066).
  * Neither changed on the 510 cross-validation people. With 7 targets on 90 people, one or two intervals like these are expected by chance, so neither is counted.
* **All 7 stay within a few percent of the constant guess,** with or without the 1,800 frames. On the test people, saturation, cholesterol and stress are still slightly below it.
* **Age, BMI and recorded sex are unchanged** from Step 12.

**What Step 13 shows** (saved run of 19 October 2026):

1. **Step 13's final model uses the 1,800 frames for the 7 physiological targets,** as the student decided. Of the three ways tried, the 110 summary numbers of Step 2h did best (rule 2), so the final model uses them.
2. **They help where the physiology says they should, but only a little.**
   * On 510 people, respiratory rate and arterial stiffness are measurably better than without the frames.
   * Stiffness reaches +3.0% skill; respiratory rate only reaches the level of the constant guess.
   * The test people point the same way, but they are too few to confirm it.
3. **The gain seems to come from the pulse rate.** With the summary numbers, the predictions follow the pulse that the finger sensor measured during the video, and both labels are a little higher when the pulse is higher.
4. **The other 5 targets gain nothing.** For temperature, HbA1c and cholesterol this is expected: they do not change within a minute.
5. **The time-series network can read a pulse when it is taught one** (4.5 bpm error), but the weak labels alone did not teach it to. That points to the next step.

**Next steps** (proposals, not run):
* **Teach the network the pulse first, then the targets.** Train it on the finger pulse (13b-2), then build the 7 targets on what it learned. This makes the finger pulse a training target, so it needs the student's decision; it is a step towards Stage 2 (contact PPG).
* **Ask the mentor when the after-exercise labels were measured.** They may have been taken before filming (Step 2h-5). If so, no video model can match them closely.